# 06 — Model Results & Comprehensive Evaluation

**Pipeline position:** Step 6 of 7

**Purpose:** Load the saved bootstrap ensemble and compute comprehensive
performance metrics (AUC-ROC, AUC-PR, calibration, DCA, clinical metrics)
with 1,000-iteration bootstrap 95% CI across all three dataset splits.
Generates publication-quality figures and summary tables.

**License:** MIT

## Prerequisites
- Run `05_bootstrap_ensemble.ipynb` first
- Inputs: `data_*_selected.csv` files, `./bootstrap_models/model_*.pkl`

## Outputs
- Publication-quality PNG / TIFF figures in `./figures/`
- `performance_summary.csv`


In [ ]:
# SPDX-License-Identifier: MIT
# ============================================================
# Dependency check — run this cell first
# ============================================================
import importlib.util

_required = ['numpy', 'pandas', 'sklearn', 'xgboost', 'optuna', 'matplotlib', 'seaborn', 'scipy', 'openpyxl']
_missing = [p for p in _required if importlib.util.find_spec(p) is None]
if _missing:
    raise ImportError(
        f"Missing packages: {_missing}\n"
        f"Install with: pip install {' '.join(_missing)}"
    )
print("Core dependencies satisfied.")

# Bootstrap Ensemble Model — Complete Analysis Pipeline

**Structure:**
1. **Setup** — Imports, configuration, GPU detection
2. **Data** — Load CSV, encode, prepare X/y splits
3. **Models** — Load 300 bootstrap XGBoost models + generate predictions
4. **Evaluation (Single Split)** — Per-split comprehensive evaluation (6 panels)
5. **Evaluation (Cross-Split)** — ROC / PR / Calibration comparison across 3 splits
6. **Evaluation (Cross-Split + CI)** — Same with Bootstrap 95% CI
7. **Risk Stratification** — 4-zone uncertainty × risk scatter plots
8. **Risk Stratification (Enhanced)** — With per-zone clinical metrics
9. **High-Confidence Evaluation** — TU-based confidence filtering
10. **High-Confidence Evaluation (Extended)** — Tri-split version
11. **Subgroup & Table 4** — Stratified performance (age, residence) + formatted table
12. **Tri-Split Framework** — Calibration, DCA, stratified ROC, heatmap across all splits
13. **Feature Importance** — Gain / Weight / SHAP from 300 models + aggregation
14. **Feature Importance Plots** — Bar, boxplot, SHAP beeswarm, dot-ranking, heatmap
15. **Feature Importance Export** — Save results to Excel


## 1. Setup — Imports & Configuration

In [31]:
import pandas as pd

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from matplotlib.patches import Patch
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve,
    precision_recall_curve, auc, average_precision_score,
    brier_score_loss, balanced_accuracy_score
)
from sklearn.calibration import calibration_curve
import pickle
import os
import gc
import time
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

%matplotlib inline
%config InlineBackend.figure_format = 'svg'

# ============================================================
# Configuration
# ============================================================
OUTCOME = 'SpontAbortion'
RANDOM_STATE = 42
N_MODELS = 300       # bootstrap ensemble size
OPTUNA_TRIALS = 50    # Optuna trials per model
SHARE_PARAMS = False
EARLY_STOPPING = 20
DEVICE = "cuda"
MODEL_DIR = "./bootstrap_models"
os.makedirs(MODEL_DIR, exist_ok=True)



## 2. Utility Functions

Bootstrap sampling and ensemble prediction helpers.

In [32]:
from sklearn.utils import resample
import numpy as np


def create_bootstrap_samples(X, y, n_samples=5, random_state=42):
    """Create balanced 1:1 bootstrap sub-samples (both classes sampled with replacement)."""
    # Separate majority and minority classes
    majority_class = 0 if (y == 0).sum() > (y == 1).sum() else 1
    minority_class = 1 - majority_class

    X_majority = X[y == majority_class]
    y_majority = y[y == majority_class]
    X_minority = X[y == minority_class]
    y_minority = y[y == minority_class]

    print(f"\nOriginal class distribution:")
    print(f"Majority class ({majority_class}): {len(X_majority)} samples")
    print(f"Minority class ({minority_class}): {len(X_minority)} samples")

    bootstrap_samples = []

    coverage_rates = []
    # Used to track coverage
    majority_indices_used = set()

    for i in range(n_samples):
        # Bootstrap-sample the majority class (with replacement)
        n_minority = len(X_minority)

        # Key: use replace=True with a different seed per iteration
        X_majority_bootstrap, majority_indices = resample(
            X_majority,
            np.arange(len(X_majority)),  # Also return indices to track majority-class coverage
            replace=True,  # with replacement
            n_samples=n_minority,
            random_state=random_state + i  # different seed per iteration
        )
        y_majority_bootstrap = np.array([majority_class] * n_minority)

        # Track used indices for coverage statistics
        majority_indices_used.update(majority_indices)

        # Bootstrap-sample the minority class (with replacement)
        X_minority_bootstrap = resample(
            X_minority,
            replace=True,  # with replacement
            n_samples=n_minority,
            random_state=random_state + i
        )
        y_minority_bootstrap = np.array([minority_class] * n_minority)

        # Merge balanced sub-sample
        X_bootstrap = np.vstack([X_majority_bootstrap, X_minority_bootstrap])
        y_bootstrap = np.hstack([y_majority_bootstrap, y_minority_bootstrap])

        # Shuffle the merged sample
        indices = np.arange(len(X_bootstrap))
        np.random.seed(random_state + i)
        np.random.shuffle(indices)

        bootstrap_samples.append((X_bootstrap[indices], y_bootstrap[indices]))

        # Verify class distribution
        unique, counts = np.unique(y_bootstrap[indices], return_counts=True)
        coverage_rates.append(len(majority_indices_used) / len(X_majority) * 100)

        # if (i + 1) % 10 == 0 or i < 5:  # print first 5 and every 10th
        #     print(f"\nSub-sample {i+1}:")
        #     print(f"  Total samples: {len(X_bootstrap)}")
        #     print(f"  Class distribution: {dict(zip(unique, counts))}")
        #     print(f"  Ratio (0:1): {counts[0]/counts[1]:.2f}:1")
        #     print(f"  Majority coverage: {len(majority_indices_used)/len(X_majority)*100:.2f}%")

    # Final coverage statistics
    final_coverage = len(majority_indices_used) / len(X_majority) * 100
    print(f"\n" + "="*60)
    print(f"Total sub-samples created: {n_samples}")
    print(f"Majority class coverage: {final_coverage:.2f}%")
    print(f"Covered samples: {len(majority_indices_used)}/{len(X_majority)}")
    print(f"Uncovered samples: {len(X_majority) - len(majority_indices_used)}")
    print("="*60)

    return bootstrap_samples,coverage_rates

In [33]:
def predict_bootstrap_ensemble(models, X, method='mean', return_individual=False):
    """
    Generate ensemble predictions from bootstrap models

    Parameters:
    -----------
    models : list
        List of trained models
    X : array-like
        Feature matrix for prediction
    method : str, default='mean'
        Aggregation method: 'mean', 'median', 'voting', 'weighted'
    return_individual : bool, default=False
        If True, return individual model predictions along with ensemble prediction

    Returns:
    --------
    predictions : array
        Ensemble predictions (probabilities)
    individual_preds : array (optional)
        Individual model predictions if return_individual=True
    """
    from tqdm import tqdm
    import numpy as np

    print(f"\n{'='*70}")
    print(f"Generating Ensemble Predictions")
    print(f"{'='*70}")
    print(f"Number of models: {len(models)}")
    print(f"Number of samples: {X.shape[0]:,}")
    print(f"Aggregation method: {method}")
    print(f"{'='*70}\n")

    # Collect predictions from all models
    all_predictions = []

    for i, model in enumerate(tqdm(models, desc="Predicting", ncols=100)):
        # Get probability predictions for class 1
        y_pred_proba = model.predict_proba(X)[:, 1]
        all_predictions.append(y_pred_proba)

    # Stack predictions into array: (n_models, n_samples)
    all_predictions = np.array(all_predictions)

    return all_predictions





# Method 4: Load models from disk if needed
def load_models_from_disk(model_dir='./bootstrap_models', n_models=None):
    """Load saved models from disk"""
    import os
    import pickle
    from tqdm import tqdm

    model_files = sorted([f for f in os.listdir(model_dir) if f.endswith('.pkl')])

    if n_models is not None:
        model_files = model_files[:n_models]

    models = []
    print(f"Loading {len(model_files)} models from {model_dir}...")

    for model_file in tqdm(model_files, desc="Loading models"):
        model_path = os.path.join(model_dir, model_file)
        with open(model_path, 'rb') as f:
            model = pickle.load(f)
            models.append(model)

    print(f"Successfully loaded {len(models)} models")
    return models



## 3. Data Loading & Encoding

Read CSV files, LabelEncoder for categoricals, convert to numpy.

In [34]:
# ============================================================
# Load data and apply LabelEncoder encoding
# ============================================================
from sklearn.preprocessing import LabelEncoder

TRAIN_FILE  = "data_training_selected.csv"
INTVAL_FILE = "data_internal_validation_selected.csv"
TEMPVAL_FILE = "data_temporal_validation_selected.csv"

df_train_raw = pd.read_csv(TRAIN_FILE)
df_val_raw   = pd.read_csv(INTVAL_FILE)
df_test_raw  = pd.read_csv(TEMPVAL_FILE)

drop_cols = ['Index', 'BaseInfoDate', 'Dataset']

# Encode all three dataset splits consistently
def encode_all(df_train, df_val, df_test):
    """
    Handle actual dtypes (bool + object + int + float):
      - bool    → int (0/1)
      - object  → LabelEncoder (fit on training, transform on val/test)
      - int/float → unchanged
      - unknown categories → most frequent training-set class (no NaN produced)
    """
    dfs = {"train": df_train.copy(), "val": df_val.copy(), "test": df_test.copy()}

    for name in dfs:
        dfs[name] = dfs[name].drop(
            columns=[c for c in drop_cols if c in dfs[name].columns], errors='ignore'
        )

    # 1) Outcome variable → int
    for name in dfs:
        df = dfs[name]
        if df[OUTCOME].dtype == bool:
            df[OUTCOME] = df[OUTCOME].astype(int)
        elif df[OUTCOME].dtype == object:
            df[OUTCOME] = df[OUTCOME].map(
                {'True': 1, 'False': 0, 'TRUE': 1, 'FALSE': 0}
            ).astype(int)
        else:
            df[OUTCOME] = pd.to_numeric(df[OUTCOME], errors='coerce').astype(int)

    # 2) Boolean columns → int
    for name in dfs:
        for col in dfs[name].select_dtypes(include=['bool']).columns:
            dfs[name][col] = dfs[name][col].astype(int)

    # 3) Object columns → LabelEncoder
    feature_cols = [c for c in dfs["train"].columns if c != OUTCOME]
    obj_cols = [c for c in feature_cols if dfs["train"][c].dtype == object]
    print(f"  LabelEncoder columns: {obj_cols}")

    label_encoders = {}
    for col in obj_cols:
        le = LabelEncoder()
        train_vals = dfs["train"][col].astype(str)
        le.fit(train_vals)
        label_encoders[col] = le

        # Most frequent training category (replaces unknown categories in val/test)
        most_freq = train_vals.mode()[0]
        known = set(le.classes_)

        for name in dfs:
            col_str = dfs[name][col].astype(str)
            col_safe = col_str.where(col_str.isin(known), most_freq)
            dfs[name][col] = le.transform(col_safe).astype(np.float64)

    # 4) Convert remaining columns to numeric
    for name in dfs:
        for col in feature_cols:
            if col not in obj_cols:
                dfs[name][col] = pd.to_numeric(dfs[name][col], errors='coerce')

    # 5) Drop any remaining NaN rows
    for name in dfs:
        n0 = len(dfs[name])
        dfs[name] = dfs[name].dropna()
        if len(dfs[name]) < n0:
            print(f"  {name}: dropped {n0 - len(dfs[name])} NaN rows")

    features = feature_cols
    return dfs, features, label_encoders

dfs, features, label_encoders = encode_all(df_train_raw, df_val_raw, df_test_raw)

print(f"\nFeatures: {len(features)}")
print(f"  {features}")
print(f"  LabelEncoded: {list(label_encoders.keys())}")


  LabelEncoder columns: ['HusbandEdu', 'WifeEdu', 'HusbandOcc', 'WifeOcc']

Features: 36
  ['WifeAge', 'HusbandAge', 'PrevPregnancy', 'Contraception', 'WifeBMI', 'NeutrophilPct', 'LymphocytePct', 'WifeStress', 'WifeResident', 'WifeFinance', 'HusbDBP', 'TSH', 'RBC', 'HusbandResident', 'HusbandEdu', 'CMVIgG', 'WBC', 'HusbBMI', 'WifeEdu', 'WifeDBP', 'HusbSBP', 'WifeCreat', 'WifeSBP', 'Glucose', 'LeftTestVol', 'GynUltrasound', 'WifeHR', 'RightTestVol', 'HusbandOcc', 'Hemoglobin', 'WifeOcc', 'HusbStress', 'HusbHR', 'Platelet', 'HusbCreat', 'Dysmenorrhea']
  LabelEncoded: ['HusbandEdu', 'WifeEdu', 'HusbandOcc', 'WifeOcc']


In [35]:
# ============================================================
# Convert to numpy float32 arrays
# ============================================================
X_train = dfs["train"][features].astype(np.float32)
X_val   = dfs["val"][features].astype(np.float32)
X_test  = dfs["test"][features].astype(np.float32)
y_train = dfs["train"][OUTCOME].astype(int)
y_val   = dfs["val"][OUTCOME].astype(int)
y_test  = dfs["test"][OUTCOME].astype(int)

print(f"Features : {len(features)}")
print(f"Train    : N={len(y_train):>7,}  SA={y_train.sum():>5,} ({y_train.mean()*100:.2f}%)")
print(f"Val      : N={len(y_val):>7,}  SA={y_val.sum():>5,} ({y_val.mean()*100:.2f}%)")
print(f"Test     : N={len(y_test):>7,}  SA={y_test.sum():>5,} ({y_test.mean()*100:.2f}%)")
# print(f"Dtype    : {X_train.dtype}")
# print(f"NaN      : train={np.isnan(X_train).sum()}, val={np.isnan(X_val).sum()}, test={np.isnan(X_test).sum()}")


Features : 36
Train    : N=360,363  SA=11,753 (3.26%)
Val      : N= 40,041  SA=1,306 (3.26%)
Test     : N=  1,822  SA=  262 (14.38%)


## 4. Load Models & Generate Predictions

In [36]:
models = load_models_from_disk(model_dir='./bootstrap_models', n_models=300)
# models = load_models_from_disk(model_dir='./bootstrap_models', n_models=300)


Loading 300 models from ./bootstrap_models...


Loading models: 100%|██████████| 300/300 [00:07<00:00, 41.84it/s]


Successfully loaded 300 models


In [37]:
predictions_train = predict_bootstrap_ensemble(
    models, X_train
)
predictions_val = predict_bootstrap_ensemble(
    models, X_val
)
predictions_test = predict_bootstrap_ensemble(
    models, X_test
)


Generating Ensemble Predictions
Number of models: 300
Number of samples: 360,363
Aggregation method: mean



Predicting: 100%|█████████████████████████████████████████████████| 300/300 [00:23<00:00, 12.92it/s]



Generating Ensemble Predictions
Number of models: 300
Number of samples: 40,041
Aggregation method: mean



Predicting: 100%|█████████████████████████████████████████████████| 300/300 [00:05<00:00, 56.89it/s]



Generating Ensemble Predictions
Number of models: 300
Number of samples: 1,822
Aggregation method: mean



Predicting: 100%|█████████████████████████████████████████████████| 300/300 [00:03<00:00, 89.97it/s]


## 5. Single-Split Evaluation

Comprehensive 6-panel evaluation for each dataset independently.

In [ ]:
"""
Bootstrap Ensemble Model — Comprehensive Evaluation Module
==========================================================
Design principles:
  - Each analysis step is an independent, reusable function
  - All plot text is in English
  - Constants are centralized at the top
  - Clear docstrings, minimal inline comments
"""

from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score,
    roc_auc_score, roc_curve,
)
from scipy.special import logit as _logit
from scipy.optimize import minimize

# ─────────────────────────────────────────────
# Global configuration
# ─────────────────────────────────────────────
FIGSIZE             = (16, 16)
DPI                 = 300
OUTPUT_FIG          = "bootstrap_ensemble_analysis.png"
SELECTIVE_POINTS    = 50
COVERAGE_THRESHOLDS = [0.9, 0.8, 0.7, 0.6, 0.5]
TOP_K_SAMPLES       = 10
PROB_INTERVALS      = [(0.0, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0)]

# Keys used in uncertainty measure dict (for ROC plotting)
ERROR_ROC_PLOT_ORDER = [
    "TU (Predictive Entropy)",
    "EU (Mutual Information)",
    "AU (Expected Entropy)",
    "Std (Disagreement)",
    "Boundary Uncertainty",
]

# ─────────────────────────────────────────────
# Utility helpers
# ─────────────────────────────────────────────

def _to_1d(arr) -> np.ndarray:
    """Ensure input is a flat numpy array."""
    return np.asarray(arr).reshape(-1)


def _section(title: str) -> None:
    """Print a section header."""
    print(f"\n{'=' * 60}\n{title}\n{'=' * 60}")

# ─────────────────────────────────────────────
# Uncertainty computation
# ─────────────────────────────────────────────

def compute_uncertainties(predictions: np.ndarray, eps: float = 1e-10) -> dict:
    """
    Decompose predictive uncertainty via Shannon entropy.

    Parameters
    ----------
    predictions : (n_models, n_samples) probability array for class 1
    eps         : numerical stability clamp

    Returns
    -------
    dict with keys: mean_pred, TU, AU, EU, std_pred, ensemble_pred_class
        TU  = H(E[p])          — total uncertainty (predictive entropy)
        AU  = E[H(p)]          — aleatoric uncertainty (expected entropy)
        EU  = TU - AU          — epistemic uncertainty (mutual information)
        std = std(predictions) — disagreement proxy
    """
    mean_pred = predictions.mean(axis=0)

    # TU: entropy of the mean prediction
    p0 = np.clip(1 - mean_pred, eps, 1 - eps)
    p1 = np.clip(mean_pred,     eps, 1 - eps)
    TU = -(p0 * np.log(p0) + p1 * np.log(p1))

    # AU: mean entropy across individual model predictions
    q0 = np.clip(1 - predictions, eps, 1 - eps)
    q1 = np.clip(predictions,     eps, 1 - eps)
    AU = -(q0 * np.log(q0) + q1 * np.log(q1)).mean(axis=0)

    EU = TU - AU  # epistemic = total − aleatoric

    return {
        "mean_pred":           mean_pred,
        "TU":                  TU,
        "AU":                  AU,
        "EU":                  EU,
        "std_pred":            predictions.std(axis=0),
        "ensemble_pred_class": (mean_pred > 0.5).astype(int),
    }

# ─────────────────────────────────────────────
# Metrics
# ─────────────────────────────────────────────



def _calibration_slope_intercept(y_true, y_prob, eps=1e-7):
    """
    Logistic calibration regression: logit(y) = intercept + slope * logit(p_hat).
    Perfect calibration: slope = 1, intercept = 0.
    Reference: Van Calster et al. (2016) J Clin Epidemiol.
    """
    lp = _logit(np.clip(y_prob, eps, 1 - eps))
    y_f = y_true.astype(float)
    def neg_ll(params):
        a, b = params
        prob = 1 / (1 + np.exp(-(a + b * lp)))
        prob = np.clip(prob, eps, 1 - eps)
        return -np.mean(y_f * np.log(prob) + (1 - y_f) * np.log(1 - prob))
    res = minimize(neg_ll, x0=[0.0, 1.0], method="Nelder-Mead",
                   options={"xatol": 1e-6, "fatol": 1e-6, "maxiter": 5000})
    return float(res.x[1]), float(res.x[0])  # slope, intercept

def evaluate_ensemble(y_true: np.ndarray,
                      y_pred: np.ndarray,
                      mean_pred: np.ndarray) -> dict:
    """Return standard binary classification metrics (aligned with Methods §2.5)."""
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp = cm[0, 0], cm[0, 1]

    # AUPR (added per Methods §2.5)
    aupr = average_precision_score(y_true, mean_pred)

    # Brier score (added per Methods §2.5)
    brier = brier_score_loss(y_true, mean_pred)

    # Calibration slope & intercept (added per TRIPOD+AI)
    try:
        cal_slope, cal_intercept = _calibration_slope_intercept(y_true, mean_pred)
    except Exception:
        cal_slope, cal_intercept = float("nan"), float("nan")

    return {
        "accuracy":            accuracy_score(y_true, y_pred),
        "precision":           precision_score(y_true, y_pred, zero_division=0),
        "recall":              recall_score(y_true, y_pred, zero_division=0),
        "f1":                  f1_score(y_true, y_pred, zero_division=0),
        "roc_auc":             roc_auc_score(y_true, mean_pred),
        "aupr":                aupr,
        "brier":               brier,
        "calibration_slope":   cal_slope,
        "calibration_intercept": cal_intercept,
        "specificity":         tn / (tn + fp + 1e-12),
        "confusion_matrix":    cm,
    }


def build_uncertainty_measures(TU, EU, AU, std_pred, mean_pred) -> dict:
    """Build the uncertainty measure dict used for error-detection ROC."""
    boundary = 0.5 - np.abs(mean_pred - 0.5)
    std_norm = (std_pred - std_pred.min()) / (std_pred.ptp() + 1e-12)
    return {
        "TU (Predictive Entropy)": TU,
        "EU (Mutual Information)": EU,
        "AU (Expected Entropy)":   AU,
        "Std (Disagreement)":      std_pred,
        "Boundary Uncertainty":    boundary,
        "Std (Normalized)":        std_norm,
    }

# ─────────────────────────────────────────────
# Curve computation
# ─────────────────────────────────────────────

from sklearn.metrics import balanced_accuracy_score

def selective_prediction_curve(y_true: np.ndarray,
                               y_pred: np.ndarray,
                               uncertainty: np.ndarray,
                               n_points: int = SELECTIVE_POINTS):
    """
    Coverage–accuracy trade-off curve (selective prediction).
    Samples with uncertainty ≤ threshold are retained;
    accuracy is recalculated on the retained subset.
    """
    thresholds = np.percentile(uncertainty, np.linspace(0, 99, n_points))
    coverages, accuracies = [], []
    for thr in thresholds:
        mask = uncertainty <= thr
        if mask.any():
            coverages.append(mask.mean())
            accuracies.append(
                # balanced_accuracy_score(y_true[mask], y_pred[mask])
                  accuracy_score(y_true[mask], y_pred[mask])
            )
    return np.array(coverages), np.array(accuracies)


def error_detection_roc(error_labels: np.ndarray,
                        uncertainty_measures: dict) -> dict:
    """
    For each uncertainty measure, compute the ROC curve treating
    prediction errors as positives.

    Returns a dict[name] with fpr, tpr, auc, best_threshold,
    best_tpr, best_specificity, error_cm.
    """
    results = {}
    for name, scores in uncertainty_measures.items():
        scores = _to_1d(scores)
        fpr, tpr, thresholds = roc_curve(error_labels, scores)
        auc = roc_auc_score(error_labels, scores)

        # Youden's J — maximize TPR − FPR
        j = tpr - fpr
        i = np.argmax(j)

        pred_at_best = (scores >= thresholds[i]).astype(int)
        results[name] = {
            "fpr":             fpr,
            "tpr":             tpr,
            "auc":             auc,
            "best_threshold":  thresholds[i],
            "best_tpr":        tpr[i],
            "best_specificity": 1 - fpr[i],
            "error_cm":        confusion_matrix(error_labels, pred_at_best, labels=[0, 1]),
        }
    return results

# ─────────────────────────────────────────────
# Console reporting
# ─────────────────────────────────────────────

def report_metrics(metrics: dict) -> None:
    _section("Ensemble Model Performance")
    m = metrics
    print(
        f"Accuracy:    {m['accuracy']*100:.2f}%\n"
        f"Precision:   {m['precision']*100:.2f}%\n"
        f"Recall:      {m['recall']*100:.2f}%\n"
        f"F1-Score:    {m['f1']*100:.2f}%\n"
        f"AUC-ROC:     {m['roc_auc']:.4f}\n"
        f"AUPR:        {m['aupr']:.4f}\n"
        f"Brier Score: {m['brier']:.4f}\n"
        f"Cal. Slope:  {m['calibration_slope']:.4f}\n"
        f"Cal. Intcpt: {m['calibration_intercept']:.4f}\n"
        f"Specificity: {m['specificity']*100:.2f}%"
    )
    cm = m["confusion_matrix"]
    print(f"\nConfusion Matrix [0,1]:\n"
          f"  TN={cm[0,0]:<5} FP={cm[0,1]:<5}\n"
          f"  FN={cm[1,0]:<5} TP={cm[1,1]:<5}")


def report_uncertainty_stats(unc: dict) -> None:
    _section("Uncertainty Statistics")
    for key, label in [("TU", "TU  H(E[p])"),
                        ("AU", "AU  E[H(p)]"),
                        ("EU", "EU  TU−AU  "),
                        ("std_pred", "Std         ")]:
        v = unc[key]
        print(f"{label}  mean={v.mean():.4f}  range=[{v.min():.4f}, {v.max():.4f}]")


def report_individual_models(predictions: np.ndarray,
                              y_true: np.ndarray,
                              ensemble_acc: float) -> None:
    _section("Individual Model Accuracy")
    accs = [accuracy_score(y_true, (p > 0.5).astype(int)) for p in predictions]
    for i, a in enumerate(accs):
        print(f"  Model {i+1:>2}: {a*100:.2f}%")
    best = max(accs)
    print(f"\n  Ensemble: {ensemble_acc*100:.2f}%  |  Best single: {best*100:.2f}%"
          f"  |  Gain: {(ensemble_acc - best)*100:+.2f}%")


def report_error_roc(roc_results: dict) -> None:
    _section("Uncertainty → Error Detection ROC")
    for name, r in roc_results.items():
        cm = r["error_cm"]
        print(f"\n{name}")
        print(f"  AUC={r['auc']:.4f}  threshold={r['best_threshold']:.4f}"
              f"  TPR={r['best_tpr']:.4f}  Spec={r['best_specificity']:.4f}")
        print(f"  Error CM:  TN={cm[0,0]}  FP={cm[0,1]}  FN={cm[1,0]}  TP={cm[1,1]}")


def report_selective_prediction(coverages, accuracies, baseline: float) -> None:
    _section("Selective Prediction (TU-based)")
    for thr in COVERAGE_THRESHOLDS:
        i = int(np.argmin(np.abs(coverages - thr)))
        lift = (accuracies[i] - baseline) * 100
        print(f"  Coverage ≈ {coverages[i]*100:.1f}%  →  "
              f"Accuracy {accuracies[i]*100:.2f}%  (lift {lift:+.2f}%)")


def report_threshold_analysis(TU, y_true, y_pred, errors, roc_results) -> tuple:
    _section("Threshold-based Error Detection (TU)")
    thr = roc_results["TU (Predictive Entropy)"]["best_threshold"]
    high, low = TU >= thr, TU < thr
    print(f"Best threshold: {thr:.4f}")
    for label, mask in [("High uncertainty", high), ("Low  uncertainty", low)]:
        n = mask.sum()
        print(f"\n  {label} ({n} samples, {mask.mean()*100:.1f}%)")
        print(f"    Error rate: {errors[mask].mean()*100:.1f}%  |  "
              f"Accuracy: {accuracy_score(y_true[mask], y_pred[mask])*100:.1f}%")
    return high, low


def report_prob_intervals(mean_pred, y_true, y_pred, TU, AU, EU, errors) -> None:
    _section("Analysis by Predicted Probability Interval")
    for lo, hi in PROB_INTERVALS:
        mask = (mean_pred >= lo) & (mean_pred < hi)
        n = mask.sum()
        if n == 0:
            continue
        acc = accuracy_score(y_true[mask], y_pred[mask]) * 100
        err = errors[mask].mean() * 100
        print(f"\n  [{lo:.1f}, {hi:.1f})  {n} samples ({n/len(y_true)*100:.1f}%)")
        print(f"    Accuracy={acc:.1f}%  Error={err:.1f}%  "
              f"TU={TU[mask].mean():.4f}  AU={AU[mask].mean():.4f}  EU={EU[mask].mean():.4f}")


def report_extreme_samples(TU, mean_pred, y_true, y_pred, correct_mask) -> None:
    _section(f"Extreme Uncertainty Samples (Top/Bottom {TOP_K_SAMPLES})")
    for desc, idx in [("Highest TU", np.argsort(TU)[-TOP_K_SAMPLES:][::-1]),
                      ("Lowest  TU", np.argsort(TU)[:TOP_K_SAMPLES])]:
        print(f"\n  {desc}:")
        for rank, i in enumerate(idx, 1):
            print(f"    {rank:02d}  TU={TU[i]:.4f}  p={mean_pred[i]:.3f}"
                  f"  y={y_true[i]}  pred={y_pred[i]}"
                  f"  correct={bool(correct_mask[i])}")


def report_summary(accuracy, roc_auc, roc_results,
                   errors, high_mask, low_mask,
                   coverages, accuracies) -> None:
    _section("Summary Report")
    print(
        f"  1. Ensemble accuracy:      {accuracy*100:.2f}%\n"
        f"  2. Ensemble AUC-ROC:       {roc_auc:.4f}\n"
        f"  3. TU error-detection AUC: {roc_results['TU (Predictive Entropy)']['auc']:.4f}\n"
        f"  4. Error rate (high TU):   {errors[high_mask].mean()*100:.1f}%\n"
        f"  5. Error rate (low  TU):   {errors[low_mask].mean()*100:.1f}%"
    )
    if len(accuracies):
        i = int(np.argmax(accuracies))
        print(f"  6. Max selective lift:     "
              f"{(accuracies[i]-accuracy)*100:.2f}% @ coverage={coverages[i]*100:.1f}%")
    print("=" * 60)

# ─────────────────────────────────────────────
# Visualization (3 × 3)
# ─────────────────────────────────────────────

def _plot_roc_curve(ax, y_true, mean_pred) -> None:
    fpr, tpr, _ = roc_curve(y_true, mean_pred)
    auc = roc_auc_score(y_true, mean_pred)
    ax.plot(fpr, tpr, lw=2.5, label=f"AUC = {auc:.3f}")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.5)
    ax.set(title="ROC Curve", xlabel="FPR", ylabel="TPR")
    ax.legend(); ax.grid(alpha=0.3)


def _plot_uncertainty_distribution(ax, TU, correct_mask) -> None:
    for mask, label in [(correct_mask, "Correct"), (~correct_mask, "Incorrect")]:
        ax.hist(TU[mask], bins=30, alpha=0.6, label=label, density=True)
    ax.set(title="TU Distribution by Prediction Correctness",
           xlabel="TU (Predictive Entropy)", ylabel="Density")
    ax.legend(); ax.grid(alpha=0.3)


def _plot_selective_prediction(ax, coverages, accuracies, baseline) -> None:
    ax.plot(coverages, accuracies, lw=2.5, marker="o", markersize=3)
    ax.axhline(baseline, ls="--", lw=2, label=f"Baseline ({baseline*100:.2f}%)")
    ax.fill_between(coverages, baseline, accuracies,
                    where=(accuracies >= baseline), alpha=0.25, label="Improvement zone")
    ax.set(title="Selective Prediction Curve",
           xlabel="Coverage", ylabel="Accuracy", xlim=[0, 1], ylim=[0, 1])
    ax.legend(); ax.grid(alpha=0.3)


def _plot_confusion_matrix(ax, cm) -> None:
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                annot_kws={"size": 14}, cbar_kws={"label": "Count"})
    ax.set(title="Confusion Matrix", xlabel="Predicted Label", ylabel="True Label")
    ax.set_xticklabels(["Negative (0)", "Positive (1)"])
    ax.set_yticklabels(["Negative (0)", "Positive (1)"], rotation=0)


def _plot_pred_prob_distribution(ax, mean_pred, y_true) -> None:
    bins = np.linspace(0, 1, 30)
    for cls, label in [(0, "Negative"), (1, "Positive")]:
        ax.hist(mean_pred[y_true == cls], bins=bins, alpha=0.6,
                label=label, density=True)
    ax.axvline(0.5, ls="--", lw=2, label="Decision threshold")
    ax.set(title="Predicted Probability Distribution by True Class",
           xlabel="Mean Predicted Probability", ylabel="Density")
    ax.legend(); ax.grid(alpha=0.3)


def _plot_agreement_vs_pred(ax, mean_pred, predictions, TU) -> None:
    agreement = (predictions > 0.5).mean(axis=0)
    sc = ax.scatter(mean_pred, agreement, c=TU, cmap="viridis",
                    alpha=0.7, s=40, edgecolors="black", linewidths=0.3)
    plt.colorbar(sc, ax=ax, label="TU (Predictive Entropy)")
    ax.set(title="Mean Prediction vs. Model Agreement",
           xlabel="Mean Predicted Probability", ylabel="Vote Agreement Ratio")
    ax.grid(alpha=0.3)


def _plot_error_roc(ax, roc_results) -> None:
    for name in ERROR_ROC_PLOT_ORDER:
        if name not in roc_results:
            continue
        r = roc_results[name]
        ax.plot(r["fpr"], r["tpr"], lw=2, label=f"{name} ({r['auc']:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1.5, alpha=0.5, label="Random")
    ax.set(title="Error Detection ROC (Uncertainty vs. Prediction Error)",
           xlabel="FPR", ylabel="TPR", xlim=[-0.01, 1.01], ylim=[-0.01, 1.01])
    ax.legend(loc="lower right", fontsize=8); ax.grid(alpha=0.3)


def _plot_uncertainty_boxplot(ax, TU, EU, AU, correct_mask) -> None:
    COLOR_WRONG, COLOR_RIGHT = "#FF6B6B", "#4ECDC4"
    positions = np.arange(3) * 3
    for offset, mask, color in [(-0.5, ~correct_mask, COLOR_WRONG),
                                 (+0.5,  correct_mask, COLOR_RIGHT)]:
        bp = ax.boxplot([TU[mask], EU[mask], AU[mask]],
                        positions=positions + offset, widths=0.4,
                        patch_artist=True, medianprops=dict(linewidth=2))
        for box in bp["boxes"]:
            box.set(facecolor=color, alpha=0.65)

    ax.set_xticks(positions)
    ax.set_xticklabels(["TU\n(Predictive Entropy)", "EU\n(Mutual Info)", "AU\n(Expected Entropy)"])
    ax.set(title="Uncertainty Comparison: Correct vs. Incorrect Predictions",
           ylabel="Uncertainty Value")
    ax.legend(handles=[mpatches.Patch(color=COLOR_WRONG, alpha=0.65, label="Incorrect"),
                       mpatches.Patch(color=COLOR_RIGHT, alpha=0.65, label="Correct")],
              loc="upper right")
    ax.grid(alpha=0.3, axis="y")


def _plot_correlation_heatmap(ax, corr_df) -> None:
    sns.heatmap(corr_df, annot=True, fmt=".3f", cmap="coolwarm",
                center=0, square=True, cbar_kws={"shrink": 0.8}, ax=ax)
    ax.set_title("Uncertainty Measure Correlation Matrix")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)


def plot_comprehensive(y_true, mean_pred, y_pred, TU, AU, EU, std_pred,
                       predictions, roc_results, coverages, accuracies,
                       corr_matrix, cm, accuracy,
                       output_path: str = OUTPUT_FIG) -> None:
    """Render and save the 3×3 analysis dashboard."""
    correct_mask = y_pred == y_true
    errors       = (y_pred != y_true).astype(int)

    corr_df = pd.DataFrame({
        "TU": TU, "AU": AU, "EU": EU, "Std": std_pred,
        "Boundary": 0.5 - np.abs(mean_pred - 0.5),
        "Error":    errors,
        "MeanPred": mean_pred,
    }).corr()

    fig, axes = plt.subplots(3, 3, figsize=FIGSIZE)
    axes = axes.flatten()

    _plot_roc_curve(axes[0], y_true, mean_pred)
    _plot_uncertainty_distribution(axes[1], TU, correct_mask)
    _plot_selective_prediction(axes[2], coverages, accuracies, accuracy)
    _plot_confusion_matrix(axes[3], cm)
    _plot_pred_prob_distribution(axes[4], mean_pred, y_true)
    _plot_agreement_vs_pred(axes[5], mean_pred, predictions, TU)
    _plot_error_roc(axes[6], roc_results)
    _plot_uncertainty_boxplot(axes[7], TU, EU, AU, correct_mask)
    _plot_correlation_heatmap(axes[8], corr_df)

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")
    print(f"\nFigure saved: {output_path}")
    plt.show()
    plt.close(fig)

# ─────────────────────────────────────────────
# Main pipeline
# ─────────────────────────────────────────────

def run_evaluation(predictions_array: np.ndarray, y_train, output_path) -> None:
    """
    Full evaluation pipeline for a bootstrap ensemble.

    Parameters
    ----------
    predictions_array : (n_models, n_samples) — per-model probabilities
    y_train           : true binary labels
    """
    y_true = _to_1d(y_train)

    # ── Uncertainty decomposition ──────────────────────────
    unc          = compute_uncertainties(predictions_array)
    mean_pred    = unc["mean_pred"]
    TU, AU, EU   = unc["TU"], unc["AU"], unc["EU"]
    std_pred     = unc["std_pred"]
    y_pred       = unc["ensemble_pred_class"]

    # ── Classification metrics ─────────────────────────────
    metrics = evaluate_ensemble(y_true, y_pred, mean_pred)
    report_metrics(metrics)
    report_uncertainty_stats(unc)
    report_individual_models(predictions_array, y_true, metrics["accuracy"])

    # ── Curves ────────────────────────────────────────────
    coverages, accuracies = selective_prediction_curve(y_true, y_pred, TU)

    errors           = (y_pred != y_true).astype(int)
    unc_measures     = build_uncertainty_measures(TU, EU, AU, std_pred, mean_pred)
    roc_results      = error_detection_roc(errors, unc_measures)

    print(f"\nPrediction errors: {errors.sum()} / {len(errors)} "
          f"({errors.mean()*100:.2f}%)")
    report_error_roc(roc_results)

    # ── Correlation ───────────────────────────────────────
    corr_df = pd.DataFrame({
        "TU": TU, "AU": AU, "EU": EU, "Std": std_pred,
        "Boundary": 0.5 - np.abs(mean_pred - 0.5),
        "Error": errors, "MeanPred": mean_pred,
    })
    _section("Uncertainty Measure Correlations")
    print(corr_df.corr().round(4))

    # ── Visualization ─────────────────────────────────────
    plot_comprehensive(
        y_true, mean_pred, y_pred, TU, AU, EU, std_pred,
        predictions_array, roc_results, coverages, accuracies,
        corr_df.corr(), metrics["confusion_matrix"], metrics["accuracy"],output_path
    )

    # ── Detailed reports ──────────────────────────────────
    report_selective_prediction(coverages, accuracies, metrics["accuracy"])
    high_mask, low_mask = report_threshold_analysis(
        TU, y_true, y_pred, errors, roc_results)
    report_summary(
        metrics["accuracy"], metrics["roc_auc"],
        roc_results, errors, high_mask, low_mask, coverages, accuracies,
    )
    report_prob_intervals(mean_pred, y_true, y_pred, TU, AU, EU, errors)
    report_extreme_samples(TU, mean_pred, y_true, y_pred, y_pred == y_true)
    print("\nDone.")

    return




In [ ]:
# Run evaluation for each split
run_evaluation(predictions_train, y_train, output_path='predictions_train.tiff')

In [ ]:
run_evaluation(predictions_val, y_val, output_path='predictions_val.tiff')

In [ ]:
run_evaluation(predictions_test, y_test, output_path='predictions_test.tiff')

## 6. Cross-Split ROC / PR / Calibration Comparison

In [ ]:
"""
ROC / PR / Calibration Curve Comparison
========================================
Compare ensemble model performance across Train, Validation, and Test sets.

Expected inputs (defined in outer scope):
  predictions_train : (n_models, n_train_samples)
  predictions_val   : (n_models, n_val_samples)
  predictions_test  : (n_models, n_test_samples)
  y_train, y_val, y_test : true binary labels
"""

from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────
OUTPUT_FIG   = "roc_pr_calibration_comparison.png"
DPI          = 300
FIGSIZE      = (15, 5)
CAL_BINS     = 10          # bins for calibration curve
HIST_BINS    = 30          # bins for confidence histogram

# Visual style per split
SPLIT_STYLE = {
    "Train": dict(color="#2196F3", ls="-",  lw=2.2, alpha=0.9),
    "Val":   dict(color="#FF9800", ls="--", lw=2.2, alpha=0.9),
    "Test":  dict(color="#E91E63", ls="-.", lw=2.2, alpha=0.9),
}

# ─────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────

def _to_1d(arr) -> np.ndarray:
    return np.asarray(arr).reshape(-1)


def mean_prob(predictions: np.ndarray) -> np.ndarray:
    """Average per-model probabilities → ensemble probability."""
    return predictions.mean(axis=0)


def compute_split_curves(y_true: np.ndarray,
                          prob: np.ndarray,
                          n_cal_bins: int = CAL_BINS) -> dict:
    """
    Compute ROC, PR, and calibration data for one split.

    Returns
    -------
    dict with keys:
        roc   : (fpr, tpr, auc)
        pr    : (recall, precision, ap)
        cal   : (mean_pred_prob, fraction_pos, brier)
        prob  : raw probabilities (for histogram)
    """
    y_true = _to_1d(y_true)

    # ROC
    fpr, tpr, _ = roc_curve(y_true, prob)
    auc = roc_auc_score(y_true, prob)

    # Precision-Recall
    precision, recall, _ = precision_recall_curve(y_true, prob)
    ap = average_precision_score(y_true, prob)

    # Calibration
    frac_pos, mean_prob_cal = calibration_curve(y_true, prob,
                                                n_bins=n_cal_bins,
                                                strategy="uniform")
    brier = brier_score_loss(y_true, prob)

    return {
        "roc":  (fpr, tpr, auc),
        "pr":   (recall, precision, ap),
        "cal":  (mean_prob_cal, frac_pos, brier),
        "prob": prob,
    }

# ─────────────────────────────────────────────
# Sub-plot renderers
# ─────────────────────────────────────────────

def _draw_roc(ax, splits: dict) -> None:
    """ROC curves for all splits."""
    for name, data in splits.items():
        fpr, tpr, auc = data["roc"]
        sty = SPLIT_STYLE[name]
        ax.plot(fpr, tpr, label=f"{name}  AUC={auc:.3f}",
                color=sty["color"], ls=sty["ls"], lw=sty["lw"], alpha=sty["alpha"])

    ax.plot([0, 1], [0, 1], "k--", lw=1.2, alpha=0.5, label="Random")
    ax.fill_between([0, 1], [0, 1], alpha=0.04, color="gray")
    ax.set(title="ROC Curve", xlabel="False Positive Rate (FPR)",
           ylabel="True Positive Rate (TPR)", xlim=[-0.02, 1.02], ylim=[-0.02, 1.02])
    ax.legend(loc="lower right", fontsize=9)
    ax.grid(alpha=0.3)


def _draw_pr(ax, splits: dict) -> None:
    """Precision-Recall curves for all splits."""
    for name, data in splits.items():
        recall, precision, ap = data["pr"]
        sty = SPLIT_STYLE[name]
        ax.plot(recall, precision, label=f"{name}  AP={ap:.3f}",
                color=sty["color"], ls=sty["ls"], lw=sty["lw"], alpha=sty["alpha"])

    ax.set(title="Precision-Recall Curve", xlabel="Recall",
           ylabel="Precision", xlim=[-0.02, 1.02], ylim=[-0.02, 1.02])
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(alpha=0.3)


def _draw_calibration(ax, ax_hist, splits: dict) -> None:
    """
    Calibration curve (reliability diagram) with confidence histogram.

    ax      — main calibration axes
    ax_hist — inset / secondary axes for the histogram
    """
    ax.plot([0, 1], [0, 1], "k--", lw=1.2, alpha=0.5, label="Perfect calibration")

    for name, data in splits.items():
        mean_prob_cal, frac_pos, brier = data["cal"]
        sty = SPLIT_STYLE[name]
        ax.plot(mean_prob_cal, frac_pos,
                marker="o", markersize=6,
                label=f"{name}  Brier={brier:.4f}",
                color=sty["color"], ls=sty["ls"], lw=sty["lw"], alpha=sty["alpha"])

        # Confidence histogram (stacked, semi-transparent)
        ax_hist.hist(data["prob"], bins=HIST_BINS, density=True,
                     alpha=0.35, color=sty["color"], label=name)

    ax.set(title="Calibration Curve (Reliability Diagram)",
           xlabel="Mean Predicted Probability",
           ylabel="Fraction of Positives",
           xlim=[-0.02, 1.02], ylim=[-0.02, 1.02])
    ax.legend(loc="upper left", fontsize=9)
    ax.grid(alpha=0.3)

    ax_hist.set(xlabel="Predicted Probability", ylabel="Density")
    ax_hist.tick_params(labelsize=8)
    ax_hist.grid(alpha=0.25)
    ax_hist.legend(fontsize=8)

# ─────────────────────────────────────────────
# Main plot
# ─────────────────────────────────────────────

def plot_comparison(splits: dict, output_path: str = OUTPUT_FIG) -> None:
    """
    Render the 1×3 panel: ROC | PR | Calibration (+ histogram inset).

    Parameters
    ----------
    splits : dict[str, dict]  — output of compute_split_curves per split name
    """
    fig = plt.figure(figsize=FIGSIZE)
    fig.suptitle("Ensemble Model — Train / Val / Test Comparison",
                 fontsize=14, fontweight="bold", y=1.02)

    gs = gridspec.GridSpec(2, 3, height_ratios=[3, 1], hspace=0.05)

    ax_roc = fig.add_subplot(gs[:, 0])   # full-height ROC
    ax_pr  = fig.add_subplot(gs[:, 1])   # full-height PR
    ax_cal = fig.add_subplot(gs[0, 2])   # top: calibration curve
    ax_his = fig.add_subplot(gs[1, 2])   # bottom: confidence histogram

    _draw_roc(ax_roc, splits)
    _draw_pr(ax_pr, splits)
    _draw_calibration(ax_cal, ax_his, splits)

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")
    print(f"Figure saved: {output_path}")
    plt.show()
    plt.close(fig)

# ─────────────────────────────────────────────
# Console summary
# ─────────────────────────────────────────────

def print_summary(splits: dict) -> None:
    """Print a compact metric table across all splits."""
    header = f"{'Split':<8} {'AUC-ROC':>8} {'AP (PR)':>9} {'Brier':>8}"
    print("\n" + "=" * 40)
    print("Performance Summary")
    print("=" * 40)
    print(header)
    print("-" * 40)
    for name, data in splits.items():
        auc   = data["roc"][2]
        ap    = data["pr"][2]
        brier = data["cal"][2]
        print(f"{name:<8} {auc:>8.4f} {ap:>9.4f} {brier:>8.4f}")
    print("=" * 40)

# ─────────────────────────────────────────────
# Entry point
# ─────────────────────────────────────────────

def run(predictions_train, y_train,
        predictions_val,   y_val,
        predictions_test,  y_test) -> None:
    """
    Compute curves and render comparison figure.

    Each predictions_* array has shape (n_models, n_samples).
    """
    raw_splits = {
        "Train": (predictions_train, y_train),
        "Val":   (predictions_val,   y_val),
        "Test":  (predictions_test,  y_test),
    }

    splits = {
        name: compute_split_curves(_to_1d(y), mean_prob(preds))
        for name, (preds, y) in raw_splits.items()
    }

    print_summary(splits)
    plot_comparison(splits)


if __name__ == "__main__":
    # Variables must be defined in the outer scope before calling run()
    run(predictions_train, y_train,
        predictions_val,   y_val,
        predictions_test,  y_test)

## 7. Cross-Split Comparison with Bootstrap 95% CI

Bootstrap percentile 95% CI for ROC, PR, and calibration curves.

In [ ]:
"""
ROC / PR / Calibration Curve Comparison  —  with Bootstrap 95% CI
====================================================================
Adds Bootstrap Percentile 95% CI to all three panels:
  · ROC curve  : AUC-ROC (95% CI) + TPR confidence band
  · PR  curve  : AP / AUC-PR (95% CI) + Precision confidence band
  · Calibration: Brier Score (95% CI) + per-bin fraction-of-positives CI bars

Fully compatible with the original run() signature:
    run(predictions_train, y_train,
        predictions_val,   y_val,
        predictions_test,  y_test)

Each predictions_* : (n_models, n_samples)  — bootstrap ensemble output matrix
"""

from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

# ═══════════════════════════════════════════════════════════════════════════
# § 0  Global configuration
# ═══════════════════════════════════════════════════════════════════════════

OUTPUT_FIG  = "roc_pr_calibration_ci.png"
DPI         = 300
FIGSIZE     = (15, 5)
CAL_BINS    = 10
N_BOOT      = 200          # Bootstrap resampling iterations
ALPHA_CI    = 0.05          # Two-sided 95% CI
ROC_GRID    = np.linspace(0, 1, 200)   # Interpolation FPR grid (ROC band)
REC_GRID    = np.linspace(0, 1, 200)   # Interpolation Recall grid (PR band)
RANDOM_SEED = 2026

# Visual style per dataset split
SPLIT_STYLE = {
    "Train": dict(color="#2196F3", ls="-",  lw=2.2, alpha=0.90, band_alpha=0.12),
    "Val":   dict(color="#FF9800", ls="--", lw=2.2, alpha=0.90, band_alpha=0.14),
    "Test":  dict(color="#E91E63", ls="-.", lw=2.2, alpha=0.90, band_alpha=0.16),
}

SPLIT_LABEL = {
    "Train": "Training Set",
    "Val":   "Internal Validation Set",
    "Test":  "Temporal Validation Set",
}


# ═══════════════════════════════════════════════════════════════════════════
# § 1  Utility functions
# ═══════════════════════════════════════════════════════════════════════════

def _to_1d(arr) -> np.ndarray:
    return np.asarray(arr).reshape(-1)


def mean_prob(predictions: np.ndarray) -> np.ndarray:
    """(n_models, n_samples) → ensemble mean probability (n_samples,)"""
    return predictions.mean(axis=0)


def _interp_tpr(fpr_grid, fpr, tpr) -> np.ndarray:
    """Interpolate ROC curve onto a fixed FPR grid (monotone guarantee)."""
    return np.interp(fpr_grid, fpr, tpr)


def _interp_precision(rec_grid, recall, precision) -> np.ndarray:
    """
    Interpolate PR curve onto a fixed Recall grid.
    sklearn's precision_recall_curve returns recall in descending order;
    flip before interpolation.
    """
    recall_asc    = recall[::-1]
    precision_asc = precision[::-1]
    return np.interp(rec_grid, recall_asc, precision_asc)


# ═══════════════════════════════════════════════════════════════════════════
# § 2  Bootstrap CI computation
# ═══════════════════════════════════════════════════════════════════════════

def _bootstrap_roc(y_true: np.ndarray,
                   prob:   np.ndarray,
                   n_boot: int  = N_BOOT,
                   seed:   int  = RANDOM_SEED,
                   alpha:  float = ALPHA_CI) -> dict:
    """
    Bootstrap the ROC curve. Returns:
      auc_lo, auc_hi  — 95% CI for AUC-ROC
      tpr_lo, tpr_hi  — 95% CI band for interpolated TPR on ROC_GRID
    """
    rng   = np.random.default_rng(seed)
    n     = len(y_true)
    aucs  = []
    tprs  = []   # (n_boot, len(ROC_GRID))

    for _ in range(n_boot):
        idx  = rng.integers(0, n, n)
        yt_b = y_true[idx]
        yp_b = prob[idx]
        if len(np.unique(yt_b)) < 2:
            continue
        fpr_b, tpr_b, _ = roc_curve(yt_b, yp_b)
        aucs.append(roc_auc_score(yt_b, yp_b))
        tprs.append(_interp_tpr(ROC_GRID, fpr_b, tpr_b))

    tprs_arr = np.array(tprs)          # (B, 200)
    lo_p, hi_p = alpha / 2 * 100, (1 - alpha / 2) * 100

    return {
        "auc_lo":  float(np.percentile(aucs, lo_p)),
        "auc_hi":  float(np.percentile(aucs, hi_p)),
        "tpr_lo":  np.percentile(tprs_arr, lo_p, axis=0),
        "tpr_hi":  np.percentile(tprs_arr, hi_p, axis=0),
    }


def _bootstrap_pr(y_true: np.ndarray,
                  prob:   np.ndarray,
                  n_boot: int  = N_BOOT,
                  seed:   int  = RANDOM_SEED,
                  alpha:  float = ALPHA_CI) -> dict:
    """
    Bootstrap the PR curve. Returns:
      ap_lo, ap_hi      — 95% CI for Average Precision (AUC-PR)
      prec_lo, prec_hi  — 95% CI band for interpolated Precision on REC_GRID
    """
    rng   = np.random.default_rng(seed)
    n     = len(y_true)
    aps   = []
    precs = []

    for _ in range(n_boot):
        idx  = rng.integers(0, n, n)
        yt_b = y_true[idx]
        yp_b = prob[idx]
        if len(np.unique(yt_b)) < 2:
            continue
        prec_b, rec_b, _ = precision_recall_curve(yt_b, yp_b)
        aps.append(average_precision_score(yt_b, yp_b))
        precs.append(_interp_precision(REC_GRID, rec_b, prec_b))

    precs_arr = np.array(precs)
    lo_p, hi_p = alpha / 2 * 100, (1 - alpha / 2) * 100

    return {
        "ap_lo":   float(np.percentile(aps, lo_p)),
        "ap_hi":   float(np.percentile(aps, hi_p)),
        "prec_lo": np.percentile(precs_arr, lo_p, axis=0),
        "prec_hi": np.percentile(precs_arr, hi_p, axis=0),
    }


def _bootstrap_calibration(y_true:    np.ndarray,
                            prob:      np.ndarray,
                            n_bins:    int   = CAL_BINS,
                            n_boot:    int   = N_BOOT,
                            seed:      int   = RANDOM_SEED,
                            alpha:     float = ALPHA_CI) -> dict:
    """
    Bootstrap the calibration (reliability) curve. Returns:
      brier_lo, brier_hi      — 95% CI for Brier Score
      frac_lo[k], frac_hi[k]  — per-bin 95% CI for fraction_of_positives
      mean_pred, frac_pos     — point-estimate calibration curve (x, y)
    """
    rng    = np.random.default_rng(seed)
    n      = len(y_true)
    briers = []
    # Per-bootstrap storage of per-bin fraction_of_positives (NaN if bin empty)
    frac_matrix   = []   # (B, n_bins)
    mean_p_matrix = []

    for _ in range(n_boot):
        idx  = rng.integers(0, n, n)
        yt_b = y_true[idx]
        yp_b = prob[idx]
        if len(np.unique(yt_b)) < 2:
            continue
        briers.append(brier_score_loss(yt_b, yp_b))
        try:
            fp_b, mp_b = calibration_curve(yt_b, yp_b,
                                           n_bins=n_bins,
                                           strategy="uniform")
            # Align to fixed bin centres (uniform strategy → fixed centres)
            bin_centers = np.linspace(1 / (2 * n_bins), 1 - 1 / (2 * n_bins), n_bins)
            fp_aligned  = np.full(n_bins, np.nan)
            mp_aligned  = np.full(n_bins, np.nan)
            for i, bc in enumerate(bin_centers):
                lo_e = bc - 1 / (2 * n_bins)
                hi_e = bc + 1 / (2 * n_bins)
                mask = (mp_b >= lo_e) & (mp_b < hi_e)
                if mask.any():
                    fp_aligned[i] = fp_b[mask].mean()
                    mp_aligned[i] = mp_b[mask].mean()
            frac_matrix.append(fp_aligned)
            mean_p_matrix.append(mp_aligned)
        except Exception:
            frac_matrix.append(np.full(n_bins, np.nan))
            mean_p_matrix.append(np.full(n_bins, np.nan))

    frac_arr = np.array(frac_matrix)     # (B, n_bins)
    lo_p, hi_p = alpha / 2 * 100, (1 - alpha / 2) * 100

    # Per-bin CI (NaN-safe)
    frac_lo = np.nanpercentile(frac_arr, lo_p, axis=0)
    frac_hi = np.nanpercentile(frac_arr, hi_p, axis=0)

    # Point-estimate calibration curve (for main line)
    frac_pos_pt, mean_pred_pt = calibration_curve(y_true, prob,
                                                   n_bins=n_bins,
                                                   strategy="uniform")
    return {
        "brier_lo":  float(np.percentile(briers, lo_p)),
        "brier_hi":  float(np.percentile(briers, hi_p)),
        "frac_lo":   frac_lo,
        "frac_hi":   frac_hi,
        "mean_pred": mean_pred_pt,
        "frac_pos":  frac_pos_pt,
    }


# ═══════════════════════════════════════════════════════════════════════════
# § 3  Per-split curve computation
# ═══════════════════════════════════════════════════════════════════════════

def compute_split_curves(y_true: np.ndarray,
                         prob:   np.ndarray,
                         n_boot: int = N_BOOT,
                         seed:   int = RANDOM_SEED) -> dict:
    """
    Compute ROC / PR / Calibration curves + Bootstrap 95% CI for one split.

    Parameters
    ----------
    y_true : (n,) true binary labels
    prob   : (n,) ensemble mean probabilities
    n_boot : number of Bootstrap resamples

    Returns
    -------
    dict with keys:
        roc      : (fpr, tpr, auc)          — point-estimate curve
        roc_ci   : {auc_lo, auc_hi, tpr_lo, tpr_hi}
        pr       : (recall, precision, ap)  — point-estimate curve
        pr_ci    : {ap_lo, ap_hi, prec_lo, prec_hi}
        cal      : (mean_pred, frac_pos, brier)
        cal_ci   : {brier_lo, brier_hi, frac_lo, frac_hi, mean_pred, frac_pos}
        prob     : raw ensemble probabilities
    """
    y_true = _to_1d(y_true)
    prob   = _to_1d(prob)

    print(f"    Computing Bootstrap CI (B={n_boot})...", end=" ", flush=True)

    # ── Point estimates ──────────────────────────────────────────────
    fpr, tpr, _         = roc_curve(y_true, prob)
    auc_val             = roc_auc_score(y_true, prob)
    prec, rec, _        = precision_recall_curve(y_true, prob)
    ap_val              = average_precision_score(y_true, prob)
    frac_pos, mean_pred = calibration_curve(y_true, prob,
                                            n_bins=CAL_BINS, strategy="uniform")
    brier_val           = brier_score_loss(y_true, prob)

    # ── Bootstrap CI ─────────────────────────────────────────────────
    roc_ci = _bootstrap_roc(y_true, prob, n_boot=n_boot, seed=seed)
    pr_ci  = _bootstrap_pr( y_true, prob, n_boot=n_boot, seed=seed)
    cal_ci = _bootstrap_calibration(y_true, prob, n_boot=n_boot, seed=seed)

    print("done")

    return {
        "roc":    (fpr, tpr, auc_val),
        "roc_ci": roc_ci,
        "pr":     (rec, prec, ap_val),
        "pr_ci":  pr_ci,
        "cal":    (mean_pred, frac_pos, brier_val),
        "cal_ci": cal_ci,
        "prob":   prob,
    }


# ═══════════════════════════════════════════════════════════════════════════
# § 4  Sub-plot renderers
# ═══════════════════════════════════════════════════════════════════════════

def _draw_roc(ax, splits: dict) -> None:
    """ROC curves + 95% CI confidence bands."""
    ax.plot([0, 1], [0, 1], "k--", lw=1.2, alpha=0.45, label="Random (AUC=0.500)")
    ax.fill_between([0, 1], [0, 1], alpha=0.03, color="gray")

    for name, data in splits.items():
        fpr, tpr, auc = data["roc"]
        ci  = data["roc_ci"]
        sty = SPLIT_STYLE[name]
        lbl = SPLIT_LABEL[name]

        ax.plot(fpr, tpr,
                color=sty["color"], ls=sty["ls"], lw=sty["lw"], alpha=sty["alpha"],
                label=f"{lbl}  AUC={auc:.3f}\n"
                      f"        (95%CI: {ci['auc_lo']:.3f}–{ci['auc_hi']:.3f})",
                zorder=3)

        # Confidence band on fixed FPR grid
        ax.fill_between(ROC_GRID,
                        ci["tpr_lo"], ci["tpr_hi"],
                        color=sty["color"], alpha=sty["band_alpha"],
                        linewidth=0, zorder=2)

    ax.set(title="ROC Curve (95% CI)",
           xlabel="False Positive Rate (1 − Specificity)",
           ylabel="True Positive Rate (Sensitivity)",
           xlim=[-0.02, 1.02], ylim=[-0.02, 1.02])
    ax.legend(loc="lower right", fontsize=8.5, framealpha=0.9)
    ax.grid(alpha=0.28)
    _add_ci_note(ax)


def _draw_pr(ax, splits: dict) -> None:
    """PR curves + 95% CI confidence bands."""
    for name, data in splits.items():
        rec, prec, ap = data["pr"]
        ci  = data["pr_ci"]
        sty = SPLIT_STYLE[name]
        lbl = SPLIT_LABEL[name]

        ax.plot(rec, prec,
                color=sty["color"], ls=sty["ls"], lw=sty["lw"], alpha=sty["alpha"],
                label=f"{lbl}  AP={ap:.3f}\n"
                      f"        (95%CI: {ci['ap_lo']:.3f}–{ci['ap_hi']:.3f})",
                zorder=3)

        # Confidence band on fixed Recall grid
        ax.fill_between(REC_GRID,
                        ci["prec_lo"], ci["prec_hi"],
                        color=sty["color"], alpha=sty["band_alpha"],
                        linewidth=0, zorder=2)

    ax.set(title="Precision-Recall Curve (95% CI)",
           xlabel="Recall (Sensitivity)",
           ylabel="Precision (PPV)",
           xlim=[-0.02, 1.02], ylim=[-0.02, 1.02])
    ax.legend(loc="upper right", fontsize=8.5, framealpha=0.9)
    ax.grid(alpha=0.28)
    _add_ci_note(ax)


def _draw_calibration(ax, splits: dict) -> None:
    """
    Reliability diagram + per-bin 95% CI error bars + Brier Score CI.
    No histogram panel.
    """
    ax.plot([0, 1], [0, 1], "k--", lw=1.2, alpha=0.45,
            label="Perfect calibration")

    for name, data in splits.items():
        mean_pred, frac_pos, brier = data["cal"]
        ci  = data["cal_ci"]
        sty = SPLIT_STYLE[name]
        lbl = SPLIT_LABEL[name]

        # Point-estimate calibration line
        ax.plot(mean_pred, frac_pos,
                color=sty["color"], ls=sty["ls"], lw=sty["lw"], alpha=sty["alpha"],
                marker="o", markersize=6, zorder=4,
                label=f"{lbl}  Brier={brier:.4f}\n"
                      f"        (95%CI: {ci['brier_lo']:.4f}–{ci['brier_hi']:.4f})")

        # Per-bin vertical CI error bars
        valid = ~np.isnan(ci["frac_lo"]) & ~np.isnan(ci["frac_hi"])
        if valid.any():
            n_pt = len(mean_pred)
            n_ci = len(ci["frac_lo"])
            m    = min(n_pt, n_ci)
            v    = valid[:m]
            x_err = mean_pred[:m][v]
            lo_   = ci["frac_lo"][:m][v]
            hi_   = ci["frac_hi"][:m][v]

            ax.errorbar(x_err,
                        (lo_ + hi_) / 2,
                        yerr=[(lo_ + hi_) / 2 - lo_,
                              hi_ - (lo_ + hi_) / 2],
                        fmt="none",
                        ecolor=sty["color"], elinewidth=1.4,
                        capsize=4, capthick=1.4, alpha=0.60, zorder=3)

    ax.set(title="Calibration Curve — Reliability Diagram (95% CI)",
           xlabel="Mean Predicted Probability",
           ylabel="Fraction of Positives",
           xlim=[-0.02, 1.02], ylim=[-0.02, 1.02])
    ax.legend(loc="upper left", fontsize=8.5, framealpha=0.9)
    ax.grid(alpha=0.28)
    _add_ci_note(ax)


def _add_ci_note(ax) -> None:
    """Add a small Bootstrap CI annotation in the lower-right corner."""
    ax.text(0.98, 0.03,
            f"Bootstrap 95% CI  (B={N_BOOT})",
            transform=ax.transAxes,
            fontsize=7, color="gray",
            ha="right", va="bottom",
            style="italic")


# ═══════════════════════════════════════════════════════════════════════════
# § 5  Main rendering function
# ═══════════════════════════════════════════════════════════════════════════

def plot_comparison(splits: dict, output_path: str = OUTPUT_FIG) -> None:
    """
    1×3 panel: ROC | PR | Calibration Curve (no histogram).
    """
    fig, axes = plt.subplots(1, 3, figsize=FIGSIZE, facecolor="white")
    # fig.suptitle(
    #     "Bootstrap Ensemble Model (300 XGBoost)  —  "
    #     "Training / Internal Validation / Temporal Validation\n"
    #     "with Bootstrap 95% Confidence Intervals",
    #     fontsize=12, fontweight="bold", y=1.03,
    # )

    ax_roc, ax_pr, ax_cal = axes

    _draw_roc(ax_roc, splits)
    _draw_pr( ax_pr,  splits)
    _draw_calibration(ax_cal, splits)

    for ax in axes:
        ax.spines[["top", "right"]].set_visible(False)

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")
    print(f"\nFigure saved → {output_path}")
    plt.close(fig)


# ═══════════════════════════════════════════════════════════════════════════
# § 6  Console summary
# ═══════════════════════════════════════════════════════════════════════════

def print_summary(splits: dict) -> None:
    """Print a metric summary table with 95% CI across all splits."""
    print("\n" + "═" * 72)
    print("  Performance Summary  —  Bootstrap 95% CI")
    print("═" * 72)
    hdr = (f"  {'Split':<24}  {'AUC-ROC':>7}  {'95%CI':>16}  "
           f"{'AP(PR)':>7}  {'95%CI':>16}  {'Brier':>7}  {'95%CI':>16}")
    print(hdr)
    print("  " + "─" * 90)

    for name, data in splits.items():
        auc   = data["roc"][2]
        ap    = data["pr"][2]
        brier = data["cal"][2]
        rci   = data["roc_ci"]
        pci   = data["pr_ci"]
        cci   = data["cal_ci"]
        lbl   = SPLIT_LABEL[name]
        print(
            f"  {lbl:<24}  {auc:>7.4f}  "
            f"[{rci['auc_lo']:.4f},{rci['auc_hi']:.4f}]  "
            f"{ap:>7.4f}  "
            f"[{pci['ap_lo']:.4f},{pci['ap_hi']:.4f}]  "
            f"{brier:>7.4f}  "
            f"[{cci['brier_lo']:.4f},{cci['brier_hi']:.4f}]"
        )
    print("═" * 72)
    print(f"  Bootstrap: Percentile method, B={N_BOOT}, α=0.05 (two-sided)")
    print("═" * 72)


# ═══════════════════════════════════════════════════════════════════════════
# § 7  Entry point  (fully compatible with original run() signature)
# ═══════════════════════════════════════════════════════════════════════════

def run(predictions_train, y_train,
        predictions_val,   y_val,
        predictions_test,  y_test,
        output_path: str = OUTPUT_FIG) -> dict:
    """
    Compute curves + 95% CI for all three splits, render and save figure.

    Parameters
    ----------
    predictions_* : (n_models, n_samples) — ensemble member probability matrices
    y_*           : (n_samples,)          — true binary labels
    output_path   : output image file path

    Returns
    -------
    splits dict (available for downstream analysis)
    """
    raw_splits = {
        "Train": (predictions_train, y_train),
        "Val":   (predictions_val,   y_val),
        "Test":  (predictions_test,  y_test),
    }

    print("\n" + "═" * 56)
    print("  ROC / PR / Calibration  —  Bootstrap 95% CI")
    print("═" * 56)

    splits = {}
    for name, (preds, y) in raw_splits.items():
        print(f"\n  [{SPLIT_LABEL[name]}]  n={len(_to_1d(y))}  "
              f"events={int(_to_1d(y).sum())}  "
              f"prevalence={_to_1d(y).mean()*100:.1f}%")
        splits[name] = compute_split_curves(
            _to_1d(y), mean_prob(np.asarray(preds))
        )

    print_summary(splits)
    plot_comparison(splits, output_path=output_path)

    return splits


# ═══════════════════════════════════════════════════════════════════════════
# § 8  Self-validation (simulated data)
# ═══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    import numpy as _np

    _rng = _np.random.default_rng(42)
    N_MODELS = 300

    def _sim(n):
        age   = _rng.integers(15, 50, n).astype(float)
        resid = _rng.choice([0, 1], n).astype(float)
        logit = 0.05 * age - 0.3 * resid + _rng.normal(0, 0.6, n)
        prob  = 1 / (1 + _np.exp(-logit))
        y     = _rng.binomial(1, prob).astype(int)
        noise = _rng.normal(0, 0.05, (N_MODELS, n))
        mat   = _np.clip(prob[None, :] + noise, 1e-4, 1 - 1e-4)
        return mat, y

    pred_tr, y_tr = _sim(1000)
    pred_va, y_va = _sim(400)
    pred_te, y_te = _sim(400)

    results =run(predictions_train, y_train,
        predictions_val,   y_val,
        predictions_test,  y_test)

## 8. Risk Stratification Scatter Plot

4-zone plot: predicted probability × uncertainty (TU).

In [ ]:
"""
Risk Stratification Scatter Plot
==================================
Stratify samples into 4 zones based on:
  - Predicted probability  (x-axis) → clinical risk level
  - TU uncertainty         (y-axis) → model confidence

Zones
-----
  High Risk   + Certain     : high prob,  low  TU  → Act (green border)
  High Risk   + Uncertain   : high prob,  high TU  → Review (orange)
  Low Risk    + Certain     : low  prob,  low  TU  → Monitor (blue)
  Low Risk    + Uncertain   : low  prob,  high TU  → Flag (purple)

Thresholds (configurable)
--------------------------
  PROB_THRESHOLD : predicted probability cut-off  (default 0.5)
  TU_QUANTILE    : TU percentile used as cut-off  (default 0.5 → median)
"""

from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from typing import Optional

# ─────────────────────────────────────────────
# Configuration
# ─────────────────────────────────────────────
OUTPUT_FIG     = "risk_stratification.png"
DPI            = 300
FIGSIZE        = (10, 8)

PROB_THRESHOLD = 0.5    # x-axis split
TU_QUANTILE    = 0.5    # y-axis split (percentile of TU)

# Zone definitions  {label: (x_side, y_side, color, marker, priority)}
#   x_side: "high" = prob >= threshold, "low" = prob < threshold
#   y_side: "low"  = TU  <  threshold (certain), "high" = TU >= threshold
ZONE_CFG = {
    "High Risk · Certain":   dict(x="high", y="low",  color="#E53935", marker="o", z=4),
    "High Risk · Uncertain": dict(x="high", y="high", color="#FF8F00", marker="^", z=3),
    "Low Risk · Uncertain":  dict(x="low",  y="high", color="#7B1FA2", marker="s", z=2),
    "Low Risk · Certain":    dict(x="low",  y="low",  color="#1565C0", marker="D", z=1),
}

ALPHA      = 0.72
POINT_SIZE = 55

# ─────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────

def _to_1d(arr) -> np.ndarray:
    return np.asarray(arr).reshape(-1)


def assign_zones(prob: np.ndarray,
                 TU: np.ndarray,
                 prob_thr: float,
                 tu_thr: float) -> np.ndarray:
    """
    Return an integer zone label per sample.

        0 — High Risk · Certain
        1 — High Risk · Uncertain
        2 — Low Risk  · Uncertain
        3 — Low Risk  · Certain
    """
    high_prob = prob >= prob_thr
    high_tu   = TU   >= tu_thr

    zones = np.empty(len(prob), dtype=int)
    zones[ high_prob & ~high_tu] = 0   # High Risk · Certain
    zones[ high_prob &  high_tu] = 1   # High Risk · Uncertain
    zones[~high_prob &  high_tu] = 2   # Low Risk  · Uncertain
    zones[~high_prob & ~high_tu] = 3   # Low Risk  · Certain
    return zones


def compute_zone_stats(y_true: np.ndarray,
                       zones: np.ndarray) -> dict:
    """Count n, positive rate per zone."""
    stats = {}
    names = list(ZONE_CFG.keys())
    for i, name in enumerate(names):
        mask = zones == i
        n    = mask.sum()
        pos_rate = y_true[mask].mean() if n > 0 else float("nan")
        stats[name] = {"n": int(n), "pos_rate": pos_rate, "mask": mask}
    return stats

# ─────────────────────────────────────────────
# Plotting
# ─────────────────────────────────────────────

def _background_quads(ax, prob_thr: float, tu_thr: float,
                      x_lim: tuple, y_lim: tuple) -> None:
    """Fill four quadrant backgrounds with low-alpha colors."""
    bg_alpha = 0.07
    x0, x1 = x_lim
    y0, y1 = y_lim

    fills = [
        # (x_lo, x_hi, y_lo, y_hi, color)
        (prob_thr, x1,   y0, tu_thr,  "#E53935"),  # High Risk · Certain
        (prob_thr, x1,   tu_thr, y1,  "#FF8F00"),  # High Risk · Uncertain
        (x0, prob_thr,   tu_thr, y1,  "#7B1FA2"),  # Low Risk  · Uncertain
        (x0, prob_thr,   y0, tu_thr,  "#1565C0"),  # Low Risk  · Certain
    ]
    for xlo, xhi, ylo, yhi, color in fills:
        ax.fill([xlo, xhi, xhi, xlo],
                [ylo, ylo, yhi, yhi],
                color=color, alpha=bg_alpha, zorder=0)


def _zone_labels(ax, prob_thr: float, tu_thr: float,
                 x_lim: tuple, y_lim: tuple) -> None:
    """Place zone annotation text in each quadrant corner."""
    pad_x = (x_lim[1] - x_lim[0]) * 0.02
    pad_y = (y_lim[1] - y_lim[0]) * 0.02

    annotations = [
        # (x, y, ha, va, text, color)
        (x_lim[1] - pad_x, y_lim[0] + pad_y,  "right", "bottom",
         "High Risk\nCertain",   "#E53935"),
        (x_lim[1] - pad_x, y_lim[1] - pad_y,  "right", "top",
         "High Risk\nUncertain", "#E65100"),
        (x_lim[0] + pad_x, y_lim[1] - pad_y,  "left",  "top",
         "Low Risk\nUncertain",  "#6A1B9A"),
        (x_lim[0] + pad_x, y_lim[0] + pad_y,  "left",  "bottom",
         "Low Risk\nCertain",    "#0D47A1"),
    ]
    for x, y, ha, va, txt, col in annotations:
        ax.text(x, y, txt, ha=ha, va=va, fontsize=8.5, color=col,
                alpha=0.55, fontstyle="italic",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.0))


def plot_risk_stratification(prob: np.ndarray,
                             TU: np.ndarray,
                             y_true: Optional[np.ndarray] = None,
                             prob_thr: float = PROB_THRESHOLD,
                             tu_quantile: float = TU_QUANTILE,
                             split_name: str = "Dataset",
                             output_path: str = OUTPUT_FIG) -> None:
    """
    Render the risk-stratification scatter plot.

    Parameters
    ----------
    prob        : ensemble predicted probabilities  (n_samples,)
    TU          : total uncertainty values          (n_samples,)
    y_true      : optional true labels — if given, markers are outlined
                  by correctness (correct=black edge, wrong=red edge)
    prob_thr    : probability threshold for risk split
    tu_quantile : TU percentile used as uncertainty threshold
    split_name  : label shown in title (e.g. "Test")
    output_path : file path for saved figure
    """
    prob   = _to_1d(prob)
    TU     = _to_1d(TU)
    tu_thr = float(np.percentile(TU, tu_quantile * 100))

    zones      = assign_zones(prob, TU, prob_thr, tu_thr)
    zone_stats = compute_zone_stats(
        _to_1d(y_true) if y_true is not None else np.zeros_like(prob),
        zones
    )

    # ── figure setup ──────────────────────────
    fig, ax = plt.subplots(figsize=FIGSIZE)

    x_pad = (prob.max() - prob.min()) * 0.05
    y_pad = (TU.max()   - TU.min())   * 0.05
    x_lim = (prob.min() - x_pad, prob.max() + x_pad)
    y_lim = (TU.min()   - y_pad, TU.max()   + y_pad)

    _background_quads(ax, prob_thr, tu_thr, x_lim, y_lim)
    _zone_labels(ax, prob_thr, tu_thr, x_lim, y_lim)

    # ── threshold lines ────────────────────────
    ax.axvline(prob_thr, color="gray", lw=1.4, ls="--", alpha=0.7, zorder=1)
    ax.axhline(tu_thr,   color="gray", lw=1.4, ls="--", alpha=0.7, zorder=1)
    ax.text(prob_thr + 0.005, y_lim[1] * 0.98,
            f"p = {prob_thr:.2f}", color="gray", fontsize=8, va="top")
    ax.text(x_lim[1] * 0.98, tu_thr + (y_lim[1] - y_lim[0]) * 0.01,
            f"TU = {tu_thr:.3f}", color="gray", fontsize=8, ha="right")

    # ── scatter per zone ──────────────────────
    names = list(ZONE_CFG.keys())
    for i, name in enumerate(names):
        cfg  = ZONE_CFG[name]
        mask = zones == i
        if not mask.any():
            continue

        # Edge color by correctness (if y_true provided)
        if y_true is not None:
            yt = _to_1d(y_true)
            ec = np.where(yt[mask] == (prob[mask] >= prob_thr).astype(int),
                          "black", "#FF1744")
            ec_kw = dict(edgecolors=ec, linewidths=0.8)
        else:
            ec_kw = dict(edgecolors="white", linewidths=0.5)

        ax.scatter(prob[mask], TU[mask],
                   c=cfg["color"], marker=cfg["marker"],
                   s=POINT_SIZE, alpha=ALPHA, zorder=cfg["z"] + 1,
                   label=f"{name}  (n={mask.sum()})",
                   **ec_kw)

    # ── legend ────────────────────────────────
    legend_handles = []
    for name, cfg in ZONE_CFG.items():
        stats = zone_stats[name]
        lbl = (f"{name}  "
               f"n={stats['n']}  "
               f"({stats['pos_rate']*100:.1f}% pos)"
               if y_true is not None
               else f"{name}  n={stats['n']}")
        legend_handles.append(
            Line2D([0], [0], marker=cfg["marker"], color="w",
                   markerfacecolor=cfg["color"], markersize=9,
                   alpha=0.9, label=lbl)
        )

    if y_true is not None:
        legend_handles += [
            Line2D([0], [0], marker="o", color="w", markerfacecolor="gray",
                   markeredgecolor="black", markersize=8, label="Correct prediction"),
            Line2D([0], [0], marker="o", color="w", markerfacecolor="gray",
                   markeredgecolor="#FF1744", markersize=8, label="Wrong prediction"),
        ]

    ax.legend(handles=legend_handles, loc="upper center",
              bbox_to_anchor=(0.5, -0.12), ncol=2,
              fontsize=9, framealpha=0.9)

    # ── axes formatting ───────────────────────
    ax.set(title=f"Risk Stratification — {split_name}\n"
                 f"(prob threshold={prob_thr:.2f}, "
                 f"TU threshold={tu_thr:.3f} [{tu_quantile*100:.0f}th pct])",
           xlabel="Ensemble Predicted Probability",
           ylabel="Total Uncertainty (TU — Predictive Entropy)",
           xlim=x_lim, ylim=y_lim)
    ax.grid(alpha=0.2, zorder=0)

    # ── zone count annotation (corner box) ───
    summary_lines = [f"Zone distribution ({split_name})"]
    for name in names:
        n   = zone_stats[name]["n"]
        pct = n / len(prob) * 100
        summary_lines.append(f"  {name:<28} {n:>4}  ({pct:.1f}%)")
    ax.text(0.01, 0.99, "\n".join(summary_lines),
            transform=ax.transAxes, fontsize=7.5,
            va="top", ha="left",
            bbox=dict(boxstyle="round,pad=0.4", fc="white", alpha=0.85, ec="gray"))

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")
    print(f"Figure saved: {output_path}")
    plt.show()
    plt.close(fig)

# ─────────────────────────────────────────────
# Multi-split comparison (optional)
# ─────────────────────────────────────────────

def plot_multi_split(splits: dict,
                     prob_thr: float = PROB_THRESHOLD,
                     tu_quantile: float = TU_QUANTILE,
                     output_path: str = "risk_stratification_multi.png") -> None:
    """
    Side-by-side risk stratification for multiple splits.

    Parameters
    ----------
    splits : dict[str, dict]
        Each entry: {"prob": array, "TU": array, "y_true": array (optional)}
    """
    n      = len(splits)
    fig, axes = plt.subplots(1, n, figsize=(FIGSIZE[0] * n / 2, FIGSIZE[1]),
                             sharey=False)
    if n == 1:
        axes = [axes]

    # Shared TU threshold across all splits for comparability
    all_TU  = np.concatenate([d["TU"] for d in splits.values()])
    tu_thr  = float(np.percentile(all_TU, tu_quantile * 100))

    for ax, (name, data) in zip(axes, splits.items()):
        prob   = _to_1d(data["prob"])
        TU     = _to_1d(data["TU"])
        y_true = _to_1d(data["y_true"]) if "y_true" in data else None

        zones      = assign_zones(prob, TU, prob_thr, tu_thr)
        zone_stats = compute_zone_stats(
            y_true if y_true is not None else np.zeros_like(prob), zones)

        x_pad = (prob.max() - prob.min()) * 0.05
        y_pad = (TU.max()   - TU.min())   * 0.05
        x_lim = (prob.min() - x_pad, prob.max() + x_pad)
        y_lim = (TU.min()   - y_pad, TU.max()   + y_pad)

        _background_quads(ax, prob_thr, tu_thr, x_lim, y_lim)
        _zone_labels(ax, prob_thr, tu_thr, x_lim, y_lim)

        ax.axvline(prob_thr, color="gray", lw=1.2, ls="--", alpha=0.6)
        ax.axhline(tu_thr,   color="gray", lw=1.2, ls="--", alpha=0.6)

        names_list = list(ZONE_CFG.keys())
        for i, zone_name in enumerate(names_list):
            cfg  = ZONE_CFG[zone_name]
            mask = zones == i
            if not mask.any():
                continue
            if y_true is not None:
                ec = np.where(y_true[mask] == (prob[mask] >= prob_thr).astype(int),
                              "black", "#FF1744")
                ec_kw = dict(edgecolors=ec, linewidths=0.7)
            else:
                ec_kw = dict(edgecolors="white", linewidths=0.4)

            ax.scatter(prob[mask], TU[mask],
                       c=cfg["color"], marker=cfg["marker"],
                       s=POINT_SIZE * 0.8, alpha=ALPHA, zorder=cfg["z"] + 1,
                       **ec_kw)

        # Zone count inset
        lines = []
        for i, zone_name in enumerate(names_list):
            n_z  = zone_stats[zone_name]["n"]
            pct  = n_z / len(prob) * 100
            lines.append(f"{zone_name.split('·')[0].strip()}\n"
                         f"  {zone_name.split('·')[1].strip()}  "
                         f"n={n_z} ({pct:.0f}%)")
        ax.text(0.02, 0.98, "\n".join(lines),
                transform=ax.transAxes, fontsize=7,
                va="top", bbox=dict(boxstyle="round", fc="white", alpha=0.8, ec="gray"))

        ax.set(title=f"{name}",
               xlabel="Predicted Probability",
               ylabel="TU (Predictive Entropy)" if ax is axes[0] else "",
               xlim=x_lim, ylim=y_lim)
        ax.grid(alpha=0.2)

    fig.suptitle(
        f"Risk Stratification Comparison  "
        f"(prob_thr={prob_thr:.2f}, TU_thr={tu_thr:.3f} [{tu_quantile*100:.0f}th pct])",
        fontsize=13, fontweight="bold", y=1.02,
    )

    # Shared legend below
    legend_handles = [
        Line2D([0], [0], marker=cfg["marker"], color="w",
               markerfacecolor=cfg["color"], markersize=9,
               alpha=0.9, label=zone_name)
        for zone_name, cfg in ZONE_CFG.items()
    ]
    fig.legend(handles=legend_handles, loc="lower center",
               bbox_to_anchor=(0.5, -0.06), ncol=4, fontsize=9, framealpha=0.9)

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")
    print(f"Figure saved: {output_path}")
    plt.show()
    plt.close(fig)

# ─────────────────────────────────────────────
# Entry point
# ─────────────────────────────────────────────

def run(predictions_train, y_train,
        predictions_val,   y_val,
        predictions_test,  y_test,
        compute_uncertainties_fn) -> None:
    """
    Compute uncertainties and render risk stratification figures.

    Parameters
    ----------
    predictions_* : (n_models, n_samples)
    compute_uncertainties_fn : callable returning dict with 'mean_pred', 'TU'
    """
    splits_raw = {
        "Train": (predictions_train, y_train),
        "Val":   (predictions_val,   y_val),
        "Test":  (predictions_test,  y_test),
    }

    splits = {}
    for name, (preds, y) in splits_raw.items():
        unc = compute_uncertainties_fn(preds)
        splits[name] = {
            "prob":   unc["mean_pred"],
            "TU":     unc["TU"],
            "y_true": _to_1d(y),
        }

    # Single test-set plot
    plot_risk_stratification(
        prob=splits["Test"]["prob"],
        TU=splits["Test"]["TU"],
        y_true=splits["Test"]["y_true"],
        split_name="Test",
        output_path="risk_stratification_test.png",
    )

    # Multi-split comparison
    plot_multi_split(splits, output_path="risk_stratification_multi.png")


if __name__ == "__main__":
    run(predictions_train, y_train,
        predictions_val,   y_val,
        predictions_test,  y_test,
        compute_uncertainties)   # pass your uncertainty function here

## 9. Risk Stratification — Enhanced with Clinical Metrics

Uncertainty-aware clinical decision zones:
- **High Risk + Certain** → Act (intervene)
- **High Risk + Uncertain** → Review (gather information)
- **Low Risk + Uncertain** → Flag (monitor closely)
- **Low Risk + Certain** → Monitor (routine follow-up)

Per-zone metrics: observed incidence, PPV, NPV, calibration gap.
Thresholds: prob=0.5 fixed, TU=median of validation set.

In [ ]:
"""
Risk Stratification — Uncertainty-Aware Clinical Decision Zones
================================================================
Stratify predictions into 4 clinical zones using:
  - x-axis: Predicted probability  → risk level
  - y-axis: Total Uncertainty (TU) → model confidence

Zones:
  High Risk + Certain    →  Act     (red)     — strong evidence, intervene
  High Risk + Uncertain  →  Review  (orange)  — gather more information
  Low Risk  + Uncertain  →  Flag    (purple)  — monitor closely
  Low Risk  + Certain    →  Monitor (blue)    — routine follow-up

Per-zone metrics:
  Observed incidence, PPV, NPV, calibration gap (mean pred − observed)

Threshold strategy:
  - Probability threshold: fixed at 0.5 (Methods §2.6)
  - TU threshold: median of validation-set TU (independent of test set)

Outputs:
  youden_roc_{split}.tiff           ROC + error-detection ROC
  risk_stratification_{split}.tiff  Single-split scatter + metrics table
  risk_stratification_multi.tiff    Tri-split comparison panel
  Console metrics report

Usage (after run_evaluation cell has defined compute_uncertainties):
  >>> run_risk_stratification(
  ...     predictions_train, y_train,
  ...     predictions_val,   y_val,
  ...     predictions_test,  y_test)
"""

from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix

# ═══════════════════════════════════════════════════════════════
# § 0  Configuration
# ═══════════════════════════════════════════════════════════════

DPI            = 300
FIGSIZE_SINGLE = (12, 9)
FIGSIZE_MULTI  = (18, 9)
PROB_THRESHOLD = 0.5
TU_QUANTILE    = 0.5        # percentile of val-set TU for threshold

ZONE_CFG = {
    "High Risk · Certain":   dict(x="high", y="low",  color="#E53935", marker="o", z=4),
    "High Risk · Uncertain": dict(x="high", y="high", color="#FF8F00", marker="^", z=3),
    "Low Risk · Uncertain":  dict(x="low",  y="high", color="#7B1FA2", marker="s", z=2),
    "Low Risk · Certain":    dict(x="low",  y="low",  color="#1565C0", marker="D", z=1),
}

ZONE_SHORT = {
    "High Risk · Certain": "HR·C", "High Risk · Uncertain": "HR·U",
    "Low Risk · Uncertain": "LR·U", "Low Risk · Certain": "LR·C",
}

ALPHA = 0.50
POINT_SIZE = 30


# ═══════════════════════════════════════════════════════════════
# § 1  Core Utilities
# ═══════════════════════════════════════════════════════════════

def _1d(arr) -> np.ndarray:
    """Ensure flat numpy array."""
    return np.asarray(arr).reshape(-1)


def _fmt(val, pct=True, sign=False) -> str:
    """Format a value as percentage or float."""
    if np.isnan(val):
        return "—"
    if pct:
        return f"{'+' if sign and val > 0 else ''}{val * 100:.1f}%"
    return f"{val:.3f}"


def assign_zones(prob: np.ndarray, tu: np.ndarray,
                 prob_thr: float, tu_thr: float) -> np.ndarray:
    """Assign each sample to one of 4 zones (0–3)."""
    hi_p = prob >= prob_thr
    hi_t = tu >= tu_thr
    zones = np.empty(len(prob), dtype=int)
    zones[hi_p & ~hi_t] = 0   # High Risk · Certain
    zones[hi_p &  hi_t] = 1   # High Risk · Uncertain
    zones[~hi_p & hi_t] = 2   # Low Risk · Uncertain
    zones[~hi_p & ~hi_t] = 3  # Low Risk · Certain
    return zones


def compute_zone_metrics(y_true: np.ndarray, prob: np.ndarray,
                         zones: np.ndarray, prob_thr: float) -> dict:
    """
    Per-zone: n, observed incidence, mean predicted prob,
    calibration gap, PPV, NPV, confusion matrix counts.
    """
    names = list(ZONE_CFG.keys())
    stats = {}
    for i, name in enumerate(names):
        mask = zones == i
        n = int(mask.sum())
        if n == 0:
            stats[name] = dict(n=0, observed_rate=np.nan, mean_pred=np.nan,
                               calibration_gap=np.nan, PPV=np.nan, NPV=np.nan,
                               TP=0, FP=0, TN=0, FN=0)
            continue
        y_z = y_true[mask]
        p_z = prob[mask]
        pred_z = (p_z >= prob_thr).astype(int)
        TP = int(((pred_z == 1) & (y_z == 1)).sum())
        FP = int(((pred_z == 1) & (y_z == 0)).sum())
        TN = int(((pred_z == 0) & (y_z == 0)).sum())
        FN = int(((pred_z == 0) & (y_z == 1)).sum())
        obs = float(y_z.mean())
        mp = float(p_z.mean())
        stats[name] = dict(
            n=n, observed_rate=obs, mean_pred=mp,
            calibration_gap=mp - obs,
            PPV=TP / (TP + FP) if (TP + FP) > 0 else np.nan,
            NPV=TN / (TN + FN) if (TN + FN) > 0 else np.nan,
            TP=TP, FP=FP, TN=TN, FN=FN,
        )
    return stats


# ═══════════════════════════════════════════════════════════════
# § 2  Youden's J Threshold
# ═══════════════════════════════════════════════════════════════

def youden_threshold(y_true: np.ndarray, prob: np.ndarray,
                     uncertainty_measures: dict | None = None) -> dict:
    """
    Compute Youden-optimal probability threshold from ROC curve.
    NOTE: Currently overridden to fixed 0.5 per Methods §2.6.
    Also computes error-detection ROC for each uncertainty measure.
    """
    y_true = _1d(y_true)
    prob = _1d(prob)
    fpr, tpr, thresholds = roc_curve(y_true, prob)
    auc_val = roc_auc_score(y_true, prob)
    j = tpr - fpr
    idx = int(np.argmax(j))

    result = dict(
        prob_threshold=0.5,  # Fixed per Methods §2.6
        youden_j=float(j[idx]),
        best_tpr=float(tpr[idx]),
        best_specificity=float(1 - fpr[idx]),
        auc=float(auc_val),
        fpr=fpr, tpr=tpr, thresholds=thresholds,
        error_roc={},
    )

    if uncertainty_measures:
        error_labels = (y_true != (prob >= result["prob_threshold"]).astype(int)).astype(int)
        for name, scores in uncertainty_measures.items():
            scores = _1d(scores)
            if scores.std() == 0:
                continue
            efpr, etpr, ethr = roc_curve(error_labels, scores)
            eauc = roc_auc_score(error_labels, scores)
            ej = etpr - efpr
            ei = int(np.argmax(ej))
            result["error_roc"][name] = dict(
                fpr=efpr, tpr=etpr, thresholds=ethr, auc=eauc,
                best_threshold=float(ethr[ei]),
                best_tpr=float(etpr[ei]),
                best_specificity=float(1 - efpr[ei]),
            )

    return result


# ═══════════════════════════════════════════════════════════════
# § 3  Plotting Components (reusable building blocks)
# ═══════════════════════════════════════════════════════════════

def _draw_background(ax, prob_thr, tu_thr, xlim, ylim):
    """Draw translucent colored quadrants."""
    x0, x1 = xlim
    y0, y1 = ylim
    quads = [
        (prob_thr, x1, y0, tu_thr, "#E53935"),  # HR·C
        (prob_thr, x1, tu_thr, y1, "#FF8F00"),   # HR·U
        (x0, prob_thr, tu_thr, y1, "#7B1FA2"),   # LR·U
        (x0, prob_thr, y0, tu_thr, "#1565C0"),   # LR·C
    ]
    for xlo, xhi, ylo, yhi, color in quads:
        ax.fill([xlo, xhi, xhi, xlo], [ylo, ylo, yhi, yhi],
                color=color, alpha=0.07, zorder=0)


def _draw_zone_labels(ax, prob_thr, tu_thr, xlim, ylim):
    """Corner zone labels."""
    px = (xlim[1] - xlim[0]) * 0.02
    py = (ylim[1] - ylim[0]) * 0.02
    corners = [
        (xlim[1] - px, ylim[0] + py, "right", "bottom", "High Risk\nCertain", "#E53935"),
        (xlim[1] - px, ylim[1] - py, "right", "top", "High Risk\nUncertain", "#E65100"),
        (xlim[0] + px, ylim[1] - py, "left", "top", "Low Risk\nUncertain", "#6A1B9A"),
        (xlim[0] + px, ylim[0] + py, "left", "bottom", "Low Risk\nCertain", "#0D47A1"),
    ]
    for x, y, ha, va, txt, col in corners:
        ax.text(x, y, txt, ha=ha, va=va, fontsize=8.5, color=col,
                alpha=0.55, fontstyle="italic")


def _draw_scatter(ax, prob, tu, y_true, zones, prob_thr):
    """Draw the scatter points colored by zone, edge-colored by correctness."""
    has_y = y_true is not None
    y_arr = y_true if has_y else np.zeros_like(prob, dtype=int)
    for i, (name, cfg) in enumerate(ZONE_CFG.items()):
        mask = zones == i
        if not mask.any():
            continue
        if has_y:
            correct = y_arr[mask] == (prob[mask] >= prob_thr).astype(int)
            ec = np.where(correct, "black", "#FF1744")
            ec_kw = dict(edgecolors=ec, linewidths=0.8)
        else:
            ec_kw = dict(edgecolors="white", linewidths=0.5)
        ax.scatter(prob[mask], tu[mask], c=cfg["color"], marker=cfg["marker"],
                   s=POINT_SIZE, alpha=ALPHA, zorder=cfg["z"] + 1, **ec_kw)


def _draw_calibration_bars(ax, metrics):
    """Mini bar chart: predicted vs observed rate per zone."""
    names = list(ZONE_CFG.keys())
    colors = [ZONE_CFG[z]["color"] for z in names]
    short = [ZONE_SHORT[z] for z in names]
    x = np.arange(len(names))
    w = 0.35
    pv = [metrics[z]["mean_pred"] if not np.isnan(metrics[z]["mean_pred"]) else 0 for z in names]
    ov = [metrics[z]["observed_rate"] if not np.isnan(metrics[z]["observed_rate"]) else 0 for z in names]
    ax.bar(x - w / 2, pv, w, label="Mean Pred.", color=colors, alpha=0.55, edgecolor="white")
    ax.bar(x + w / 2, ov, w, label="Observed Rate", color=colors, alpha=1.0,
           edgecolor="black", linewidth=0.8, hatch="//")
    ax.set_xticks(x)
    ax.set_xticklabels(short, fontsize=8)
    ax.set_ylabel("Rate", fontsize=8)
    ax.set_ylim(0, 1.12)
    ax.set_title("Calibration by Zone", fontsize=8, pad=3)
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(axis="y", alpha=0.3)
    ax.tick_params(axis="y", labelsize=7)
    for i, (p_, o_) in enumerate(zip(pv, ov)):
        gap = p_ - o_
        ax.text(i, max(p_, o_) + 0.04, f"{gap:+.2f}", ha="center", fontsize=7,
                fontweight="bold", color="#C62828" if gap > 0 else "#1B5E20")


def _draw_metrics_table(fig, metrics, title, rect=(0.05, 0.21, 0.90, 0.14)):
    """Per-zone metrics table at the bottom of the figure."""
    names = list(ZONE_CFG.keys())
    colors = [ZONE_CFG[z]["color"] for z in names]
    n_total = sum(metrics[z]["n"] for z in names)
    col_labels = ["Zone", "n (%)", "Observed Rate", "Mean Pred.",
                  "Calibration Gap", "PPV", "NPV"]
    rows = []
    for name in names:
        m = metrics[name]
        n = m["n"]
        pct = f"{n / n_total * 100:.1f}%" if n_total > 0 else "—"
        rows.append([name, f"{n} ({pct})", _fmt(m["observed_rate"]),
                     _fmt(m["mean_pred"]), _fmt(m["calibration_gap"], sign=True),
                     _fmt(m["PPV"]), _fmt(m["NPV"])])

    ax_t = fig.add_axes(rect)
    ax_t.axis("off")
    tbl = ax_t.table(cellText=rows, colLabels=col_labels, cellLoc="center", loc="center")
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.5)
    tbl.scale(1, 1.6)
    for j in range(len(col_labels)):
        tbl[(0, j)].set_facecolor("#37474F")
        tbl[(0, j)].set_text_props(color="white", fontweight="bold")
    for i, color in enumerate(colors):
        tbl[(i + 1, 0)].set_facecolor(color + "28")
        for j in range(1, len(col_labels)):
            tbl[(i + 1, j)].set_facecolor("#F5F5F5" if i % 2 == 0 else "white")
    ax_t.set_title(title, fontsize=9, fontweight="bold", pad=4)


# ═══════════════════════════════════════════════════════════════
# § 4  Youden ROC Plot
# ═══════════════════════════════════════════════════════════════

def plot_youden_roc(youden_result: dict, split_name: str = "Val",
                    output_path: str = "youden_roc.tiff") -> None:
    """Plot main ROC + error-detection ROC curves."""
    n_extra = len(youden_result["error_roc"])
    n_cols = 1 + (1 if n_extra > 0 else 0)
    fig, axes = plt.subplots(1, n_cols, figsize=(6 * n_cols, 5))
    if n_cols == 1:
        axes = [axes]

    # Main ROC
    ax = axes[0]
    ax.plot(youden_result["fpr"], youden_result["tpr"],
            color="#1565C0", lw=2,
            label=f"ROC  AUC = {youden_result['auc']:.3f}")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.5)
    ax.set(title=f"ROC Curve — {split_name}",
           xlabel="False Positive Rate (1 − Specificity)",
           ylabel="True Positive Rate (Sensitivity)",
           xlim=(-0.02, 1.02), ylim=(-0.02, 1.02))
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(alpha=0.25)

    # Error-detection ROC
    if n_extra > 0:
        ax2 = axes[1]
        palette = ["#7B1FA2", "#FF8F00", "#1B5E20", "#37474F"]
        for k, (name, er) in enumerate(youden_result["error_roc"].items()):
            ax2.plot(er["fpr"], er["tpr"],
                     color=palette[k % len(palette)], lw=1.8,
                     label=f"{name}  AUC = {er['auc']:.3f}")
        ax2.plot([0, 1], [0, 1], "k--", lw=0.8, alpha=0.5)
        ax2.set(title=f"Error-Detection ROC — {split_name}",
                xlabel="False Positive Rate", ylabel="TPR",
                xlim=(-0.02, 1.02), ylim=(-0.02, 1.02))
        ax2.legend(fontsize=8, loc="lower right")
        ax2.grid(alpha=0.25)

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")
    print(f"  Saved: {output_path}")
    plt.show()
    plt.close(fig)


# ═══════════════════════════════════════════════════════════════
# § 5  Single-Split Risk Stratification Plot
# ═══════════════════════════════════════════════════════════════

def plot_single_split(prob, tu, y_true, prob_thr, tu_thr,
                      split_name="", output_path="risk_strat.tiff"):
    """Scatter plot + calibration bars + metrics table for one dataset."""
    prob = _1d(prob)
    tu = _1d(tu)
    y_true = _1d(y_true) if y_true is not None else None
    y_arr = y_true if y_true is not None else np.zeros_like(prob, dtype=int)
    zones = assign_zones(prob, tu, prob_thr, tu_thr)
    metrics = compute_zone_metrics(y_arr, prob, zones, prob_thr)

    fig = plt.figure(figsize=FIGSIZE_SINGLE)
    gs = GridSpec(2, 2, figure=fig, height_ratios=[4, 1.5], width_ratios=[3, 1],
                  hspace=0.45, wspace=0.35, top=0.92, bottom=0.18, left=0.07, right=0.97)
    ax = fig.add_subplot(gs[0, 0])
    ax_cal = fig.add_subplot(gs[0, 1])

    xpad = (prob.max() - prob.min()) * 0.05
    ypad = (tu.max() - tu.min()) * 0.05
    xlim = (prob.min() - xpad, prob.max() + xpad)
    ylim = (tu.min() - 0.1, tu.max() + 0.1)

    # Draw components
    _draw_background(ax, prob_thr, tu_thr, xlim, ylim)
    _draw_zone_labels(ax, prob_thr, tu_thr, xlim, ylim)
    ax.axvline(prob_thr, color="gray", lw=1.4, ls="--", alpha=0.7, zorder=1)
    ax.axhline(tu_thr, color="gray", lw=1.4, ls="--", alpha=0.7, zorder=1)
    ax.text(prob_thr + 0.005, ylim[1] * 0.98, f"p={prob_thr:.2f}",
            color="gray", fontsize=8, va="top")
    ax.text(xlim[1] * 0.98, tu_thr + (ylim[1] - ylim[0]) * 0.01,
            f"TU={tu_thr:.3f}", color="gray", fontsize=8, ha="right")

    _draw_scatter(ax, prob, tu, y_true, zones, prob_thr)

    # Legend
    handles = [Line2D([0], [0], marker=cfg["marker"], color="w",
                      markerfacecolor=cfg["color"], markersize=8,
                      alpha=0.9, label=name)
               for name, cfg in ZONE_CFG.items()]
    if y_true is not None:
        handles += [
            Line2D([0], [0], marker="o", color="w", markerfacecolor="gray",
                   markeredgecolor="black", markersize=8, label="Correct prediction"),
            Line2D([0], [0], marker="o", color="w", markerfacecolor="gray",
                   markeredgecolor="#FF1744", markersize=8, label="Wrong prediction"),
        ]
    ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.3),
              ncol=1, fontsize=7.8, framealpha=0.9)

    ax.set(title=f"Risk Stratification{' — ' + split_name if split_name else ''}",
           xlabel="Ensemble Predicted Probability",
           ylabel="Total Uncertainty (TU — Predictive Entropy)",
           xlim=xlim, ylim=ylim)
    ax.grid(alpha=0.2, zorder=0)

    # Calibration
    if y_true is not None:
        _draw_calibration_bars(ax_cal, metrics)
    else:
        ax_cal.axis("off")
        ax_cal.text(0.5, 0.5, "y_true not\nprovided", ha="center", va="center",
                    transform=ax_cal.transAxes, fontsize=9, color="gray")

    # Metrics table
    if y_true is not None:
        _draw_metrics_table(fig, metrics,
                            f"Per-Zone Clinical Metrics — {split_name}")

    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")
    print(f"  Saved: {output_path}")
    plt.show()
    plt.close(fig)
    return metrics


# ═══════════════════════════════════════════════════════════════
# § 6  Multi-Split Comparison Panel
# ═══════════════════════════════════════════════════════════════

def plot_multi_split(splits: dict, prob_thr: float, tu_thr: float,
                     output_path: str = "risk_stratification_multi.tiff") -> dict:
    """
    Side-by-side scatter + mini metrics table for all splits.

    Parameters
    ----------
    splits : dict[name → dict(prob, tu, y_true)]
    prob_thr, tu_thr : shared thresholds
    """
    n = len(splits)
    fig, axes = plt.subplots(2, n, figsize=(FIGSIZE_MULTI[0] * n / 3, FIGSIZE_MULTI[1]),
                             gridspec_kw={"height_ratios": [4, 1.5]})
    if n == 1:
        axes = axes.reshape(2, 1)

    all_metrics = {}

    for col, (name, data) in enumerate(splits.items()):
        ax = axes[0, col]
        ax_tbl = axes[1, col]

        prob = _1d(data["prob"])
        tu = _1d(data["tu"])
        y_true = _1d(data["y_true"]) if "y_true" in data else None
        y_arr = y_true if y_true is not None else np.zeros_like(prob, dtype=int)

        zones = assign_zones(prob, tu, prob_thr, tu_thr)
        metrics = compute_zone_metrics(y_arr, prob, zones, prob_thr)
        all_metrics[name] = metrics

        xpad = (prob.max() - prob.min()) * 0.05
        ypad = (tu.max() - tu.min()) * 0.05
        xlim = (prob.min() - xpad, prob.max() + xpad)
        ylim = (tu.min() - ypad, tu.max() + ypad)

        _draw_background(ax, prob_thr, tu_thr, xlim, ylim)
        _draw_zone_labels(ax, prob_thr, tu_thr, xlim, ylim)
        ax.axvline(prob_thr, color="gray", lw=1.2, ls="--", alpha=0.6)
        ax.axhline(tu_thr, color="gray", lw=1.2, ls="--", alpha=0.6)
        _draw_scatter(ax, prob, tu, y_true, zones, prob_thr)

        # Inset zone summary
        n_total = len(prob)
        lines = []
        for zn, cfg in ZONE_CFG.items():
            m = metrics[zn]
            pct = m["n"] / n_total * 100
            obs = _fmt(m["observed_rate"]) if y_true is not None else "—"
            ppv = _fmt(m["PPV"]) if y_true is not None else "—"
            short = ZONE_SHORT[zn]
            lines.append(f"{short}  n={m['n']}({pct:.0f}%)  Obs={obs}  PPV={ppv}")
        ax.text(0.02, 0.98, "\n".join(lines), transform=ax.transAxes, fontsize=6.5,
                va="top", bbox=dict(boxstyle="round", fc="white", alpha=0.85, ec="gray"))

        ax.set(xlabel="Predicted Probability",
               ylabel="TU (Predictive Entropy)" if col == 0 else "",
               xlim=xlim, ylim=ylim)
        ax.grid(alpha=0.2)

        # Mini metrics table
        ax_tbl.axis("off")
        if y_true is not None:
            col_labels = ["Zone", "n", "Obs. Rate", "PPV", "NPV", "Cal. Gap"]
            rows = [[zn, str(metrics[zn]["n"]),
                     _fmt(metrics[zn]["observed_rate"]),
                     _fmt(metrics[zn]["PPV"]),
                     _fmt(metrics[zn]["NPV"]),
                     _fmt(metrics[zn]["calibration_gap"], sign=True)]
                    for zn in ZONE_CFG]
            tbl = ax_tbl.table(cellText=rows, colLabels=col_labels,
                               cellLoc="center", loc="center")
            tbl.auto_set_font_size(False)
            tbl.set_fontsize(7)
            tbl.scale(1, 1.45)
            for j in range(len(col_labels)):
                tbl[(0, j)].set_facecolor("#37474F")
                tbl[(0, j)].set_text_props(color="white", fontweight="bold")
            for i, (zn, cfg) in enumerate(ZONE_CFG.items()):
                tbl[(i + 1, 0)].set_facecolor(cfg["color"] + "28")
            ax_tbl.set_title(f"Metrics — {name}", fontsize=8, fontweight="bold", pad=3)

    fig.suptitle(f"Risk Stratification Comparison  "
                 f"(prob_thr={prob_thr:.2f}, TU_thr={tu_thr:.3f})",
                 fontsize=13, fontweight="bold", y=1.01)

    # Shared legend
    legend_handles = [Line2D([0], [0], marker=cfg["marker"], color="w",
                             markerfacecolor=cfg["color"], markersize=9,
                             alpha=0.9, label=zn)
                      for zn, cfg in ZONE_CFG.items()]
    fig.legend(handles=legend_handles, loc="lower center",
               bbox_to_anchor=(0.5, -0.03), ncol=4, fontsize=9, framealpha=0.9)

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")
    print(f"  Saved: {output_path}")
    plt.show()
    plt.close(fig)
    return all_metrics


# ═══════════════════════════════════════════════════════════════
# § 7  Console Report
# ═══════════════════════════════════════════════════════════════

def print_metrics_report(all_metrics: dict) -> None:
    """Formatted console output for all splits."""
    sep = "─" * 112
    print(f"\n{'RISK STRATIFICATION — CLINICAL METRICS REPORT':^112}")
    print(sep)
    print(f"{'Zone':<28} {'Split':<8} {'n':>5} "
          f"{'Obs.Rate':>9} {'MeanPred':>9} {'Cal.Gap':>9} "
          f"{'PPV':>7} {'NPV':>7} {'TP':>5} {'FP':>5} {'TN':>5} {'FN':>5}")
    print(sep)
    for zone_name in ZONE_CFG:
        for split_name, metrics in all_metrics.items():
            m = metrics[zone_name]
            print(f"{zone_name:<28} {split_name:<8} {m['n']:>5} "
                  f"{_fmt(m['observed_rate']):>9} {_fmt(m['mean_pred']):>9} "
                  f"{_fmt(m['calibration_gap'], sign=True):>9} "
                  f"{_fmt(m['PPV']):>7} {_fmt(m['NPV']):>7} "
                  f"{m['TP']:>5} {m['FP']:>5} {m['TN']:>5} {m['FN']:>5}")
        print()
    print(sep)


# ═══════════════════════════════════════════════════════════════
# § 8  Main Entry Point
# ═══════════════════════════════════════════════════════════════

def run_risk_stratification(predictions_train, y_train,
                            predictions_val, y_val,
                            predictions_test, y_test,
                            compute_uncertainties_fn=None):
    """
    Complete risk stratification pipeline:
      1. Compute uncertainties for all splits
      2. Derive thresholds from validation set
      3. Plot Youden ROC for validation set
      4. Plot single-split for test set
      5. Plot multi-split comparison (all 3)
      6. Print metrics report

    Parameters
    ----------
    predictions_{split} : ndarray (n_models, n_samples) — bootstrap output
    y_{split}           : ndarray (n_samples,) — true labels
    compute_uncertainties_fn : callable or None
        Function: predictions → dict with keys 'mean_pred', 'TU', etc.
        If None, uses `compute_uncertainties` from the global scope.
    """
    # Resolve uncertainty function
    if compute_uncertainties_fn is None:
        compute_uncertainties_fn = globals().get("compute_uncertainties")
        if compute_uncertainties_fn is None:
            raise RuntimeError(
                "compute_uncertainties not found in global scope. "
                "Either pass compute_uncertainties_fn or run the evaluation cell first.")

    print("=" * 70)
    print("RISK STRATIFICATION ANALYSIS")
    print("=" * 70)

    # Step 1: Compute uncertainties for all splits
    split_data = {}
    for name, preds, y in [("Training", predictions_train, y_train),
                            ("Validation", predictions_val, y_val),
                            ("Test", predictions_test, y_test)]:
        unc = compute_uncertainties_fn(preds)
        split_data[name] = {
            "prob": _1d(unc["mean_pred"]),
            "tu": _1d(unc["TU"]),
            "y_true": _1d(y),
        }
        print(f"  {name:12s}: N={len(y):>7,}  mean_prob={unc['mean_pred'].mean():.4f}  "
              f"mean_TU={unc['TU'].mean():.4f}")

    # Step 2: Derive thresholds from validation set
    val = split_data["Validation"]
    tu_thr = float(np.percentile(val["tu"], TU_QUANTILE * 100))
    prob_thr = PROB_THRESHOLD

    youden = youden_threshold(
        y_true=val["y_true"],
        prob=val["prob"],
        uncertainty_measures={"TU": val["tu"]},
    )
    print(f"\n  Thresholds: prob_thr={prob_thr:.3f} (fixed),  "
          f"TU_thr={tu_thr:.4f} ({TU_QUANTILE * 100:.0f}th pct of Val TU)")
    print(f"  Youden J={youden['youden_j']:.3f}  "
          f"AUC={youden['auc']:.3f}  "
          f"TPR={youden['best_tpr']:.3f}  "
          f"Spec={youden['best_specificity']:.3f}")

    # Step 3: Plot Youden ROC
    print("\nPlotting Youden ROC...")
    plot_youden_roc(youden, split_name="Validation",
                    output_path="youden_roc_val.tiff")

    # Step 4: Single-split for test set
    print("\nPlotting test-set risk stratification...")
    test_metrics = plot_single_split(
        prob=split_data["Test"]["prob"],
        tu=split_data["Test"]["tu"],
        y_true=split_data["Test"]["y_true"],
        prob_thr=prob_thr, tu_thr=tu_thr,
        split_name="Temporal Validation",
        output_path="risk_stratification_test.tiff",
    )

    # Step 5: Multi-split comparison
    print("\nPlotting tri-split comparison...")
    all_metrics = plot_multi_split(
        split_data, prob_thr=prob_thr, tu_thr=tu_thr,
        output_path="risk_stratification_multi.tiff",
    )

    # Step 6: Print report
    print_metrics_report(all_metrics)

    print("\n" + "=" * 70)
    print("Risk stratification complete.")
    print("=" * 70)

    return all_metrics


print("Risk Stratification module loaded.")
print("  Usage: run_risk_stratification(predictions_train, y_train,")
print("           predictions_val, y_val, predictions_test, y_test)")


In [ ]:
# ── Run the complete risk stratification pipeline ──
run_risk_stratification(
    predictions_train, y_train,
    predictions_val,   y_val,
    predictions_test,  y_test,
    compute_uncertainties_fn=compute_uncertainties,
)

## 10. High-Confidence Evaluation

Progressive exclusion of high-uncertainty samples; TU-based confidence filtering.

In [ ]:
"""
High-Confidence Sample Discriminative Performance Evaluation
=============================================================
Evaluate how the model performs when restricted to "certain" predictions
(low TU), across multiple uncertainty thresholds.

Methods alignment (§2.5):
  Progressive exclusion of high-uncertainty samples → recalculate
  AUROC, AUPR, Brier, F1, Accuracy, Sensitivity, Specificity on the
  retained (high-confidence) subset.

Analyses:
  1. Confidence-Stratified Metrics Table
     — Divide into TU quartiles; compute full metric set per quartile
  2. Progressive Confidence Filtering Curves
     — Sweep TU percentile threshold from 100% → 10%;
       plot AUROC, AUPR, Brier, F1, Accuracy as a function of retention
  3. Certain vs Uncertain Head-to-Head
     — Split at validation-set median TU; compare all metrics with 95% CI
  4. High-Confidence Calibration
     — Calibration curve for certain subset only vs full sample
  5. Clinical Summary Table
     — Formatted for manuscript Table / supplementary material

Output: high_confidence_evaluation.tiff  (composite figure, 300 dpi)
        high_confidence_metrics.csv       (machine-readable)
"""

from __future__ import annotations
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    confusion_matrix, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, precision_recall_curve,
)
from sklearn.calibration import calibration_curve


# ═══════════════════════════════════════════════════════════════════════════
# § 0  Configuration
# ═══════════════════════════════════════════════════════════════════════════

DPI            = 300
N_BOOT_CI      = 200
RANDOM_SEED    = 42
PROB_THRESHOLD = 0.5
N_SWEEP_POINTS = 50          # points for progressive filtering curve
QUARTILE_LABELS = ["Q1 (Most certain)", "Q2", "Q3", "Q4 (Most uncertain)"]

SPLIT_CFG = {
    "Training":            dict(color="#2196F3", ls="-",  lw=2.2),
    "Internal Validation": dict(color="#FF9800", ls="--", lw=2.2),
    "Temporal Validation": dict(color="#E91E63", ls="-.", lw=2.2),
}


# ═══════════════════════════════════════════════════════════════════════════
# § 1  Core metric computation (with bootstrap 95% CI)
# ═══════════════════════════════════════════════════════════════════════════

def _compute_full_metrics(y_true, y_prob, threshold=PROB_THRESHOLD):
    """Compute the complete metric set for a single sample."""
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp = cm[0, 0], cm[0, 1]
    return {
        "n":            len(y_true),
        "Events":       int(y_true.sum()),
        "Prevalence":   y_true.mean(),
        "AUROC":        roc_auc_score(y_true, y_prob)   if len(np.unique(y_true)) == 2 else np.nan,
        "AUPR":         average_precision_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "Brier":        brier_score_loss(y_true, y_prob),
        "Accuracy":     accuracy_score(y_true, y_pred),
        "Sensitivity":  recall_score(y_true, y_pred, zero_division=0),
        "Specificity":  tn / max(1, tn + fp),
        "PPV":          precision_score(y_true, y_pred, zero_division=0),
        "NPV":          tn / max(1, tn + cm[1, 0]),
        "F1":           f1_score(y_true, y_pred, zero_division=0),
    }


def _bootstrap_ci(y_true, y_prob, metric_fn, n_boot=N_BOOT_CI, seed=RANDOM_SEED):
    """Bootstrap percentile 95% CI for a scalar metric."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt, yp = y_true[idx], y_prob[idx]
        if len(np.unique(yt)) < 2:
            continue
        try:
            vals.append(metric_fn(yt, yp))
        except Exception:
            continue
    if len(vals) < 10:
        return (np.nan, np.nan)
    return (float(np.percentile(vals, 2.5)),
            float(np.percentile(vals, 97.5)))


def compute_metrics_with_ci(y_true, y_prob, threshold=PROB_THRESHOLD):
    """Full metrics + 95% CI for AUROC, AUPR, Brier."""
    m = _compute_full_metrics(y_true, y_prob, threshold)

    for name, fn in [
        ("AUROC", lambda yt, yp: roc_auc_score(yt, yp)),
        ("AUPR",  lambda yt, yp: average_precision_score(yt, yp)),
        ("Brier", lambda yt, yp: brier_score_loss(yt, yp)),
    ]:
        lo, hi = _bootstrap_ci(y_true, y_prob, fn)
        m[f"{name}_CI_lo"] = lo
        m[f"{name}_CI_hi"] = hi

    return m


# ═══════════════════════════════════════════════════════════════════════════
# § 2  Analysis 1 — TU-Quartile Stratified Metrics
# ═══════════════════════════════════════════════════════════════════════════

def quartile_stratified_metrics(y_true, y_prob, TU, split_name=""):
    """Divide by TU quartiles; compute full metrics per quartile."""
    quartile_bounds = np.percentile(TU, [25, 50, 75])
    q_masks = [
        TU <= quartile_bounds[0],
        (TU > quartile_bounds[0]) & (TU <= quartile_bounds[1]),
        (TU > quartile_bounds[1]) & (TU <= quartile_bounds[2]),
        TU > quartile_bounds[2],
    ]
    rows = []
    for i, (mask, label) in enumerate(zip(q_masks, QUARTILE_LABELS)):
        if mask.sum() < 5 or len(np.unique(y_true[mask])) < 2:
            continue
        m = compute_metrics_with_ci(y_true[mask], y_prob[mask])
        m["Quartile"] = label
        m["TU_range"] = f"[{TU[mask].min():.4f}, {TU[mask].max():.4f}]"
        m["TU_mean"]  = TU[mask].mean()
        m["Split"]    = split_name
        rows.append(m)
    return pd.DataFrame(rows)


# ═══════════════════════════════════════════════════════════════════════════
# § 3  Analysis 2 — Progressive Confidence Filtering Curves
# ═══════════════════════════════════════════════════════════════════════════

def progressive_confidence_curves(y_true, y_prob, TU, n_points=N_SWEEP_POINTS):
    """
    Progressively retain only the most confident samples (lowest TU).
    At each retention level, compute AUROC, AUPR, Brier, F1, Accuracy.
    """
    retention_levels = np.linspace(1.0, 0.1, n_points)
    records = []
    for ret in retention_levels:
        thr = np.percentile(TU, ret * 100)
        mask = TU <= thr
        n = mask.sum()
        if n < 20 or len(np.unique(y_true[mask])) < 2:
            continue
        yt, yp = y_true[mask], y_prob[mask]
        y_pred = (yp >= PROB_THRESHOLD).astype(int)
        records.append({
            "Retention":   mask.mean(),
            "n":           n,
            "TU_threshold": thr,
            "AUROC":       roc_auc_score(yt, yp),
            "AUPR":        average_precision_score(yt, yp),
            "Brier":       brier_score_loss(yt, yp),
            "F1":          f1_score(yt, y_pred, zero_division=0),
            "Accuracy":    accuracy_score(yt, y_pred),
            "Sensitivity": recall_score(yt, y_pred, zero_division=0),
            "Specificity": 1 - (confusion_matrix(yt, y_pred, labels=[0,1])[0,1] /
                                max(1, confusion_matrix(yt, y_pred, labels=[0,1])[0].sum())),
            "Prevalence":  yt.mean(),
        })
    return pd.DataFrame(records)


# ═══════════════════════════════════════════════════════════════════════════
# § 4  Analysis 3 — Certain vs Uncertain Head-to-Head
# ═══════════════════════════════════════════════════════════════════════════

def certain_vs_uncertain(y_true, y_prob, TU, tu_thr, split_name=""):
    """Compare metrics: certain (TU < tu_thr) vs uncertain (TU >= tu_thr)."""
    masks = {
        "Certain (TU < median)":   TU < tu_thr,
        "Uncertain (TU ≥ median)": TU >= tu_thr,
        "Full sample":             np.ones(len(TU), dtype=bool),
    }
    rows = []
    for label, mask in masks.items():
        if mask.sum() < 10 or len(np.unique(y_true[mask])) < 2:
            continue
        m = compute_metrics_with_ci(y_true[mask], y_prob[mask])
        m["Subset"] = label
        m["Split"]  = split_name
        rows.append(m)
    return pd.DataFrame(rows)


# ═══════════════════════════════════════════════════════════════════════════
# § 5  Plotting — Composite Figure
# ═══════════════════════════════════════════════════════════════════════════

def _fmt_ci(val, lo, hi):
    if np.isnan(val):
        return "—"
    return f"{val:.4f} [{lo:.4f}–{hi:.4f}]"


def plot_high_confidence_evaluation(
    splits_data: dict,
    tu_thr: float,
    output_path: str = "high_confidence_evaluation.tiff",
):
    """
    Composite 2×3 figure:
      Row 1: [Progressive AUROC] [Progressive AUPR] [Progressive Brier]
      Row 2: [Progressive F1/Acc] [Calibration: certain vs all] [Quartile heatmap]

    Parameters
    ----------
    splits_data : dict[split_name → dict(y_true, y_prob, TU, predictions)]
    tu_thr      : uncertainty threshold (from validation set median)
    """
    fig = plt.figure(figsize=(20, 13), facecolor="white")
    gs = gridspec.GridSpec(2, 3, hspace=0.35, wspace=0.32,
                           top=0.93, bottom=0.06, left=0.06, right=0.97)

    # ── Row 1: Progressive curves (AUROC / AUPR / Brier) ─────────────
    metric_panels = [
        ("AUROC", "higher → better", True),
        ("AUPR",  "higher → better", True),
        ("Brier", "lower → better",  False),
    ]
    for col, (metric, direction, higher_better) in enumerate(metric_panels):
        ax = fig.add_subplot(gs[0, col])
        for split_name, sdata in splits_data.items():
            df = progressive_confidence_curves(
                sdata["y_true"], sdata["y_prob"], sdata["TU"]
            )
            sty = SPLIT_CFG[split_name]
            ax.plot(df["Retention"] * 100, df[metric],
                    color=sty["color"], ls=sty["ls"], lw=sty["lw"],
                    label=split_name)
            # Mark the full-sample baseline
            baseline = df[metric].iloc[0]
            ax.axhline(baseline, color=sty["color"], ls=":", lw=0.8, alpha=0.4)

        ax.set_xlabel("Retention Rate (%)", fontsize=10)
        ax.set_ylabel(metric, fontsize=10)
        ax.set_title(f"{metric} vs. Confidence Retention\n({direction})",
                     fontsize=11, fontweight="bold")
        ax.legend(fontsize=8, loc="best")
        ax.grid(alpha=0.25)
        ax.set_xlim([10, 100])
        ax.invert_xaxis()
        ax.spines[["top", "right"]].set_visible(False)

    # ── Row 2, Col 0: F1 + Accuracy curves ───────────────────────────
    ax_f1 = fig.add_subplot(gs[1, 0])
    for split_name, sdata in splits_data.items():
        df = progressive_confidence_curves(
            sdata["y_true"], sdata["y_prob"], sdata["TU"]
        )
        sty = SPLIT_CFG[split_name]
        ax_f1.plot(df["Retention"] * 100, df["F1"],
                   color=sty["color"], ls=sty["ls"], lw=sty["lw"],
                   label=f"{split_name} F1")
        ax_f1.plot(df["Retention"] * 100, df["Accuracy"],
                   color=sty["color"], ls=sty["ls"], lw=1.0, alpha=0.5,
                   label=f"{split_name} Acc")
    ax_f1.set_xlabel("Retention Rate (%)", fontsize=10)
    ax_f1.set_ylabel("Score", fontsize=10)
    ax_f1.set_title("F1 & Accuracy vs. Confidence Retention",
                    fontsize=11, fontweight="bold")
    ax_f1.legend(fontsize=7, ncol=2, loc="best")
    ax_f1.grid(alpha=0.25)
    ax_f1.set_xlim([10, 100])
    ax_f1.invert_xaxis()
    ax_f1.spines[["top", "right"]].set_visible(False)

    # ── Row 2, Col 1: Calibration — certain subset vs full sample ────
    ax_cal = fig.add_subplot(gs[1, 1])
    ax_cal.plot([0, 1], [0, 1], "k--", lw=1.2, alpha=0.4,
                label="Perfect calibration")
    cal_colors = {"Certain": "#2E7D32", "Full": "#78909C"}
    for split_name, sdata in splits_data.items():
        if split_name != "Temporal Validation":
            continue
        yt, yp, tu = sdata["y_true"], sdata["y_prob"], sdata["TU"]
        certain_mask = tu < tu_thr

        for subset, mask, marker, alpha in [
            ("Certain", certain_mask, "o", 1.0),
            ("Full",    np.ones(len(tu), dtype=bool), "s", 0.5),
        ]:
            y_s, p_s = yt[mask], yp[mask]
            if len(np.unique(y_s)) < 2:
                continue
            frac_pos, mean_pred = calibration_curve(y_s, p_s, n_bins=8,
                                                     strategy="uniform")
            brier = brier_score_loss(y_s, p_s)
            ax_cal.plot(mean_pred, frac_pos, marker=marker, ms=7,
                        lw=2, color=cal_colors[subset], alpha=alpha,
                        label=f"{subset} (n={mask.sum()}, Brier={brier:.4f})")

    ax_cal.set(xlabel="Mean Predicted Probability", ylabel="Observed Frequency",
               xlim=[-0.02, 1.02], ylim=[-0.02, 1.02])
    ax_cal.set_title("Calibration: Certain vs. Full\n(Temporal Validation)",
                     fontsize=11, fontweight="bold")
    ax_cal.legend(fontsize=8)
    ax_cal.grid(alpha=0.25)
    ax_cal.spines[["top", "right"]].set_visible(False)

    # ── Row 2, Col 2: Quartile heatmap (Temporal Validation) ─────────
    ax_hm = fig.add_subplot(gs[1, 2])
    test_data = splits_data.get("Temporal Validation", list(splits_data.values())[-1])
    df_q = quartile_stratified_metrics(
        test_data["y_true"], test_data["y_prob"], test_data["TU"],
        split_name="Temporal Validation"
    )
    if not df_q.empty:
        hm_metrics = ["AUROC", "AUPR", "Brier", "F1", "Accuracy", "Sensitivity", "Specificity"]
        hm_data = df_q.set_index("Quartile")[hm_metrics].astype(float).values
        hm_labels = df_q["Quartile"].values

        im = ax_hm.imshow(hm_data, aspect="auto", cmap="RdYlGn",
                          vmin=0, vmax=1, interpolation="nearest")
        ax_hm.set_xticks(range(len(hm_metrics)))
        ax_hm.set_xticklabels(hm_metrics, rotation=35, ha="right", fontsize=9)
        ax_hm.set_yticks(range(len(hm_labels)))
        ax_hm.set_yticklabels(hm_labels, fontsize=9)
        for r in range(hm_data.shape[0]):
            for c in range(hm_data.shape[1]):
                v = hm_data[r, c]
                tc = "white" if (v < 0.3 or v > 0.85) else "black"
                ax_hm.text(c, r, f"{v:.3f}", ha="center", va="center",
                           fontsize=9, color=tc, fontweight="bold")
        plt.colorbar(im, ax=ax_hm, fraction=0.04, pad=0.03)
    ax_hm.set_title("Metrics by TU Quartile\n(Temporal Validation)",
                     fontsize=11, fontweight="bold")

    fig.suptitle("High-Confidence Sample Discriminative Performance Evaluation",
                 fontsize=14, fontweight="bold")

    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")
    print(f"\n✓ Figure saved → {output_path}")
    plt.show()
    plt.close(fig)


# ═══════════════════════════════════════════════════════════════════════════
# § 6  Console report
# ═══════════════════════════════════════════════════════════════════════════

def print_confidence_report(splits_data, tu_thr):
    """Print formatted head-to-head and quartile tables."""
    sep = "═" * 100
    print(f"\n{sep}")
    print("  HIGH-CONFIDENCE SAMPLE DISCRIMINATIVE PERFORMANCE EVALUATION")
    print(f"  TU threshold (val median): {tu_thr:.4f}")
    print(sep)

    # ── Head-to-head: Certain vs Uncertain ──
    all_h2h = []
    for split_name, sdata in splits_data.items():
        df = certain_vs_uncertain(
            sdata["y_true"], sdata["y_prob"], sdata["TU"],
            tu_thr, split_name
        )
        all_h2h.append(df)
    df_h2h = pd.concat(all_h2h, ignore_index=True)

    print("\n── Certain vs. Uncertain Head-to-Head ──")
    cols = ["Split", "Subset", "n", "Events", "Prevalence",
            "AUROC", "AUROC_CI_lo", "AUROC_CI_hi",
            "AUPR", "Brier", "F1", "Accuracy", "Sensitivity", "Specificity", "PPV", "NPV"]
    available = [c for c in cols if c in df_h2h.columns]
    print(df_h2h[available].to_string(index=False, float_format="%.4f"))

    # ── Quartile analysis ──
    all_q = []
    for split_name, sdata in splits_data.items():
        df = quartile_stratified_metrics(
            sdata["y_true"], sdata["y_prob"], sdata["TU"], split_name
        )
        all_q.append(df)
    df_q = pd.concat(all_q, ignore_index=True)

    print(f"\n── TU-Quartile Stratified Metrics ──")
    q_cols = ["Split", "Quartile", "n", "Events", "TU_mean",
              "AUROC", "AUPR", "Brier", "F1", "Accuracy", "Sensitivity", "Specificity"]
    available_q = [c for c in q_cols if c in df_q.columns]
    print(df_q[available_q].to_string(index=False, float_format="%.4f"))

    # ── Key finding highlight ──
    print(f"\n── Key Findings ──")
    for split_name, sdata in splits_data.items():
        mask_c = sdata["TU"] < tu_thr
        mask_u = sdata["TU"] >= tu_thr
        if mask_c.sum() < 10 or mask_u.sum() < 10:
            continue
        yt_c, yp_c = sdata["y_true"][mask_c], sdata["y_prob"][mask_c]
        yt_u, yp_u = sdata["y_true"][mask_u], sdata["y_prob"][mask_u]
        if len(np.unique(yt_c)) < 2 or len(np.unique(yt_u)) < 2:
            continue
        auc_c = roc_auc_score(yt_c, yp_c)
        auc_u = roc_auc_score(yt_u, yp_u)
        brier_c = brier_score_loss(yt_c, yp_c)
        brier_u = brier_score_loss(yt_u, yp_u)
        acc_c = accuracy_score(yt_c, (yp_c >= 0.5).astype(int))
        acc_u = accuracy_score(yt_u, (yp_u >= 0.5).astype(int))
        print(f"\n  {split_name}:")
        print(f"    Certain  (n={mask_c.sum():>5})  AUROC={auc_c:.4f}  Brier={brier_c:.4f}  Acc={acc_c*100:.2f}%")
        print(f"    Uncertain(n={mask_u.sum():>5})  AUROC={auc_u:.4f}  Brier={brier_u:.4f}  Acc={acc_u*100:.2f}%")
        print(f"    Δ AUROC={auc_c-auc_u:+.4f}   Δ Brier={brier_c-brier_u:+.4f}   Δ Acc={((acc_c-acc_u)*100):+.2f}%")

    print(sep)
    return df_h2h, df_q


# ═══════════════════════════════════════════════════════════════════════════
# § 7  CSV export
# ═══════════════════════════════════════════════════════════════════════════

def export_confidence_metrics(splits_data, tu_thr, output_path="high_confidence_metrics.csv"):
    """Export all confidence-stratified metrics to CSV."""
    all_rows = []
    for split_name, sdata in splits_data.items():
        # Head-to-head
        df_h = certain_vs_uncertain(
            sdata["y_true"], sdata["y_prob"], sdata["TU"], tu_thr, split_name
        )
        all_rows.append(df_h)
        # Quartile
        df_q = quartile_stratified_metrics(
            sdata["y_true"], sdata["y_prob"], sdata["TU"], split_name
        )
        if not df_q.empty:
            df_q["Subset"] = df_q["Quartile"]
        all_rows.append(df_q)

    df_all = pd.concat(all_rows, ignore_index=True)
    df_all.to_csv(output_path, index=False)
    print(f"✓ CSV saved → {output_path}")
    return df_all


# ═══════════════════════════════════════════════════════════════════════════
# § 8  Entry point
# ═══════════════════════════════════════════════════════════════════════════

# ── Build splits_data from existing notebook variables ──
# tu_thr was computed from validation set in Cell 18: tu_thr = median(TU_val)
# predictions_train/val/test and y_train/val/test are defined in Cells 5/8

splits_data = {}
for split_name, preds, y in [
    ("Training",            predictions_train, y_train),
    ("Internal Validation", predictions_val,   y_val),
    ("Temporal Validation", predictions_test,  y_test),
]:
    y_arr = np.asarray(y).reshape(-1)
    preds_arr = np.asarray(preds)
    mean_p = preds_arr.mean(axis=0)
    p1 = np.clip(mean_p, 1e-10, 1 - 1e-10)
    p0 = np.clip(1 - mean_p, 1e-10, 1 - 1e-10)
    TU_arr = -(p1 * np.log(p1) + p0 * np.log(p0))
    splits_data[split_name] = {
        "y_true": y_arr,
        "y_prob": mean_p,
        "TU":     TU_arr,
        "predictions": preds_arr,
    }

# ── tu_thr from validation set (consistent with §2.6) ──
# (tu_thr should already be defined from Cell 18; recompute for safety)
val_TU = splits_data["Internal Validation"]["TU"]
tu_thr_val = float(np.percentile(val_TU, 50))
print(f"TU threshold (validation median): {tu_thr_val:.4f}")

# ── Run evaluation ──
df_h2h, df_q = print_confidence_report(splits_data, tu_thr_val)

plot_high_confidence_evaluation(
    splits_data,
    tu_thr=tu_thr_val,
    output_path="high_confidence_evaluation.tiff"
)

df_export = export_confidence_metrics(
    splits_data, tu_thr_val,
    output_path="high_confidence_metrics.csv"
)


### 10b. High-Confidence Evaluation — Tri-Split Extended

In [ ]:
"""
High-Confidence Sample Discriminative Performance Evaluation
=============================================================
Evaluate how the model performs when restricted to "certain" predictions
(low TU), across multiple uncertainty thresholds.

Methods alignment (§2.5):
  Progressive exclusion of high-uncertainty samples → recalculate
  AUROC, AUPR, Brier, F1, Accuracy, Sensitivity, Specificity on the
  retained (high-confidence) subset.

Analyses:
  1. Confidence-Stratified Metrics Table
     — Divide into TU quartiles; compute full metric set per quartile
  2. Progressive Confidence Filtering Curves
     — Sweep TU percentile threshold from 100% → 10%;
       plot AUROC, AUPR, Brier, F1, Accuracy as a function of retention
  3. Certain vs Uncertain Head-to-Head
     — Split at validation-set median TU; compare all metrics with 95% CI
  4. High-Confidence Calibration
     — Calibration curve for certain subset only vs full sample
  5. Clinical Summary Table
     — Formatted for manuscript Table / supplementary material

Output: high_confidence_evaluation.tiff  (composite figure, 300 dpi)
        high_confidence_metrics.csv       (machine-readable)
"""

from __future__ import annotations
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    confusion_matrix, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, precision_recall_curve,
)
from sklearn.calibration import calibration_curve


# ═══════════════════════════════════════════════════════════════════════════
# § 0  Configuration
# ═══════════════════════════════════════════════════════════════════════════

DPI            = 300
N_BOOT_CI      = 200
RANDOM_SEED    = 42
PROB_THRESHOLD = 0.5
N_SWEEP_POINTS = 50          # points for progressive filtering curve
QUARTILE_LABELS = ["Q1 (Most certain)", "Q2", "Q3", "Q4 (Most uncertain)"]

SPLIT_CFG = {
    "Training":            dict(color="#2196F3", ls="-",  lw=2.2),
    "Internal Validation": dict(color="#FF9800", ls="--", lw=2.2),
    "Temporal Validation": dict(color="#E91E63", ls="-.", lw=2.2),
}


# ═══════════════════════════════════════════════════════════════════════════
# § 1  Core metric computation (with bootstrap 95% CI)
# ═══════════════════════════════════════════════════════════════════════════

def _compute_full_metrics(y_true, y_prob, threshold=PROB_THRESHOLD):
    """Compute the complete metric set for a single sample."""
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp = cm[0, 0], cm[0, 1]
    return {
        "n":            len(y_true),
        "Events":       int(y_true.sum()),
        "Prevalence":   y_true.mean(),
        "AUROC":        roc_auc_score(y_true, y_prob)   if len(np.unique(y_true)) == 2 else np.nan,
        "AUPR":         average_precision_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "Brier":        brier_score_loss(y_true, y_prob),
        "Accuracy":     accuracy_score(y_true, y_pred),
        "Sensitivity":  recall_score(y_true, y_pred, zero_division=0),
        "Specificity":  tn / max(1, tn + fp),
        "PPV":          precision_score(y_true, y_pred, zero_division=0),
        "NPV":          tn / max(1, tn + cm[1, 0]),
        "F1":           f1_score(y_true, y_pred, zero_division=0),
    }


def _bootstrap_ci(y_true, y_prob, metric_fn, n_boot=N_BOOT_CI, seed=RANDOM_SEED):
    """Bootstrap percentile 95% CI for a scalar metric."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        yt, yp = y_true[idx], y_prob[idx]
        if len(np.unique(yt)) < 2:
            continue
        try:
            vals.append(metric_fn(yt, yp))
        except Exception:
            continue
    if len(vals) < 10:
        return (np.nan, np.nan)
    return (float(np.percentile(vals, 2.5)),
            float(np.percentile(vals, 97.5)))


def compute_metrics_with_ci(y_true, y_prob, threshold=PROB_THRESHOLD):
    """Full metrics + 95% CI for AUROC, AUPR, Brier."""
    m = _compute_full_metrics(y_true, y_prob, threshold)

    for name, fn in [
        ("AUROC", lambda yt, yp: roc_auc_score(yt, yp)),
        ("AUPR",  lambda yt, yp: average_precision_score(yt, yp)),
        ("Brier", lambda yt, yp: brier_score_loss(yt, yp)),
    ]:
        lo, hi = _bootstrap_ci(y_true, y_prob, fn)
        m[f"{name}_CI_lo"] = lo
        m[f"{name}_CI_hi"] = hi

    return m


# ═══════════════════════════════════════════════════════════════════════════
# § 2  Analysis 1 — TU-Quartile Stratified Metrics
# ═══════════════════════════════════════════════════════════════════════════

def quartile_stratified_metrics(y_true, y_prob, TU, split_name=""):
    """Divide by TU quartiles; compute full metrics per quartile."""
    quartile_bounds = np.percentile(TU, [25, 50, 75])
    q_masks = [
        TU <= quartile_bounds[0],
        (TU > quartile_bounds[0]) & (TU <= quartile_bounds[1]),
        (TU > quartile_bounds[1]) & (TU <= quartile_bounds[2]),
        TU > quartile_bounds[2],
    ]
    rows = []
    for i, (mask, label) in enumerate(zip(q_masks, QUARTILE_LABELS)):
        if mask.sum() < 5 or len(np.unique(y_true[mask])) < 2:
            continue
        m = compute_metrics_with_ci(y_true[mask], y_prob[mask])
        m["Quartile"] = label
        m["TU_range"] = f"[{TU[mask].min():.4f}, {TU[mask].max():.4f}]"
        m["TU_mean"]  = TU[mask].mean()
        m["Split"]    = split_name
        rows.append(m)
    return pd.DataFrame(rows)


# ═══════════════════════════════════════════════════════════════════════════
# § 3  Analysis 2 — Progressive Confidence Filtering Curves
# ═══════════════════════════════════════════════════════════════════════════

def progressive_confidence_curves(y_true, y_prob, TU, n_points=N_SWEEP_POINTS):
    """
    Progressively retain only the most confident samples (lowest TU).
    At each retention level, compute AUROC, AUPR, Brier, F1, Accuracy.
    """
    retention_levels = np.linspace(1.0, 0.1, n_points)
    records = []
    for ret in retention_levels:
        thr = np.percentile(TU, ret * 100)
        mask = TU <= thr
        n = mask.sum()
        if n < 20 or len(np.unique(y_true[mask])) < 2:
            continue
        yt, yp = y_true[mask], y_prob[mask]
        y_pred = (yp >= PROB_THRESHOLD).astype(int)
        records.append({
            "Retention":   mask.mean(),
            "n":           n,
            "TU_threshold": thr,
            "AUROC":       roc_auc_score(yt, yp),
            "AUPR":        average_precision_score(yt, yp),
            "Brier":       brier_score_loss(yt, yp),
            "F1":          f1_score(yt, y_pred, zero_division=0),
            "Accuracy":    accuracy_score(yt, y_pred),
            "Sensitivity": recall_score(yt, y_pred, zero_division=0),
            "Specificity": 1 - (confusion_matrix(yt, y_pred, labels=[0,1])[0,1] /
                                max(1, confusion_matrix(yt, y_pred, labels=[0,1])[0].sum())),
            "Prevalence":  yt.mean(),
        })
    return pd.DataFrame(records)


# ═══════════════════════════════════════════════════════════════════════════
# § 4  Analysis 3 — Certain vs Uncertain Head-to-Head
# ═══════════════════════════════════════════════════════════════════════════

def certain_vs_uncertain(y_true, y_prob, TU, tu_thr, split_name=""):
    """Compare metrics: certain (TU < tu_thr) vs uncertain (TU >= tu_thr)."""
    masks = {
        "Certain (TU < median)":   TU < tu_thr,
        "Uncertain (TU ≥ median)": TU >= tu_thr,
        "Full sample":             np.ones(len(TU), dtype=bool),
    }
    rows = []
    for label, mask in masks.items():
        if mask.sum() < 10 or len(np.unique(y_true[mask])) < 2:
            continue
        m = compute_metrics_with_ci(y_true[mask], y_prob[mask])
        m["Subset"] = label
        m["Split"]  = split_name
        rows.append(m)
    return pd.DataFrame(rows)


# ═══════════════════════════════════════════════════════════════════════════
# § 5  Plotting — Composite Figure
# ═══════════════════════════════════════════════════════════════════════════

def _fmt_ci(val, lo, hi):
    if np.isnan(val):
        return "—"
    return f"{val:.4f} [{lo:.4f}–{hi:.4f}]"


def plot_high_confidence_evaluation(
    splits_data: dict,
    tu_thr: float,
    output_path: str = "high_confidence_evaluation.tiff",
):
    """
    Composite 2×3 figure:
      Row 1: [Progressive AUROC] [Progressive AUPR] [Progressive Brier]
      Row 2: [Progressive F1/Acc] [Calibration: certain vs all] [Quartile heatmap]

    Parameters
    ----------
    splits_data : dict[split_name → dict(y_true, y_prob, TU, predictions)]
    tu_thr      : uncertainty threshold (from validation set median)
    """
    fig = plt.figure(figsize=(20, 13), facecolor="white")
    gs = gridspec.GridSpec(2, 3, hspace=0.35, wspace=0.32,
                           top=0.93, bottom=0.06, left=0.06, right=0.97)

    # ── Row 1: Progressive curves (AUROC / AUPR / Brier) ─────────────
    metric_panels = [
        ("AUROC", "higher → better", True),
        ("AUPR",  "higher → better", True),
        ("Brier", "lower → better",  False),
    ]
    for col, (metric, direction, higher_better) in enumerate(metric_panels):
        ax = fig.add_subplot(gs[0, col])
        for split_name, sdata in splits_data.items():
            df = progressive_confidence_curves(
                sdata["y_true"], sdata["y_prob"], sdata["TU"]
            )
            sty = SPLIT_CFG[split_name]
            ax.plot(df["Retention"] * 100, df[metric],
                    color=sty["color"], ls=sty["ls"], lw=sty["lw"],
                    label=split_name)
            # Mark the full-sample baseline
            baseline = df[metric].iloc[0]
            ax.axhline(baseline, color=sty["color"], ls=":", lw=0.8, alpha=0.4)

        ax.set_xlabel("Retention Rate (%)", fontsize=10)
        ax.set_ylabel(metric, fontsize=10)
        ax.set_title(f"{metric} vs. Confidence Retention\n({direction})",
                     fontsize=11, fontweight="bold")
        ax.legend(fontsize=8, loc="best")
        ax.grid(alpha=0.25)
        ax.set_xlim([10, 100])
        ax.invert_xaxis()
        ax.spines[["top", "right"]].set_visible(False)

    # ── Row 2, Col 0: F1 + Accuracy curves ───────────────────────────
    ax_f1 = fig.add_subplot(gs[1, 0])
    for split_name, sdata in splits_data.items():
        df = progressive_confidence_curves(
            sdata["y_true"], sdata["y_prob"], sdata["TU"]
        )
        sty = SPLIT_CFG[split_name]
        ax_f1.plot(df["Retention"] * 100, df["F1"],
                   color=sty["color"], ls=sty["ls"], lw=sty["lw"],
                   label=f"{split_name} F1")
        ax_f1.plot(df["Retention"] * 100, df["Accuracy"],
                   color=sty["color"], ls=sty["ls"], lw=1.0, alpha=0.5,
                   label=f"{split_name} Acc")
    ax_f1.set_xlabel("Retention Rate (%)", fontsize=10)
    ax_f1.set_ylabel("Score", fontsize=10)
    ax_f1.set_title("F1 & Accuracy vs. Confidence Retention",
                    fontsize=11, fontweight="bold")
    ax_f1.legend(fontsize=7, ncol=2, loc="best")
    ax_f1.grid(alpha=0.25)
    ax_f1.set_xlim([10, 100])
    ax_f1.invert_xaxis()
    ax_f1.spines[["top", "right"]].set_visible(False)

    # ── Row 2, Col 1: Calibration — certain subset vs full sample ────
    ax_cal = fig.add_subplot(gs[1, 1])
    ax_cal.plot([0, 1], [0, 1], "k--", lw=1.2, alpha=0.4,
                label="Perfect calibration")
    cal_colors = {"Certain": "#2E7D32", "Full": "#78909C"}
    for split_name, sdata in splits_data.items():
        if split_name != "Temporal Validation":
            continue
        yt, yp, tu = sdata["y_true"], sdata["y_prob"], sdata["TU"]
        certain_mask = tu < tu_thr

        for subset, mask, marker, alpha in [
            ("Certain", certain_mask, "o", 1.0),
            ("Full",    np.ones(len(tu), dtype=bool), "s", 0.5),
        ]:
            y_s, p_s = yt[mask], yp[mask]
            if len(np.unique(y_s)) < 2:
                continue
            frac_pos, mean_pred = calibration_curve(y_s, p_s, n_bins=8,
                                                     strategy="uniform")
            brier = brier_score_loss(y_s, p_s)
            ax_cal.plot(mean_pred, frac_pos, marker=marker, ms=7,
                        lw=2, color=cal_colors[subset], alpha=alpha,
                        label=f"{subset} (n={mask.sum()}, Brier={brier:.4f})")

    ax_cal.set(xlabel="Mean Predicted Probability", ylabel="Observed Frequency",
               xlim=[-0.02, 1.02], ylim=[-0.02, 1.02])
    ax_cal.set_title("Calibration: Certain vs. Full\n(Temporal Validation)",
                     fontsize=11, fontweight="bold")
    ax_cal.legend(fontsize=8)
    ax_cal.grid(alpha=0.25)
    ax_cal.spines[["top", "right"]].set_visible(False)

    # ── Row 2, Col 2: Quartile heatmap (Temporal Validation) ─────────
    ax_hm = fig.add_subplot(gs[1, 2])
    test_data = splits_data.get("Temporal Validation", list(splits_data.values())[-1])
    df_q = quartile_stratified_metrics(
        test_data["y_true"], test_data["y_prob"], test_data["TU"],
        split_name="Temporal Validation"
    )
    if not df_q.empty:
        hm_metrics = ["AUROC", "AUPR", "Brier", "F1", "Accuracy", "Sensitivity", "Specificity"]
        hm_data = df_q.set_index("Quartile")[hm_metrics].astype(float).values
        hm_labels = df_q["Quartile"].values

        im = ax_hm.imshow(hm_data, aspect="auto", cmap="RdYlGn",
                          vmin=0, vmax=1, interpolation="nearest")
        ax_hm.set_xticks(range(len(hm_metrics)))
        ax_hm.set_xticklabels(hm_metrics, rotation=35, ha="right", fontsize=9)
        ax_hm.set_yticks(range(len(hm_labels)))
        ax_hm.set_yticklabels(hm_labels, fontsize=9)
        for r in range(hm_data.shape[0]):
            for c in range(hm_data.shape[1]):
                v = hm_data[r, c]
                tc = "white" if (v < 0.3 or v > 0.85) else "black"
                ax_hm.text(c, r, f"{v:.3f}", ha="center", va="center",
                           fontsize=9, color=tc, fontweight="bold")
        plt.colorbar(im, ax=ax_hm, fraction=0.04, pad=0.03)
    ax_hm.set_title("Metrics by TU Quartile\n(Temporal Validation)",
                     fontsize=11, fontweight="bold")

    fig.suptitle("High-Confidence Sample Discriminative Performance Evaluation",
                 fontsize=14, fontweight="bold")

    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")
    print(f"\n✓ Figure saved → {output_path}")
    plt.show()
    plt.close(fig)


# ═══════════════════════════════════════════════════════════════════════════
# § 6  Console report
# ═══════════════════════════════════════════════════════════════════════════

def print_confidence_report(splits_data, tu_thr):
    """Print formatted head-to-head and quartile tables."""
    sep = "═" * 100
    print(f"\n{sep}")
    print("  HIGH-CONFIDENCE SAMPLE DISCRIMINATIVE PERFORMANCE EVALUATION")
    print(f"  TU threshold (val median): {tu_thr:.4f}")
    print(sep)

    # ── Head-to-head: Certain vs Uncertain ──
    all_h2h = []
    for split_name, sdata in splits_data.items():
        df = certain_vs_uncertain(
            sdata["y_true"], sdata["y_prob"], sdata["TU"],
            tu_thr, split_name
        )
        all_h2h.append(df)
    df_h2h = pd.concat(all_h2h, ignore_index=True)

    print("\n── Certain vs. Uncertain Head-to-Head ──")
    cols = ["Split", "Subset", "n", "Events", "Prevalence",
            "AUROC", "AUROC_CI_lo", "AUROC_CI_hi",
            "AUPR", "Brier", "F1", "Accuracy", "Sensitivity", "Specificity", "PPV", "NPV"]
    available = [c for c in cols if c in df_h2h.columns]
    print(df_h2h[available].to_string(index=False, float_format="%.4f"))

    # ── Quartile analysis ──
    all_q = []
    for split_name, sdata in splits_data.items():
        df = quartile_stratified_metrics(
            sdata["y_true"], sdata["y_prob"], sdata["TU"], split_name
        )
        all_q.append(df)
    df_q = pd.concat(all_q, ignore_index=True)

    print(f"\n── TU-Quartile Stratified Metrics ──")
    q_cols = ["Split", "Quartile", "n", "Events", "TU_mean",
              "AUROC", "AUPR", "Brier", "F1", "Accuracy", "Sensitivity", "Specificity"]
    available_q = [c for c in q_cols if c in df_q.columns]
    print(df_q[available_q].to_string(index=False, float_format="%.4f"))

    # ── Key finding highlight ──
    print(f"\n── Key Findings ──")
    for split_name, sdata in splits_data.items():
        mask_c = sdata["TU"] < tu_thr
        mask_u = sdata["TU"] >= tu_thr
        if mask_c.sum() < 10 or mask_u.sum() < 10:
            continue
        yt_c, yp_c = sdata["y_true"][mask_c], sdata["y_prob"][mask_c]
        yt_u, yp_u = sdata["y_true"][mask_u], sdata["y_prob"][mask_u]
        if len(np.unique(yt_c)) < 2 or len(np.unique(yt_u)) < 2:
            continue
        auc_c = roc_auc_score(yt_c, yp_c)
        auc_u = roc_auc_score(yt_u, yp_u)
        brier_c = brier_score_loss(yt_c, yp_c)
        brier_u = brier_score_loss(yt_u, yp_u)
        acc_c = accuracy_score(yt_c, (yp_c >= 0.5).astype(int))
        acc_u = accuracy_score(yt_u, (yp_u >= 0.5).astype(int))
        print(f"\n  {split_name}:")
        print(f"    Certain  (n={mask_c.sum():>5})  AUROC={auc_c:.4f}  Brier={brier_c:.4f}  Acc={acc_c*100:.2f}%")
        print(f"    Uncertain(n={mask_u.sum():>5})  AUROC={auc_u:.4f}  Brier={brier_u:.4f}  Acc={acc_u*100:.2f}%")
        print(f"    Δ AUROC={auc_c-auc_u:+.4f}   Δ Brier={brier_c-brier_u:+.4f}   Δ Acc={((acc_c-acc_u)*100):+.2f}%")

    print(sep)
    return df_h2h, df_q


# ═══════════════════════════════════════════════════════════════════════════
# § 7  CSV export
# ═══════════════════════════════════════════════════════════════════════════

def export_confidence_metrics(splits_data, tu_thr, output_path="high_confidence_metrics.csv"):
    """Export all confidence-stratified metrics to CSV."""
    all_rows = []
    for split_name, sdata in splits_data.items():
        df_h = certain_vs_uncertain(
            sdata["y_true"], sdata["y_prob"], sdata["TU"], tu_thr, split_name
        )
        all_rows.append(df_h)
        df_q = quartile_stratified_metrics(
            sdata["y_true"], sdata["y_prob"], sdata["TU"], split_name
        )
        if not df_q.empty:
            df_q["Subset"] = df_q["Quartile"]
        all_rows.append(df_q)
    df_all = pd.concat(all_rows, ignore_index=True)
    df_all.to_csv(output_path, index=False)
    print(f"✓ CSV saved → {output_path}")
    return df_all


# ═══════════════════════════════════════════════════════════════════════════
# § 7b  Formatted XLSX Export (publication-quality)
# ═══════════════════════════════════════════════════════════════════════════

def _fmt_ci_str(val, lo, hi, decimals=4):
    """Format value (95% CI) string."""
    if np.isnan(val): return "—"
    fmt = f".{decimals}f"
    return f"{val:{fmt}} [{lo:{fmt}}–{hi:{fmt}}]"

def _fmt_val(val, decimals=4):
    if val is None or (isinstance(val, float) and np.isnan(val)): return "—"
    return round(val, decimals)


def export_formatted_xlsx(splits_data, tu_thr,
                           output_path="high_confidence_metrics.xlsx"):
    """
    Generate a publication-quality formatted Excel workbook with 3 sheets:
      Sheet 1: Certain vs Uncertain head-to-head
      Sheet 2: TU Quartile Stratification
      Sheet 3: Progressive Confidence Filtering
    """
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

    wb = Workbook()

    # ── Style definitions ──
    HEADER_FILL   = PatternFill("solid", fgColor="2E4057")
    HEADER_FONT   = Font(name="Arial", size=10, bold=True, color="FFFFFF")
    SUBHDR_FILL   = PatternFill("solid", fgColor="3D5A80")
    SUBHDR_FONT   = Font(name="Arial", size=9.5, bold=True, color="FFFFFF")
    SPLIT_FILL    = PatternFill("solid", fgColor="E8EEF4")
    SPLIT_FONT    = Font(name="Arial", size=10, bold=True, color="1A237E")
    BODY_FONT     = Font(name="Arial", size=9.5)
    BOLD_FONT     = Font(name="Arial", size=9.5, bold=True)
    NOTE_FONT     = Font(name="Arial", size=8.5, italic=True, color="666666")
    CERTAIN_FILL  = PatternFill("solid", fgColor="E8F5E9")
    UNCERTAIN_FILL= PatternFill("solid", fgColor="FFF3E0")
    FULL_FILL     = PatternFill("solid", fgColor="F5F5F5")
    Q_FILLS       = [PatternFill("solid", fgColor=c) for c in ["E8F5E9","FFF9C4","FFE0B2","FFCDD2"]]
    thin_b = Border(*[Side(style="thin", color="CCCCCC")]*4)
    ctr = Alignment(horizontal="center", vertical="center", wrap_text=True)
    lft = Alignment(horizontal="left",   vertical="center", wrap_text=True)

    def style_row(ws, r, c1, c2, font=None, fill=None, align=None):
        for c in range(c1, c2+1):
            cl = ws.cell(row=r, column=c)
            cl.border = thin_b
            if font:  cl.font = font
            if fill:  cl.fill = fill
            if align: cl.alignment = align

    split_names = list(splits_data.keys())

    # ════════════════════════════════════════════════════════════
    # Sheet 1: Certain vs Uncertain
    # ════════════════════════════════════════════════════════════
    ws1 = wb.active
    ws1.title = "Certain vs Uncertain"
    ws1.sheet_properties.tabColor = "2E7D32"

    widths1 = {"A":14,"B":24,"C":8,"D":8,"E":10,"F":9,"G":22,
               "H":9,"I":22,"J":9,"K":22,"L":9,"M":10,"N":10,"O":8,"P":8,"Q":8}
    for col, w in widths1.items():
        ws1.column_dimensions[col].width = w

    ws1.merge_cells("A1:Q1")
    ws1["A1"] = ("Table: High-Confidence Discriminative Performance "
                 "— Certain vs. Uncertain (TU Median Split)")
    ws1["A1"].font = Font(name="Arial", size=12, bold=True, color="1A237E")
    ws1["A1"].alignment = lft

    for s, e, lbl in [("A2","E2","Sample"),("F2","G2","AUROC"),
                       ("H2","I2","AUPR"),("J2","K2","Brier"),
                       ("L2","Q2","Classification (threshold=0.5)")]:
        ws1.merge_cells(f"{s}:{e}"); ws1[s] = lbl
    style_row(ws1, 2, 1, 17, HEADER_FONT, HEADER_FILL, ctr)

    sub1 = ["Dataset","Subset","n","Events","Prevalence",
            "Value","95% CI","Value","95% CI","Value","95% CI",
            "Accuracy","Sensitivity","Specificity","PPV","NPV","F1"]
    for i, lbl in enumerate(sub1, 1): ws1.cell(row=3, column=i, value=lbl)
    style_row(ws1, 3, 1, 17, SUBHDR_FONT, SUBHDR_FILL, ctr)

    subset_fills = {
        "Certain (TU < median)":   CERTAIN_FILL,
        "Uncertain (TU ≥ median)": UNCERTAIN_FILL,
        "Full sample":             FULL_FILL,
    }

    r = 4
    for sname in split_names:
        ws1.merge_cells(f"A{r}:Q{r}")
        ws1.cell(row=r, column=1, value=sname)
        style_row(ws1, r, 1, 17, SPLIT_FONT, SPLIT_FILL, lft)
        r += 1

        sdata = splits_data[sname]
        df_h = certain_vs_uncertain(sdata["y_true"], sdata["y_prob"],
                                     sdata["TU"], tu_thr, sname)
        for _, row_data in df_h.iterrows():
            subset = row_data.get("Subset", "")
            fill = subset_fills.get(subset, None)
            vals = [
                "",
                subset,
                int(row_data.get("n", 0)),
                int(row_data.get("Events", 0)),
                _fmt_val(row_data.get("Prevalence"), 4),
                _fmt_val(row_data.get("AUROC"), 4),
                f"[{_fmt_val(row_data.get('AUROC_CI_lo'),4)}–{_fmt_val(row_data.get('AUROC_CI_hi'),4)}]",
                _fmt_val(row_data.get("AUPR"), 4),
                f"[{_fmt_val(row_data.get('AUPR_CI_lo'),4)}–{_fmt_val(row_data.get('AUPR_CI_hi'),4)}]",
                _fmt_val(row_data.get("Brier"), 4),
                f"[{_fmt_val(row_data.get('Brier_CI_lo'),4)}–{_fmt_val(row_data.get('Brier_CI_hi'),4)}]",
                _fmt_val(row_data.get("Accuracy"), 4),
                _fmt_val(row_data.get("Sensitivity"), 4),
                _fmt_val(row_data.get("Specificity"), 4),
                _fmt_val(row_data.get("PPV"), 4),
                _fmt_val(row_data.get("NPV"), 4),
                _fmt_val(row_data.get("F1"), 4),
            ]
            for c, v in enumerate(vals, 1):
                ws1.cell(row=r, column=c, value=v)
            style_row(ws1, r, 1, 17, BODY_FONT, fill, ctr)
            ws1.cell(row=r, column=2).font = BOLD_FONT
            ws1.cell(row=r, column=2).alignment = lft
            r += 1

    r += 1
    ws1.merge_cells(f"A{r}:Q{r}")
    ws1[f"A{r}"] = (f"Notes: TU threshold τ_u = {tu_thr:.4f} (median TU, internal validation). "
                     "95% CI: percentile bootstrap (B=200, seed=42). "
                     "AUROC = area under ROC; AUPR = area under precision-recall curve.")
    ws1[f"A{r}"].font = NOTE_FONT
    ws1[f"A{r}"].alignment = Alignment(wrap_text=True, vertical="top")
    ws1.row_dimensions[r].height = 40

    # ════════════════════════════════════════════════════════════
    # Sheet 2: TU Quartile
    # ════════════════════════════════════════════════════════════
    ws2 = wb.create_sheet("TU Quartile"); ws2.sheet_properties.tabColor = "E65100"
    widths2 = {"A":14,"B":22,"C":24,"D":10,"E":8,"F":8,"G":10,
               "H":9,"I":22,"J":9,"K":22,"L":9,"M":22,"N":9,"O":10,"P":10,"Q":8}
    for col, w in widths2.items(): ws2.column_dimensions[col].width = w

    ws2.merge_cells("A1:Q1")
    ws2["A1"] = "Table: Discriminative Performance Stratified by Uncertainty Quartile (TU)"
    ws2["A1"].font = Font(name="Arial", size=12, bold=True, color="1A237E")
    ws2["A1"].alignment = lft

    for s, e, lbl in [("A2","G2","Sample & Uncertainty"),("H2","I2","AUROC"),
                       ("J2","K2","AUPR"),("L2","M2","Brier"),
                       ("N2","Q2","Classification")]:
        ws2.merge_cells(f"{s}:{e}"); ws2[s] = lbl
    style_row(ws2, 2, 1, 17, HEADER_FONT, HEADER_FILL, ctr)

    sub2 = ["Dataset","Quartile","TU Range","Mean TU","n","Events","Prevalence",
            "Value","95% CI","Value","95% CI","Value","95% CI",
            "Accuracy","Sensitivity","Specificity","F1"]
    for i, lbl in enumerate(sub2, 1): ws2.cell(row=3, column=i, value=lbl)
    style_row(ws2, 3, 1, 17, SUBHDR_FONT, SUBHDR_FILL, ctr)

    r2 = 4
    for sname in split_names:
        ws2.merge_cells(f"A{r2}:Q{r2}")
        ws2.cell(row=r2, column=1, value=sname)
        style_row(ws2, r2, 1, 17, SPLIT_FONT, SPLIT_FILL, lft)
        r2 += 1
        sdata = splits_data[sname]
        df_q = quartile_stratified_metrics(sdata["y_true"], sdata["y_prob"],
                                            sdata["TU"], sname)
        for qi, (_, qr) in enumerate(df_q.iterrows()):
            fill = Q_FILLS[qi] if qi < 4 else None
            vals = [
                "",
                qr.get("Quartile", ""),
                qr.get("TU_range", ""),
                _fmt_val(qr.get("TU_mean"), 4),
                int(qr.get("n", 0)),
                int(qr.get("Events", 0)),
                _fmt_val(qr.get("Prevalence"), 4),
                _fmt_val(qr.get("AUROC"), 4),
                f"[{_fmt_val(qr.get('AUROC_CI_lo'),4)}–{_fmt_val(qr.get('AUROC_CI_hi'),4)}]",
                _fmt_val(qr.get("AUPR"), 4),
                f"[{_fmt_val(qr.get('AUPR_CI_lo'),4)}–{_fmt_val(qr.get('AUPR_CI_hi'),4)}]",
                _fmt_val(qr.get("Brier"), 4),
                f"[{_fmt_val(qr.get('Brier_CI_lo'),4)}–{_fmt_val(qr.get('Brier_CI_hi'),4)}]",
                _fmt_val(qr.get("Accuracy"), 4),
                _fmt_val(qr.get("Sensitivity"), 4),
                _fmt_val(qr.get("Specificity"), 4),
                _fmt_val(qr.get("F1"), 4),
            ]
            for c, v in enumerate(vals, 1): ws2.cell(row=r2, column=c, value=v)
            style_row(ws2, r2, 1, 17, BODY_FONT, fill, ctr)
            ws2.cell(row=r2, column=2).font = BOLD_FONT
            ws2.cell(row=r2, column=2).alignment = lft
            r2 += 1

    r2 += 1
    ws2.merge_cells(f"A{r2}:Q{r2}")
    ws2[f"A{r2}"] = ("Notes: Q1 = lowest 25% TU (most confident); Q4 = highest 25% TU (least confident). "
                      "Quartile boundaries computed per split. "
                      "Monotonic improvement from Q4→Q1 confirms well-calibrated uncertainty.")
    ws2[f"A{r2}"].font = NOTE_FONT
    ws2[f"A{r2}"].alignment = Alignment(wrap_text=True, vertical="top")
    ws2.row_dimensions[r2].height = 40

    # ════════════════════════════════════════════════════════════
    # Sheet 3: Progressive Filtering
    # ════════════════════════════════════════════════════════════
    ws3 = wb.create_sheet("Progressive Filtering"); ws3.sheet_properties.tabColor = "1565C0"
    widths3 = {"A":14,"B":14,"C":8,"D":14,"E":10,"F":10,"G":10,
               "H":10,"I":10,"J":11,"K":11,"L":8}
    for col, w in widths3.items(): ws3.column_dimensions[col].width = w

    ws3.merge_cells("A1:L1")
    ws3["A1"] = "Table: Model Performance Under Progressive Confidence Filtering"
    ws3["A1"].font = Font(name="Arial", size=12, bold=True, color="1A237E")
    ws3["A1"].alignment = lft

    hdr3 = ["Dataset","Retention(%)","n","TU Threshold","Prevalence",
            "AUROC","AUPR","Brier","Accuracy","Sensitivity","Specificity","F1"]
    for i, lbl in enumerate(hdr3, 1): ws3.cell(row=2, column=i, value=lbl)
    style_row(ws3, 2, 1, 12, HEADER_FONT, HEADER_FILL, ctr)

    r3 = 3
    for sname in split_names:
        ws3.merge_cells(f"A{r3}:L{r3}")
        ws3.cell(row=r3, column=1, value=sname)
        style_row(ws3, r3, 1, 12, SPLIT_FONT, SPLIT_FILL, lft)
        r3 += 1
        sdata = splits_data[sname]
        df_p = progressive_confidence_curves(sdata["y_true"], sdata["y_prob"], sdata["TU"])
        # Sample at representative retention levels
        targets = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]
        for tgt in targets:
            idx = (df_p["Retention"] - tgt).abs().idxmin()
            pr = df_p.loc[idx]
            fill = CERTAIN_FILL if tgt <= 0.5 else (FULL_FILL if tgt >= 0.9 else None)
            vals = [
                "",
                round(pr["Retention"]*100, 1),
                int(pr["n"]),
                _fmt_val(pr["TU_threshold"], 4),
                _fmt_val(pr["Prevalence"], 4),
                _fmt_val(pr["AUROC"], 4),
                _fmt_val(pr["AUPR"], 4),
                _fmt_val(pr["Brier"], 4),
                _fmt_val(pr["Accuracy"], 4),
                _fmt_val(pr["Sensitivity"], 4),
                _fmt_val(pr["Specificity"], 4),
                _fmt_val(pr["F1"], 4),
            ]
            for c, v in enumerate(vals, 1): ws3.cell(row=r3, column=c, value=v)
            style_row(ws3, r3, 1, 12, BODY_FONT, fill, ctr)
            r3 += 1

    r3 += 1
    ws3.merge_cells(f"A{r3}:L{r3}")
    ws3[f"A{r3}"] = ("Notes: Retention = % of samples retained after excluding highest-TU samples. "
                      "Green rows = ≤50% retention (high-confidence subset).")
    ws3[f"A{r3}"].font = NOTE_FONT
    ws3[f"A{r3}"].alignment = Alignment(wrap_text=True, vertical="top")
    ws3.row_dimensions[r3].height = 30

    # ── Finalize ──
    for ws in [ws1, ws2, ws3]:
        ws.freeze_panes = "A4" if ws != ws3 else "A3"
        ws.sheet_view.showGridLines = False

    wb.save(output_path)
    print(f"✓ Formatted XLSX saved → {output_path}")
    return output_path


# ═══════════════════════════════════════════════════════════════════════════
# § 8  Entry point
# ═══════════════════════════════════════════════════════════════════════════

# ── Build splits_data from existing notebook variables ──
# tu_thr was computed from validation set in Cell 18: tu_thr = median(TU_val)
# predictions_train/val/test and y_train/val/test are defined in Cells 5/8

splits_data = {}
for split_name, preds, y in [
    ("Training",            predictions_train, y_train),
    ("Internal Validation", predictions_val,   y_val),
    ("Temporal Validation", predictions_test,  y_test),
]:
    y_arr = np.asarray(y).reshape(-1)
    preds_arr = np.asarray(preds)
    mean_p = preds_arr.mean(axis=0)
    p1 = np.clip(mean_p, 1e-10, 1 - 1e-10)
    p0 = np.clip(1 - mean_p, 1e-10, 1 - 1e-10)
    TU_arr = -(p1 * np.log(p1) + p0 * np.log(p0))
    splits_data[split_name] = {
        "y_true": y_arr,
        "y_prob": mean_p,
        "TU":     TU_arr,
        "predictions": preds_arr,
    }

# ── tu_thr from validation set (consistent with §2.6) ──
# (tu_thr should already be defined from Cell 18; recompute for safety)
val_TU = splits_data["Internal Validation"]["TU"]
tu_thr_val = float(np.percentile(val_TU, 50))
print(f"TU threshold (validation median): {tu_thr_val:.4f}")

# ── Run evaluation ──
df_h2h, df_q = print_confidence_report(splits_data, tu_thr_val)

plot_high_confidence_evaluation(
    splits_data,
    tu_thr=tu_thr_val,
    output_path="high_confidence_evaluation.tiff"
)

df_export = export_confidence_metrics(
    splits_data, tu_thr_val,
    output_path="high_confidence_metrics.csv"
)

# ── Formatted XLSX for publication ──
export_formatted_xlsx(
    splits_data, tu_thr_val,
    output_path="high_confidence_metrics.xlsx"
)


## 11. Subgroup Analysis & Table 4

Stratified performance (age group, residence) with 95% CI, DeLong tests.

In [ ]:
"""
table4_analysis.py
==================
Generate all statistics for Table 4:
  Bootstrap ensemble model (300 XGBoost base learners) stratified
  performance on Training / Internal Validation / Temporal Validation sets
  with 95% bootstrap confidence intervals.

Stratification dimensions:
  - Overall (full cohort)
  - Age group  : <25 / 25-29 / 30-34 / >=35
  - Residence  : Urban / Rural

Metrics:
  n (events)  Accuracy  Sensitivity  Specificity  F1
  AUC-ROC (95% CI)   AUC-PR (95% CI)
  Calibration Slope (95% CI)   Calibration Intercept (95% CI)

Outputs:
  table4_data_raw.csv      -- long-format raw data (machine-readable)
  table4_formatted.csv     -- journal-ready format (value + CI merged)
  table4_delong_tests.csv  -- DeLong AUC comparison between strata
  table4_stats_report.txt  -- console report snapshot

Real-data integration
---------------------
Call load_data() with your arrays.  The returned dict must satisfy:
    {
      "train" / "val" / "test": {
          "y"          : np.ndarray (n,)      -- true labels 0/1
          "y_prob_mat" : np.ndarray (300, n)  -- per-learner class-1 probs
          "meta"       : pd.DataFrame         -- columns WifeAge / WifeResident
      }
    }
"""

from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    confusion_matrix, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve,
)

# ===========================================================================
# § 0  Global configuration
# ===========================================================================

RANDOM_STATE = 42
N_BOOT       = 200   # Bootstrap resampling iterations (for CI) — aligned with Cell 16
N_BASE       = 300   # Number of base learners (XGBoost)
THRESHOLD    = 0.5   # Decision threshold
OUTPUT_DIR   = "./outputs"

# Dataset labels (Table 4 column headers)
SPLIT_LABELS = {
    "train": "Training Set",
    "val":   "Internal Validation Set",
    "test":  "Temporal Validation Set",
}

# Stratification axes
# WifeAge     : continuous -> 4 fixed age bands
# WifeResident: categorical, original encoding  1=Urban / 0=Rural
STRAT_AXES = {
    "WifeAge": {
        "age_lt25":  "Age <25",
        "age_25_29": "Age 25-29",
        "age_30_34": "Age 30-34",
        "age_ge35":  "Age >=35",
    },
    "WifeResident": {
        "urban": "Urban",
        "rural": "Rural",
    },
}

# Table 4 metric columns (matches journal table header order)
METRIC_COLS = [
    "Accuracy", "Sensitivity", "Specificity", "F1",
    "AUC_ROC", "AUC_PR",
    "CalibSlope", "CalibIntercept",
]


# ===========================================================================
# § 1  Data interface
# ===========================================================================

def _build_meta(X: pd.DataFrame) -> pd.DataFrame:
    """
    Build the meta DataFrame from feature matrix X.

    Stratification rules
    --------------------
    WifeAge (continuous) -> 4 fixed age bands (no data leakage):
        < 25            -> 'age_lt25'
        25 <= age < 30  -> 'age_25_29'
        30 <= age < 35  -> 'age_30_34'
        >= 35           -> 'age_ge35'

    WifeResident (categorical, original encoding):
        1 -> 'urban'
        0 -> 'rural'

    Parameters
    ----------
    X : pd.DataFrame with columns 'WifeAge' and 'WifeResident'

    Returns
    -------
    pd.DataFrame with columns ['WifeAge', 'WifeResident'] (string keys)
    """
    X = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X.copy()

    # -- WifeAge: 4-band grouping ----------------------------------------
    if "WifeAge" not in X.columns:
        raise KeyError("Column 'WifeAge' not found in X.")

    age = X["WifeAge"].values
    age_labels = np.select(
        [age < 25, (age >= 25) & (age < 30), (age >= 30) & (age < 35)],
        ["age_lt25", "age_25_29", "age_30_34"],
        default="age_ge35",
    )

    # -- WifeResident: Urban / Rural -------------------------------------
    if "WifeResident" not in X.columns:
        raise KeyError("Column 'WifeResident' not found in X.")

    region_raw    = X["WifeResident"].values
    region_labels = np.where(region_raw == 1, "urban", "rural")

    return pd.DataFrame({
        "WifeAge":      age_labels,
        "WifeResident": region_labels,
    })


def load_data(
    X_train:           pd.DataFrame | np.ndarray,
    y_train:           np.ndarray,
    predictions_train: np.ndarray,
    X_val:             pd.DataFrame | np.ndarray,
    y_val:             np.ndarray,
    predictions_val:   np.ndarray,
    X_test:            pd.DataFrame | np.ndarray,
    y_test:            np.ndarray,
    predictions_test:  np.ndarray,
) -> dict:
    """
    Wrap external real data into the framework's standard format.

    Parameters
    ----------
    X_train / X_val / X_test
        Feature matrices -- pd.DataFrame or np.ndarray (must contain columns
        'WifeAge' and 'WifeResident').  If ndarray, convert first:
            X_train = pd.DataFrame(arr, columns=feature_names)
    y_train / y_val / y_test
        True labels, shape (n,), values 0 or 1.
    predictions_train / predictions_val / predictions_test
        Class-1 probabilities from 300 XGBoost base learners, shape (300, n).

    Returns
    -------
    dict with keys 'train' / 'val' / 'test' in the framework's standard format.

    Typical usage
    -------------
    data = load_data(
        X_train, y_train, predictions_train,
        X_val,   y_val,   predictions_val,
        X_test,  y_test,  predictions_test,
    )
    results = run_table4_analysis(data)
    """
    def _to_df(X):
        return X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)

    X_tr = _to_df(X_train)
    X_va = _to_df(X_val)
    X_te = _to_df(X_test)

    # Print age-band and residence distributions for verification
    for name, X_df in [("Train", X_tr), ("Val", X_va), ("Test", X_te)]:
        age = X_df["WifeAge"].values
        print(f"  [load_data] {name} WifeAge: "
              f"<25={( age<25).sum()}  25-29={(( age>=25)&(age<30)).sum()}  "
              f"30-34={(( age>=30)&(age<35)).sum()}  >=35={(age>=35).sum()}")
        uniq = sorted(X_df["WifeResident"].unique().tolist())
        print(f"  [load_data] {name} WifeResident original values: {uniq}  "
              f"(1=Urban, 0=Rural)")

    return {
        "train": {
            "y":          np.asarray(y_train).astype(int),
            "y_prob_mat": np.asarray(predictions_train),   # shape (300, n_train)
            "meta":       _build_meta(X_tr),
        },
        "val": {
            "y":          np.asarray(y_val).astype(int),
            "y_prob_mat": np.asarray(predictions_val),     # shape (300, n_val)
            "meta":       _build_meta(X_va),
        },
        "test": {
            "y":          np.asarray(y_test).astype(int),
            "y_prob_mat": np.asarray(predictions_test),    # shape (300, n_test)
            "meta":       _build_meta(X_te),
        },
    }


# ===========================================================================
# § 2  Ensemble prediction aggregation
# ===========================================================================

def ensemble_predict(y_prob_mat: np.ndarray) -> np.ndarray:
    """
    Average per-model probabilities -> ensemble mean probability (n_samples,).
    """
    return y_prob_mat.mean(axis=0)


# ===========================================================================
# § 3  Bootstrap confidence intervals (unified interface)
# ===========================================================================

def _calibration_slope_intercept(
    y_true: np.ndarray,
    y_prob: np.ndarray,
) -> tuple[float, float]:
    """
    Logistic calibration regression:
        logit(y) = intercept + slope * logit(p_hat)
    Perfect calibration: intercept=0, slope=1.

    Reference: Van Calster et al. (2016) J Clin Epidemiol.
    """
    from scipy.special import logit as _logit
    from scipy.optimize import minimize

    eps    = 1e-7
    lp     = _logit(np.clip(y_prob, eps, 1 - eps))
    y_true = y_true.astype(float)

    # Negative log-likelihood (binary cross-entropy) with intercept + slope
    def neg_ll(params):
        a, b     = params
        log_odds = a + b * lp
        prob     = 1 / (1 + np.exp(-log_odds))
        prob     = np.clip(prob, eps, 1 - eps)
        return -np.mean(y_true * np.log(prob) + (1 - y_true) * np.log(1 - prob))

    res = minimize(neg_ll, x0=[0.0, 1.0], method="Nelder-Mead",
                   options={"xatol": 1e-6, "fatol": 1e-6, "maxiter": 5000})
    intercept, slope = res.x
    return float(slope), float(intercept)


def _compute_metrics_once(
    y_true:    np.ndarray,
    y_prob:    np.ndarray,
    threshold: float = THRESHOLD,
) -> dict:
    """
    Compute the full metric set for a single evaluation.

    Returns
    -------
    Accuracy / Sensitivity / Specificity / F1
    AUC_ROC        -- area under the ROC curve
    AUC_PR         -- area under the Precision-Recall curve (Average Precision)
    CalibSlope     -- calibration slope  (perfect = 1)
    CalibIntercept -- calibration intercept  (perfect = 0)
    """
    y_pred = (y_prob >= threshold).astype(int)

    cm     = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp = cm[0, 0], cm[0, 1]
    spec   = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    slope, intercept = _calibration_slope_intercept(y_true, y_prob)

    return {
        "Accuracy":       accuracy_score(y_true, y_pred),
        "Sensitivity":    recall_score(y_true, y_pred, zero_division=0),
        "Specificity":    spec,
        "F1":             f1_score(y_true, y_pred, zero_division=0),
        "AUC_ROC":        roc_auc_score(y_true, y_prob),
        "AUC_PR":         average_precision_score(y_true, y_prob),
        "CalibSlope":     slope,
        "CalibIntercept": intercept,
    }


def compute_metrics_with_ci(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    n_boot: int   = N_BOOT,
    alpha:  float = 0.05,
    seed:   int   = RANDOM_STATE,
) -> pd.DataFrame:
    """
    Compute the full metric set + Percentile Bootstrap 95% CI.

    Parameters
    ----------
    y_true  : true labels (n,)
    y_prob  : ensemble mean probabilities (n,)
    n_boot  : number of bootstrap resamples
    alpha   : significance level (default 0.05 -> 95% CI)

    Returns
    -------
    pd.DataFrame with columns
        [metric, point_est, ci_lo, ci_hi, boot_mean, boot_std, n, n_pos, prevalence]
    Metrics: Accuracy / Sensitivity / Specificity / F1 /
             AUC_ROC / AUC_PR / CalibSlope / CalibIntercept
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    n      = len(y_true)
    n_pos  = int(y_true.sum())

    # Point estimates
    point = _compute_metrics_once(y_true, y_prob)

    # Bootstrap resampling
    rng   = np.random.default_rng(seed)
    boots = {k: [] for k in METRIC_COLS}

    for _ in range(n_boot):
        idx  = rng.integers(0, n, n)
        yt_b = y_true[idx]
        yp_b = y_prob[idx]
        if len(np.unique(yt_b)) < 2:   # skip single-class resample
            continue
        m = _compute_metrics_once(yt_b, yp_b)
        for k in METRIC_COLS:
            boots[k].append(m[k])

    lo_p = alpha / 2 * 100
    hi_p = (1 - alpha / 2) * 100

    rows = []
    for k in METRIC_COLS:
        arr = np.array(boots[k])
        rows.append({
            "metric":     k,
            "point_est":  point[k],
            "ci_lo":      np.percentile(arr, lo_p) if len(arr) else np.nan,
            "ci_hi":      np.percentile(arr, hi_p) if len(arr) else np.nan,
            "boot_mean":  arr.mean()               if len(arr) else np.nan,
            "boot_std":   arr.std()                if len(arr) else np.nan,
            "n":          n,
            "n_pos":      n_pos,
            "prevalence": round(n_pos / n, 4),
        })
    return pd.DataFrame(rows)


# ===========================================================================
# § 4  DeLong test (inter-stratum AUC comparison)
# ===========================================================================

def _auc_variance_delong(
    y_true:  np.ndarray,
    y_score: np.ndarray,
) -> tuple[float, float]:
    """
    DeLong variance estimator for a single group.
    Returns (auc, variance).
    Reference: DeLong et al. (1988) Biometrics.
    """
    n1  = int(y_true.sum())           # number of positives
    n0  = int((1 - y_true).sum())     # number of negatives
    pos = y_score[y_true == 1]
    neg = y_score[y_true == 0]

    # Mann-Whitney statistic (equivalent to AUC)
    auc = 0.0
    for p in pos:
        auc += (p > neg).sum() + 0.5 * (p == neg).sum()
    auc /= (n1 * n0)

    # Structural components
    v10 = np.array(
        [(p > neg).sum() + 0.5 * (p == neg).sum() for p in pos]
    ) / n0 - auc
    v01 = np.array(
        [(neg < p).sum() + 0.5 * (neg == p).sum() for p in neg]
    ) / n1 - auc

    s10 = v10.var(ddof=1) if n1 > 1 else 0.0
    s01 = v01.var(ddof=1) if n0 > 1 else 0.0
    var = s10 / n1 + s01 / n0
    return auc, var


def delong_test(
    y_true_a:  np.ndarray, y_score_a: np.ndarray,
    y_true_b:  np.ndarray, y_score_b: np.ndarray,
    label_a:   str = "Group A",
    label_b:   str = "Group B",
) -> dict:
    """
    Independent-samples DeLong test: compare AUC between two groups.

    H0: AUC_A = AUC_B
    Two-sided z-test with normal approximation.

    Parameters
    ----------
    y_true_a/b  : true labels for each group
    y_score_a/b : predicted probabilities for each group

    Returns
    -------
    dict: auc_a, auc_b, auc_diff, z_stat, p_value, significant
    """
    auc_a, var_a = _auc_variance_delong(
        np.asarray(y_true_a), np.asarray(y_score_a))
    auc_b, var_b = _auc_variance_delong(
        np.asarray(y_true_b), np.asarray(y_score_b))

    se = np.sqrt(var_a + var_b)
    z  = (auc_a - auc_b) / se if se > 0 else np.nan
    p  = 2 * (1 - stats.norm.cdf(abs(z))) if not np.isnan(z) else np.nan

    return {
        "group_a":     label_a,
        "group_b":     label_b,
        "auc_a":       round(auc_a, 4),
        "auc_b":       round(auc_b, 4),
        "auc_diff":    round(auc_a - auc_b, 4),
        "z_stat":      round(z, 4) if not np.isnan(z) else np.nan,
        "p_value":     round(p, 4) if not np.isnan(p) else np.nan,
        "significant": bool(p < 0.05) if not np.isnan(p) else False,
    }


# ===========================================================================
# § 5  Main computation: build Table 4 data
# ===========================================================================

def compute_table4(data: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Iterate over all splits (train / val / test) and strata,
    computing metrics + 95% CI for every subgroup.

    Parameters
    ----------
    data : standard-format dict returned by load_data()

    Returns
    -------
    df_raw       : long-format raw data (one row per metric)
    df_formatted : wide-format publication table (value + CI merged strings)
    """
    records = []

    for split, d in data.items():
        y_true     = np.asarray(d["y"])
        y_prob_mat = np.asarray(d["y_prob_mat"])
        meta       = d["meta"]
        y_prob     = ensemble_predict(y_prob_mat)   # ensemble mean probability

        # -- Overall ---------------------------------------------------------
        df_m = compute_metrics_with_ci(y_true, y_prob)
        for _, row in df_m.iterrows():
            records.append({
                "split":   split,
                "stratum": "Overall",
                "group":   "Full Cohort",
                **{k: row[k] for k in
                   ["metric", "point_est", "ci_lo", "ci_hi",
                    "n", "n_pos", "prevalence"]},
            })

        # -- Stratified ------------------------------------------------------
        for cov, groups in STRAT_AXES.items():
            for grp_key, grp_label in groups.items():
                mask = (meta[cov] == grp_key).values
                y_s  = y_true[mask]
                yp_s = y_prob[mask]

                if mask.sum() < 10 or len(np.unique(y_s)) < 2:
                    _warn(f"  Skipping {split}/{cov}/{grp_key}: "
                          f"n={mask.sum()}, classes={np.unique(y_s)}")
                    continue

                stratum_label = "Age Group" if cov == "WifeAge" else "Residence"
                df_m = compute_metrics_with_ci(y_s, yp_s)
                for _, row in df_m.iterrows():
                    records.append({
                        "split":   split,
                        "stratum": stratum_label,
                        "group":   grp_label,
                        **{k: row[k] for k in
                           ["metric", "point_est", "ci_lo", "ci_hi",
                            "n", "n_pos", "prevalence"]},
                    })

    df_raw = pd.DataFrame(records)

    # -- Format: pivot into final Table 4 layout ---------------------------
    df_formatted = _format_table4(df_raw)

    return df_raw, df_formatted


def _format_table4(df_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Pivot long-format df_raw into wide-format Table 4.

    Output columns:
      Stratum | Group | n (Training Set) | AUC_ROC (Training Set) | ...
                                         | AUC_ROC (Internal Validation Set) | ...
    """
    rows = []

    # Row order matching the journal table
    group_order = [
        ("Overall",    "Full Cohort"),
        ("Age Group",  "Age <25"),
        ("Age Group",  "Age 25-29"),
        ("Age Group",  "Age 30-34"),
        ("Age Group",  "Age >=35"),
        ("Residence",  "Urban"),
        ("Residence",  "Rural"),
    ]

    for (stratum, group) in group_order:
        row_base = {"Stratum": stratum, "Group": group}

        for split in ["train", "val", "test"]:
            split_label = SPLIT_LABELS[split]
            sub = df_raw[
                (df_raw["split"]   == split) &
                (df_raw["stratum"] == stratum) &
                (df_raw["group"]   == group)
            ]

            if sub.empty:
                # stratum not present in this split
                row_base[f"n ({split_label})"] = "--"
                for m in METRIC_COLS:
                    row_base[f"{m} ({split_label})"] = "--"
                continue

            # Sample info (n identical across all metric rows; take first)
            n_row = sub.iloc[0]
            row_base[f"n ({split_label})"]     = int(n_row["n"])
            row_base[f"n_pos ({split_label})"] = int(n_row["n_pos"])
            row_base[f"Prev ({split_label})"]  = f"{n_row['prevalence']*100:.1f}%"

            for m in METRIC_COLS:
                m_row = sub[sub["metric"] == m]
                if m_row.empty:
                    row_base[f"{m} ({split_label})"] = "--"
                    continue
                r = m_row.iloc[0]
                v, lo, hi = r["point_est"], r["ci_lo"], r["ci_hi"]

                if m in ("AUC_ROC", "AUC_PR"):
                    # three decimal places with CI
                    row_base[f"{m} ({split_label})"] = \
                        f"{v:.3f} ({lo:.3f}-{hi:.3f})"
                elif m in ("CalibSlope", "CalibIntercept"):
                    # calibration metrics: two decimal places
                    row_base[f"{m} ({split_label})"] = \
                        f"{v:.2f} ({lo:.2f}-{hi:.2f})"
                else:
                    # classification metrics: percentage with CI
                    row_base[f"{m} ({split_label})"] = \
                        f"{v*100:.1f}% ({lo*100:.1f}-{hi*100:.1f})"

        rows.append(row_base)

    return pd.DataFrame(rows)


# ===========================================================================
# § 6  DeLong test matrix: inter-stratum AUC comparisons
# ===========================================================================

def compute_delong_tests(data: dict) -> pd.DataFrame:
    """
    For each split, compute the following AUC pairwise comparisons:
      Age (4 bands): adjacent pairs (lt25 vs 25-29, 25-29 vs 30-34, 30-34 vs ge35)
                     and extreme pair (lt25 vs ge35)
      Residence: Urban vs Rural
      Cross-dataset: Internal Validation vs Temporal Validation (overall AUC)
    """
    results = []

    # -- Age-band pair definitions -----------------------------------------
    age_pairs = [
        ("age_lt25",  "age_25_29"),
        ("age_25_29", "age_30_34"),
        ("age_30_34", "age_ge35"),
        ("age_lt25",  "age_ge35"),   # extreme pair
    ]

    for split, d in data.items():
        y_true = np.asarray(d["y"])
        y_prob = ensemble_predict(np.asarray(d["y_prob_mat"]))
        meta   = d["meta"]
        slabel = SPLIT_LABELS[split]

        # -- Age-band pairs --------------------------------------------------
        for ka, kb in age_pairs:
            la = STRAT_AXES["WifeAge"].get(ka, ka)
            lb = STRAT_AXES["WifeAge"].get(kb, kb)
            mask_a = (meta["WifeAge"] == ka).values
            mask_b = (meta["WifeAge"] == kb).values
            if mask_a.sum() < 10 or mask_b.sum() < 10:
                continue
            if len(np.unique(y_true[mask_a])) < 2 or \
               len(np.unique(y_true[mask_b])) < 2:
                continue
            res = delong_test(
                y_true[mask_a], y_prob[mask_a],
                y_true[mask_b], y_prob[mask_b],
                label_a=f"{la} ({slabel})",
                label_b=f"{lb} ({slabel})",
            )
            res["comparison_type"] = "Age Group AUC Comparison"
            res["dataset"]         = slabel
            results.append(res)

        # -- Residence pairs -------------------------------------------------
        for ka, kb in [("urban", "rural")]:
            la = STRAT_AXES["WifeResident"][ka]
            lb = STRAT_AXES["WifeResident"][kb]
            mask_a = (meta["WifeResident"] == ka).values
            mask_b = (meta["WifeResident"] == kb).values
            if mask_a.sum() < 10 or mask_b.sum() < 10:
                continue
            if len(np.unique(y_true[mask_a])) < 2 or \
               len(np.unique(y_true[mask_b])) < 2:
                continue
            res = delong_test(
                y_true[mask_a], y_prob[mask_a],
                y_true[mask_b], y_prob[mask_b],
                label_a=f"{la} ({slabel})",
                label_b=f"{lb} ({slabel})",
            )
            res["comparison_type"] = "Residence AUC Comparison"
            res["dataset"]         = slabel
            results.append(res)

    # -- Cross-dataset: Internal Validation vs Temporal Validation ----------
    if "val" in data and "test" in data:
        yv = np.asarray(data["val"]["y"])
        pv = ensemble_predict(np.asarray(data["val"]["y_prob_mat"]))
        yt = np.asarray(data["test"]["y"])
        pt = ensemble_predict(np.asarray(data["test"]["y_prob_mat"]))
        if len(np.unique(yv)) >= 2 and len(np.unique(yt)) >= 2:
            res = delong_test(
                yv, pv, yt, pt,
                label_a="Internal Validation (Overall)",
                label_b="Temporal Validation (Overall)",
            )
            res["comparison_type"] = "Cross-dataset Generalization"
            res["dataset"]         = "Val vs Test"
            results.append(res)

    df = pd.DataFrame(results)
    col_order = [
        "comparison_type", "dataset",
        "group_a", "group_b",
        "auc_a", "auc_b", "auc_diff",
        "z_stat", "p_value", "significant",
    ]
    return df[[c for c in col_order if c in df.columns]]


# ===========================================================================
# § 7  Console report
# ===========================================================================

def _warn(msg: str) -> None:
    print(f"  [WARNING] {msg}")


def _section(title: str) -> None:
    bar = "=" * 68
    print(f"\n{bar}\n  {title}\n{bar}")


def print_table4_report(
    df_raw:    pd.DataFrame,
    df_fmt:    pd.DataFrame,
    df_delong: pd.DataFrame,
) -> str:
    """
    Print the formatted Table 4 report and return it as a string (for .txt export).
    """
    lines = []

    def p(*args):
        s = " ".join(str(a) for a in args)
        print(s); lines.append(s)

    p("=" * 68)
    p("  Table 4 -- Bootstrap Ensemble Model (300 XGBoost Base Learners)")
    p(f"  Bootstrap CI: B={N_BOOT} resamples, Percentile method, threshold={THRESHOLD}")
    p("=" * 68)

    # -- Overall metrics per split ------------------------------------------
    _section("1. Overall Performance")
    p("")

    for split, label in SPLIT_LABELS.items():
        sub = df_raw[
            (df_raw["split"]   == split) &
            (df_raw["stratum"] == "Overall")
        ]
        if sub.empty:
            continue

        # Basic sample info
        n_row = sub.iloc[0]
        n_val = int(n_row["n"])
        n_pos = int(n_row["n_pos"])
        prev  = n_row["prevalence"]
        p(f"\n  {label}")
        p(f"  {'-' * 50}")
        p(f"  n={n_val}  Events: {n_pos}  Prevalence: {prev*100:.1f}%")
        p(f"  {'Metric':<18}  {'Point Est.':>10}  {'95% CI':>24}")
        p(f"  {'-' * 50}")

        for m in METRIC_COLS:
            m_row = sub[sub["metric"] == m]
            if m_row.empty:
                continue
            r = m_row.iloc[0]
            v, lo, hi = r["point_est"], r["ci_lo"], r["ci_hi"]
            if m in ("AUC_ROC", "AUC_PR"):
                val_str = f"{v:.4f}"
                ci_str  = f"[{lo:.4f}, {hi:.4f}]"
            elif m in ("CalibSlope", "CalibIntercept"):
                val_str = f"{v:.3f}"
                ci_str  = f"[{lo:.3f}, {hi:.3f}]"
            else:
                val_str = f"{v*100:.2f}%"
                ci_str  = f"[{lo*100:.2f}%, {hi*100:.2f}%]"
            p(f"  {m:<18}  {val_str:>10}  {ci_str:>24}")

    # -- Stratified performance ---------------------------------------------
    for stratum_label in ["Age Group", "Residence"]:
        _section(f"2. Stratified Performance -- {stratum_label}")
        p("")

        for split, label in SPLIT_LABELS.items():
            p(f"  {label}")
            p(f"  {'-' * 60}")
            header = f"  {'Group':<10}  " + "  ".join(
                f"{m:>10}" for m in METRIC_COLS)
            p(header)
            p(f"  {'-' * 60}")

            for grp in df_raw[df_raw["stratum"] == stratum_label]["group"].unique():
                sub = df_raw[
                    (df_raw["split"]   == split) &
                    (df_raw["stratum"] == stratum_label) &
                    (df_raw["group"]   == grp)
                ]
                if sub.empty:
                    continue

                vals = []
                for m in METRIC_COLS:
                    m_r = sub[sub["metric"] == m]
                    if m_r.empty:
                        vals.append("    --  ")
                        continue
                    v = m_r.iloc[0]["point_est"]
                    if m in ("AUC_ROC", "AUC_PR", "CalibSlope", "CalibIntercept"):
                        vals.append(f"{v:.3f}")
                    else:
                        vals.append(f"{v*100:.1f}%")
                p(f"  {grp:<10}  " + "  ".join(f"{v:>10}" for v in vals))
            p("")

    # -- DeLong tests -------------------------------------------------------
    _section("3. DeLong AUC Comparison Tests")
    p("")
    p(f"  {'Comparison':<52}  {'AUC_A':>7}  {'AUC_B':>7}  "
      f"{'Diff':>7}  {'Z':>7}  {'p-value':>9}  {'Sig':>5}")
    p(f"  {'-' * 104}")
    for _, r in df_delong.iterrows():
        sig_mark = "  *" if r["significant"] else "   "
        p(f"  {r['group_a'][:25]:.<25} vs {r['group_b'][:22]:.<22}"
          f"  {r['auc_a']:>7.4f}  {r['auc_b']:>7.4f}"
          f"  {r['auc_diff']:>+7.4f}  {r['z_stat']:>7.4f}"
          f"  {r['p_value']:>9.4f}{sig_mark}")
    p("\n  * p < 0.05 (two-sided)")

    # -- Key findings -------------------------------------------------------
    _section("4. Key Findings")
    p("")

    # AUC across splits
    for split in ["train", "val", "test"]:
        sub = df_raw[
            (df_raw["split"]   == split) &
            (df_raw["stratum"] == "Overall") &
            (df_raw["metric"]  == "AUC_ROC")
        ]
        if sub.empty:
            continue
        r = sub.iloc[0]
        p(f"  {SPLIT_LABELS[split]:<35}  AUC-ROC = {r['point_est']:.3f} "
          f"(95% CI: {r['ci_lo']:.3f}-{r['ci_hi']:.3f})")

    # Significant stratum differences
    sig_tests = df_delong[df_delong["significant"]]
    if not sig_tests.empty:
        p("\n  Significant AUC differences between strata (p < 0.05):")
        for _, r in sig_tests.iterrows():
            p(f"    {r['group_a']} vs {r['group_b']}: "
              f"delta AUC = {r['auc_diff']:+.4f},  p = {r['p_value']:.4f}")
    else:
        p("\n  No statistically significant AUC differences between strata "
          "(all p >= 0.05)")

    p("\n" + "=" * 68)
    return "\n".join(lines)


# ===========================================================================
# § 8  Output saving
# ===========================================================================

def save_outputs(
    df_raw:     pd.DataFrame,
    df_fmt:     pd.DataFrame,
    df_delong:  pd.DataFrame,
    report:     str,
    output_dir: str = OUTPUT_DIR,
) -> None:
    """Write all results to disk."""
    import os
    os.makedirs(output_dir, exist_ok=True)

    # 1. Raw numeric CSV (machine-readable)
    path_raw = f"{output_dir}/table4_data_raw.csv"
    df_raw.to_csv(path_raw, index=False)
    print(f"\n  OK  Raw data        -> {path_raw}")

    # 2. Formatted table CSV (paste directly into Excel / Word)
    path_fmt = f"{output_dir}/table4_formatted.csv"
    df_fmt.to_csv(path_fmt, index=False)
    print(f"  OK  Formatted table -> {path_fmt}")

    # 3. DeLong test results
    path_dl = f"{output_dir}/table4_delong_tests.csv"
    df_delong.to_csv(path_dl, index=False)
    print(f"  OK  DeLong tests    -> {path_dl}")

    # 4. Report text
    path_rpt = f"{output_dir}/table4_stats_report.txt"
    with open(path_rpt, "w", encoding="utf-8") as f:
        f.write(report)
    print(f"  OK  Stats report    -> {path_rpt}")


# ===========================================================================
# § 9  Optional Word document export (via make_table4.js)
# ===========================================================================

def export_to_docx(df_raw: pd.DataFrame, output_dir: str = OUTPUT_DIR) -> None:
    """
    Export df_raw as JSON for make_table4.js to consume.
    Requires Node.js; silently skipped if not installed.
    """
    import subprocess, shutil

    if shutil.which("node") is None:
        print("  [Skipped] Node.js not found. "
              "Use table4_formatted.csv to populate your Word template manually.")
        return

    # Serialize df_raw to JSON for the JS script
    json_path = f"{output_dir}/table4_data_raw.json"
    df_raw.to_json(json_path, orient="records", indent=2)
    print(f"  OK  JSON data       -> {json_path}  (for make_table4.js)")


# ===========================================================================
# § 10  Main pipeline
# ===========================================================================

def run_table4_analysis(
    data:       dict | None = None,
    output_dir: str         = OUTPUT_DIR,
) -> dict:
    """
    Full Table 4 generation pipeline.

    Parameters
    ----------
    data       : standard-format dict from load_data()
    output_dir : directory for all output files

    Returns
    -------
    dict with keys: df_raw, df_formatted, df_delong
    """
    print("\n" + "=" * 68)
    print("  Table 4 Analysis Pipeline")
    print(f"  N_BOOT={N_BOOT}  N_BASE={N_BASE}  Threshold={THRESHOLD}")
    print("=" * 68)

    # Step 1: data loading
    if data is None:
        raise ValueError(
            "data must not be None.\n"
            "Call load_data() first:\n\n"
            "  data = load_data(\n"
            "      X_train, y_train, predictions_train,\n"
            "      X_val,   y_val,   predictions_val,\n"
            "      X_test,  y_test,  predictions_test,\n"
            "  )\n"
            "  results = run_table4_analysis(data)"
        )
    print("\n[Step 1/4] Data loaded")

    for split, d in data.items():
        y   = np.asarray(d["y"])
        mat = np.asarray(d["y_prob_mat"])
        print(f"  {SPLIT_LABELS[split]:<35}  "
              f"n={len(y)}  n_pos={y.sum()}  "
              f"prevalence={y.mean()*100:.1f}%  "
              f"n_models={mat.shape[0]}")

    # Step 2: metric computation
    print(f"\n[Step 2/4] Computing stratified metrics + Bootstrap "
          f"{N_BOOT}-resample CI ...")
    df_raw, df_fmt = compute_table4(data)
    print(f"  OK  {len(df_raw)} records, "
          f"{df_raw[['split','stratum','group']].drop_duplicates().shape[0]} subgroups")

    # Step 3: DeLong tests
    print("\n[Step 3/4] Running DeLong AUC comparison tests ...")
    df_delong = compute_delong_tests(data)
    print(f"  OK  {len(df_delong)} comparisons, "
          f"{df_delong['significant'].sum()} significant")

    # Step 4: report and save
    print("\n[Step 4/4] Generating report and saving outputs ...")
    report = print_table4_report(df_raw, df_fmt, df_delong)
    save_outputs(df_raw, df_fmt, df_delong, report, output_dir)
    export_to_docx(df_raw, output_dir)

    print("\n" + "=" * 68)
    print("  Table 4 analysis complete")
    print("=" * 68)

    return {
        "df_raw":       df_raw,
        "df_formatted": df_fmt,
        "df_delong":    df_delong,
    }


# ===========================================================================
# § 11  Standalone utility (external use)
# ===========================================================================

def get_formatted_row(
    df_raw:  pd.DataFrame,
    split:   str,
    stratum: str,
    group:   str,
) -> dict:
    """
    Extract formatted metrics for a specific subgroup from df_raw.
    Suitable for directly populating Word / LaTeX tables.

    Example
    -------
    row = get_formatted_row(df_raw, "val", "Age Group", "Age 25-29")
    print(row["AUC_ROC"])   # -> "0.754 (0.679-0.832)"
    """
    result = {}
    sub = df_raw[
        (df_raw["split"]   == split) &
        (df_raw["stratum"] == stratum) &
        (df_raw["group"]   == group)
    ]
    if sub.empty:
        return {}

    result["n"]    = int(sub.iloc[0]["n"])
    result["prev"] = f"{sub.iloc[0]['prevalence']*100:.1f}%"

    for m in METRIC_COLS:
        m_r = sub[sub["metric"] == m]
        if m_r.empty:
            result[m] = "--"
            continue
        r = m_r.iloc[0]
        v, lo, hi = r["point_est"], r["ci_lo"], r["ci_hi"]
        if m in ("AUC_ROC", "AUC_PR", "CalibSlope", "CalibIntercept"):
            result[m] = f"{v:.3f} ({lo:.3f}-{hi:.3f})"
        else:
            result[m] = f"{v*100:.1f}% ({lo*100:.1f}-{hi*100:.1f})"

    return result


# ===========================================================================
# Entry point
# ===========================================================================

if __name__ == "__main__":
    # --------------------------------------------------------------------------
    # Real-data usage
    # The following variables must be prepared upstream:
    #
    #   X_train            pd.DataFrame, shape (n_train, n_features)
    #                      must contain columns 'WifeAge' and 'WifeResident'
    #   y_train            np.ndarray,   shape (n_train,)   labels 0/1
    #   predictions_train  np.ndarray,   shape (300, n_train)  per-learner probs
    #
    #   X_val / y_val / predictions_val    -- internal validation set (same format)
    #   X_test / y_test / predictions_test -- temporal validation set (same format)
    #
    # Usage:
    #
    #   data = load_data(
    #       X_train, y_train, predictions_train,
    #       X_val,   y_val,   predictions_val,
    #       X_test,  y_test,  predictions_test,
    #   )
    #   results = run_table4_analysis(data)
    #
    # --------------------------------------------------------------------------
    # Minimal simulation below (only WifeAge / WifeResident columns)
    # --------------------------------------------------------------------------

    import numpy as _np

    _rng = _np.random.default_rng(42)

    def _sim(n, n_models=300):
        # WifeAge: 15-49, uniform distribution (ensures all four age bands)
        age   = _rng.integers(15, 50, n).astype(float)
        resid = _rng.choice([0, 1], n, p=[0.4, 0.6]).astype(float)
        X_df  = pd.DataFrame({"WifeAge": age, "WifeResident": resid})
        logit = 0.04 * age - 0.3 * resid + _rng.normal(0, 0.5, n)
        prob  = 1 / (1 + _np.exp(-logit))
        y     = _rng.binomial(1, prob).astype(int)
        noise = _rng.normal(0, 0.05, (n_models, n))
        mat   = _np.clip(prob[None, :] + noise, 1e-4, 1 - 1e-4)
        return X_df, y, mat

    X_train_sim, y_train_sim, pred_train_sim = _sim(1000)
    X_val_sim,   y_val_sim,   pred_val_sim   = _sim(400)
    X_test_sim,  y_test_sim,  pred_test_sim  = _sim(400)

    data = load_data(
      X_train, y_train, predictions_train,
      X_val,   y_val,   predictions_val,
      X_test,  y_test,  predictions_test,
    )

    results = run_table4_analysis(data)

    # Formatted table -- AUC-ROC preview
    print("\n  Formatted table -- AUC-ROC preview:")
    pd.set_option("display.max_columns", 5)
    pd.set_option("display.width", 140)
    print(results["df_formatted"][
        ["Stratum", "Group",
         f"AUC_ROC ({SPLIT_LABELS['train']})",
         f"AUC_ROC ({SPLIT_LABELS['val']})",
         f"AUC_ROC ({SPLIT_LABELS['test']})"]
    ].to_string(index=False))

## 12. Tri-Split Comprehensive Evaluation Framework

Calibration + DCA, stratified ROC, four-group heatmap across all splits.

In [ ]:
"""
Bootstrap Ensemble Model — Tri-Split Comprehensive Evaluation Framework
=======================================================================
Extension of the base evaluation module to support three dataset splits:
  Train | Validation | Test

Evaluation modules:
  1.  Calibration curves + embedded DCA        (3 splits × 3×3 grid)
  2.  Stratified performance (age / WifeResident)    (ROC + bar charts)
  3.  Selective-prediction curve + error ROC   (per split × 3 models)
  4.  Four-group heatmap summary               (age + WifeResident × split)

Design principles:
  - Each analysis step is an independent, reusable function
  - All plot text in English; console output bilingual
  - Constants centralised at the top; clear docstrings
  - The base `run_evaluation()` interface is preserved and extended
"""

from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.calibration   import calibration_curve, CalibratedClassifierCV
from sklearn.ensemble      import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model  import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score,
    precision_score, recall_score,
    roc_auc_score, roc_curve,
)
from sklearn.model_selection import StratifiedKFold


# ═══════════════════════════════════════════════════════════════════════════
# § 0  Global configuration
# ═══════════════════════════════════════════════════════════════════════════

RANDOM_STATE   = 42
DPI            = 150
OUTPUT_DIR     = "./outputs"

# Split labels and display colours
SPLITS = {
    "train": {"label": "Train",      "color": "#2196F3"},  # blue
    "val":   {"label": "Validation", "color": "#4CAF50"},  # green
    "test":  {"label": "Test",       "color": "#F44336"},  # red
}

# Stratification axes
STRAT_AXES = {
    "WifeAge": {
        "age_lt25":  {"label": "<25",    "color": "#FF9800"},
        "age_25_29": {"label": "25-29",  "color": "#E65100"},
        "age_30_34": {"label": "30-34",  "color": "#9C27B0"},
        "age_ge35":  {"label": "≥35",    "color": "#3F51B5"},
    },
    "WifeResident": {
        "urban": {"label": "Urban", "color": "#00BCD4"},
        "rural": {"label": "Rural", "color": "#E91E63"},
    },
}

MODEL_COLORS = ["#3F51B5", "#FF5722", "#009688"]

# Selective-prediction
SELECTIVE_POINTS    = 60
COVERAGE_THRESHOLDS = [0.9, 0.8, 0.7, 0.6, 0.5]

# Error-detection ROC ordering
ERROR_ROC_ORDER = [
    "TU (Predictive Entropy)",
    "EU (Mutual Information)",
    "AU (Expected Entropy)",
    "Std (Disagreement)",
    "Boundary Uncertainty",
]

# Probability interval bins for interval analysis
PROB_INTERVALS = [(0.0, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0)]
TOP_K_SAMPLES  = 10


# ═══════════════════════════════════════════════════════════════════════════
# § 1  Data simulation  (replace with real loaders)
# ═══════════════════════════════════════════════════════════════════════════

def _build_meta(X: pd.DataFrame) -> pd.DataFrame:
    """
    Extract stratification variables from feature matrix X.

    WifeAge (continuous) → 4 age bands (fixed cut-points, no data leakage):
        < 25            → 'age_lt25'
        25 ≤ age < 30   → 'age_25_29'
        30 ≤ age < 35   → 'age_30_34'
        ≥ 35            → 'age_ge35'

    WifeResident (categorical, original encoding):
        1 → 'urban'
        0 → 'rural'
    """
    X = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X.copy()

    # ── WifeAge: 4-band grouping ─────────────────────────────────────
    if "WifeAge" not in X.columns:
        raise KeyError("Column 'WifeAge' not found in X.")
    age = X["WifeAge"].values
    age_labels = np.select(
        [age < 25, (age >= 25) & (age < 30), (age >= 30) & (age < 35)],
        ["age_lt25", "age_25_29", "age_30_34"],
        default="age_ge35",
    )

    # ── WifeResident: urban / rural ──────────────────────────────────
    if "WifeResident" not in X.columns:
        raise KeyError("Column 'WifeResident' not found in X.")
    res_labels = np.where(X["WifeResident"].values == 1, "urban", "rural")

    return pd.DataFrame({"WifeAge": age_labels, "WifeResident": res_labels})


def load_data(
    X_train, y_train, predictions_train,
    X_val,   y_val,   predictions_val,
    X_test,  y_test,  predictions_test,
) -> dict:
    """
    Wrap real data into the framework's standard format.

    Parameters
    ----------
    X_*            : pd.DataFrame or np.ndarray — must contain 'WifeAge' and
                     'WifeResident' columns. If ndarray, convert first:
                         X_train = pd.DataFrame(arr, columns=feature_names)
    y_*            : (n,) int array of true labels (0/1)
    predictions_*  : (n_models, n) float array of per-model class-1 probabilities

    Typical call
    ------------
    data = load_data(
        X_train, y_train, predictions_train,
        X_val,   y_val,   predictions_val,
        X_test,  y_test,  predictions_test,
    )
    results = run_evaluation(data)
    """
    def _to_df(X):
        return X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)

    X_tr, X_va, X_te = _to_df(X_train), _to_df(X_val), _to_df(X_test)

    # Print age-band distribution for sanity check
    for name, X_df in [("Train", X_tr), ("Val", X_va), ("Test", X_te)]:
        age = X_df["WifeAge"].values
        print(f"  [load_data] {name}  WifeAge: "
              f"<25={( age<25).sum()}  25-29={(( age>=25)&(age<30)).sum()}  "
              f"30-34={(( age>=30)&(age<35)).sum()}  ≥35={(age>=35).sum()}  "
              f"WifeResident unique={sorted(X_df['WifeResident'].unique().tolist())}")

    return {
        "train": {"y": np.asarray(y_train).astype(int),
                  "predictions": np.asarray(predictions_train),
                  "meta": _build_meta(X_tr), "X": X_tr},
        "val":   {"y": np.asarray(y_val).astype(int),
                  "predictions": np.asarray(predictions_val),
                  "meta": _build_meta(X_va), "X": X_va},
        "test":  {"y": np.asarray(y_test).astype(int),
                  "predictions": np.asarray(predictions_test),
                  "meta": _build_meta(X_te), "X": X_te},
    }


# ═══════════════════════════════════════════════════════════════════════════
# § 2  Core uncertainty decomposition  (preserved from base module)
# ═══════════════════════════════════════════════════════════════════════════

def compute_uncertainties(predictions: np.ndarray, eps: float = 1e-10) -> dict:
    """
    Decompose predictive uncertainty via Shannon entropy.

    Parameters
    ----------
    predictions : (n_models, n_samples) class-1 probability matrix

    Returns
    -------
    dict:
        mean_pred           — ensemble mean probability
        TU                  — H(E[p])   total / predictive uncertainty
        AU                  — E[H(p)]   aleatoric uncertainty
        EU                  — TU − AU   epistemic uncertainty
        std_pred            — std(predictions) across models
        ensemble_pred_class — hard predictions (threshold = 0.5)
    """
    mean_pred = predictions.mean(axis=0)

    p1 = np.clip(mean_pred,     eps, 1 - eps)
    p0 = np.clip(1 - mean_pred, eps, 1 - eps)
    TU = -(p1 * np.log(p1) + p0 * np.log(p0))

    q1 = np.clip(predictions,     eps, 1 - eps)
    q0 = np.clip(1 - predictions, eps, 1 - eps)
    AU = -(q1 * np.log(q1) + q0 * np.log(q0)).mean(axis=0)

    EU = TU - AU
    return {
        "mean_pred":           mean_pred,
        "TU": TU, "AU": AU, "EU": EU,
        "std_pred":            predictions.std(axis=0),
        "ensemble_pred_class": (mean_pred > 0.5).astype(int),
    }


# ═══════════════════════════════════════════════════════════════════════════
# § 3  Metrics helpers
# ═══════════════════════════════════════════════════════════════════════════

def compute_metrics(y_true: np.ndarray,
                    y_prob: np.ndarray,
                    threshold: float = 0.5,
                    n_boot: int = 400,
                    seed: int = RANDOM_STATE) -> dict:
    """
    Full binary classification metric set with bootstrapped 95 % CI on AUC.
    """
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp = cm[0, 0], cm[0, 1]

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc = roc_auc_score(y_true, y_prob)

    # Bootstrap CI on AUC
    rng = np.random.default_rng(seed)
    n = len(y_true)
    boot_aucs = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        boot_aucs.append(roc_auc_score(y_true[idx], y_prob[idx]))
    ci = np.percentile(boot_aucs, [2.5, 97.5]) if boot_aucs else (np.nan, np.nan)

    # AUPR (added per Methods §2.5)
    aupr = average_precision_score(y_true, y_prob)

    # Calibration slope & intercept (added per TRIPOD+AI)
    from scipy.special import logit as _logit_fn
    from scipy.optimize import minimize as _minimize
    try:
        eps_c = 1e-7
        lp = _logit_fn(np.clip(y_prob, eps_c, 1 - eps_c))
        y_f = y_true.astype(float)
        def _neg_ll(params):
            a, b = params
            prob_c = 1 / (1 + np.exp(-(a + b * lp)))
            prob_c = np.clip(prob_c, eps_c, 1 - eps_c)
            return -np.mean(y_f * np.log(prob_c) + (1 - y_f) * np.log(1 - prob_c))
        _res = _minimize(_neg_ll, x0=[0.0, 1.0], method="Nelder-Mead",
                         options={"xatol": 1e-6, "fatol": 1e-6, "maxiter": 5000})
        cal_slope, cal_intercept = float(_res.x[1]), float(_res.x[0])
    except Exception:
        cal_slope, cal_intercept = float("nan"), float("nan")

    return {
        "AUC":          round(roc_auc, 4),
        "AUC_CI_lo":    round(ci[0], 4),
        "AUC_CI_hi":    round(ci[1], 4),
        "AUPR":         round(aupr, 4),
        "Brier":        round(brier_score_loss(y_true, y_prob), 4),
        "CalibSlope":   round(cal_slope, 4),
        "CalibIntercept": round(cal_intercept, 4),
        "Accuracy":     round(accuracy_score(y_true, y_pred), 4),
        "Sensitivity":  round(recall_score(y_true, y_pred, zero_division=0), 4),
        "Specificity":  round(tn / max(1, tn + fp), 4),
        "F1":           round(f1_score(y_true, y_pred, zero_division=0), 4),
        "PPV":          round(precision_score(y_true, y_pred, zero_division=0), 4),
        "confusion_matrix": cm,
        "fpr": fpr, "tpr": tpr,
    }


def build_uncertainty_measures(TU, EU, AU, std_pred, mean_pred) -> dict:
    """Collect all uncertainty signals into a named dict for error-ROC."""
    boundary = 0.5 - np.abs(mean_pred - 0.5)
    return {
        "TU (Predictive Entropy)": TU,
        "EU (Mutual Information)": EU,
        "AU (Expected Entropy)":   AU,
        "Std (Disagreement)":      std_pred,
        "Boundary Uncertainty":    boundary,
    }


# ═══════════════════════════════════════════════════════════════════════════
# § 4  Curve computation
# ═══════════════════════════════════════════════════════════════════════════

def selective_prediction_curve(y_true, y_pred, uncertainty,
                                n_points: int = SELECTIVE_POINTS):
    """
    Coverage–accuracy trade-off (selective prediction).
    Samples with uncertainty ≤ threshold are retained;
    accuracy is recalculated on the retained subset.
    """
    thresholds = np.percentile(uncertainty, np.linspace(0, 99, n_points))
    coverages, accuracies = [], []
    for thr in thresholds:
        mask = uncertainty <= thr
        if mask.any():
            coverages.append(mask.mean())
            accuracies.append(accuracy_score(y_true[mask], y_pred[mask]))
    return np.array(coverages), np.array(accuracies)


def error_detection_roc(errors: np.ndarray,
                        uncertainty_measures: dict) -> dict:
    """
    ROC analysis treating prediction errors as positives.
    Returns per-measure fpr, tpr, AUC, best threshold (Youden's J).
    """
    results = {}
    for name, scores in uncertainty_measures.items():
        scores = np.asarray(scores).reshape(-1)
        fpr, tpr, thresholds = roc_curve(errors, scores)
        auc = roc_auc_score(errors, scores)
        j   = tpr - fpr
        i   = np.argmax(j)
        pred_best = (scores >= thresholds[i]).astype(int)
        results[name] = {
            "fpr": fpr, "tpr": tpr, "auc": auc,
            "best_threshold": thresholds[i],
            "best_tpr":       tpr[i],
            "best_spec":      1 - fpr[i],
            "error_cm":       confusion_matrix(errors, pred_best, labels=[0, 1]),
        }
    return results


def compute_dca(y_true: np.ndarray,
                y_prob: np.ndarray,
                thresholds: np.ndarray | None = None) -> dict:
    """
    Decision Curve Analysis — net benefit over threshold probabilities.
    Returns dicts: nb_model, nb_all, nb_none.
    """
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.99, 100)
    n    = len(y_true)
    prev = y_true.mean()

    nb_model, nb_all, nb_none = [], [], []
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        tp   = ((pred == 1) & (y_true == 1)).sum()
        fp   = ((pred == 1) & (y_true == 0)).sum()
        nb_model.append(tp / n - fp / n * (t / (1 - t)))
        nb_all.append(prev - (1 - prev) * t / (1 - t))
        nb_none.append(0.0)

    return {
        "thresholds": thresholds,
        "nb_model":   np.array(nb_model),
        "nb_all":     np.array(nb_all),
        "nb_none":    np.array(nb_none),
    }


# ═══════════════════════════════════════════════════════════════════════════
# § 5  Console reporting  (preserved & extended)
# ═══════════════════════════════════════════════════════════════════════════

def _section(title: str) -> None:
    print(f"\n{'═' * 64}\n  {title}\n{'═' * 64}")


def report_split_metrics(split_results: dict) -> None:
    """Print a concise comparison table across all three splits."""
    _section("Overall Performance Across Splits")
    cols = ["AUC", "AUPR", "Brier", "CalibSlope", "CalibIntercept",
            "Accuracy", "Sensitivity", "Specificity", "F1", "PPV"]
    header = f"{'Split':<12}" + "".join(f"{c:>14}" for c in cols)
    print(header)
    print("-" * len(header))
    for split, res in split_results.items():
        m = res["metrics"]
        row = f"{SPLITS[split]['label']:<12}" + "".join(f"{m[c]:>14.4f}" for c in cols)
        print(row)


def report_stratified_metrics(strat_results: dict) -> None:
    """Print stratified AUC table for every split × covariate × group."""
    _section("Stratified AUC  (split × covariate × group)")
    for split, by_cov in strat_results.items():
        print(f"\n  ── {SPLITS[split]['label']} ──")
        for cov, by_grp in by_cov.items():
            line = f"    {cov:<12}: "
            for grp, info in by_grp.items():
                line += (f"{STRAT_AXES[cov][grp]['label']}"
                         f"  AUC={info['AUC']:.3f}"
                         f"  [{info['AUC_CI_lo']:.3f},{info['AUC_CI_hi']:.3f}]"
                         f"  (n={info['n']})    ")
            print(line)


def report_selective_prediction(split_label, coverages, accuracies,
                                baseline: float) -> None:
    _section(f"Selective Prediction — {split_label}")
    for thr in COVERAGE_THRESHOLDS:
        i = int(np.argmin(np.abs(coverages - thr)))
        lift = (accuracies[i] - baseline) * 100
        print(f"  Coverage ≈ {coverages[i]*100:.1f}%  →  "
              f"Acc {accuracies[i]*100:.2f}%  (lift {lift:+.2f}%)")


def report_error_roc(split_label: str, roc_results: dict) -> None:
    _section(f"Error-Detection ROC — {split_label}")
    for name in ERROR_ROC_ORDER:
        if name not in roc_results:
            continue
        r  = roc_results[name]
        cm = r["error_cm"]
        print(f"  {name:<34}  AUC={r['auc']:.4f}  "
              f"TPR={r['best_tpr']:.4f}  Spec={r['best_spec']:.4f}"
              f"  CM:[TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}]")


def report_prob_intervals(split_label, mean_pred, y_true, y_pred,
                          TU, AU, EU, errors) -> None:
    _section(f"Probability-Interval Analysis — {split_label}")
    for lo, hi in PROB_INTERVALS:
        mask = (mean_pred >= lo) & (mean_pred < hi)
        n = mask.sum()
        if n == 0:
            continue
        acc = accuracy_score(y_true[mask], y_pred[mask]) * 100
        err = errors[mask].mean() * 100
        print(f"  [{lo:.1f},{hi:.1f})  n={n:4d}  Acc={acc:.1f}%  "
              f"Err={err:.1f}%  TU={TU[mask].mean():.4f}  "
              f"AU={AU[mask].mean():.4f}  EU={EU[mask].mean():.4f}")


def report_extreme_samples(split_label, TU, mean_pred, y_true, y_pred) -> None:
    _section(f"Extreme Uncertainty Samples (top/bottom {TOP_K_SAMPLES}) — {split_label}")
    for desc, idx in [
        ("Highest TU", np.argsort(TU)[-TOP_K_SAMPLES:][::-1]),
        ("Lowest  TU", np.argsort(TU)[:TOP_K_SAMPLES]),
    ]:
        print(f"\n  {desc}:")
        for rank, i in enumerate(idx, 1):
            correct = y_pred[i] == y_true[i]
            print(f"    {rank:02d}  TU={TU[i]:.4f}  p={mean_pred[i]:.3f}"
                  f"  y={y_true[i]}  pred={y_pred[i]}  correct={correct}")


# ═══════════════════════════════════════════════════════════════════════════
# § 6  Figure 1 — Calibration curves + embedded DCA
#        Layout: rows = splits (train / val / test), cols = detail panels
# ═══════════════════════════════════════════════════════════════════════════

def _add_calibration_panel(ax, y_true, y_prob, color, label):
    """Draw calibration curve on `ax`; return Brier & AUC."""
    fraction_pos, mean_pred_bin = calibration_curve(y_true, y_prob, n_bins=10)
    brier = brier_score_loss(y_true, y_prob)
    auc   = roc_auc_score(y_true, y_prob)

    ax.plot([0, 1], [0, 1], "k--", lw=1.2, alpha=0.5)
    ax.plot(mean_pred_bin, fraction_pos, "o-", color=color, lw=2, ms=6,
            label=f"{label}  AUC={auc:.3f}  Brier={brier:.3f}")
    ax.set(xlabel="Mean Predicted Probability", ylabel="Observed Frequency",
           xlim=[0, 1], ylim=[0, 1])
    ax.legend(loc="upper left", fontsize=8)
    ax.grid(alpha=0.3)
    return brier, auc


def _add_dca_panel(ax, y_true, y_prob, color):
    """Draw a compact DCA on `ax`."""
    dca = compute_dca(y_true, y_prob)
    t   = dca["thresholds"]
    ax.plot(t, dca["nb_model"], color=color, lw=1.8, label="Model")
    ax.plot(t, dca["nb_all"],   "b--",  lw=1.2, label="Treat all")
    ax.plot(t, dca["nb_none"],  "k-.",  lw=1.0, label="Treat none")
    prev = y_true.mean()
    ax.set(xlabel="Threshold Probability", ylabel="Net Benefit",
           xlim=[0, 1], ylim=[-0.05, prev * 1.4])
    ax.legend(fontsize=7)
    ax.grid(alpha=0.2)
    ax.set_title("Decision Curve Analysis", fontsize=9, fontweight="bold")


def plot_calibration_dca(split_results: dict,
                          output_path: str) -> None:
    """
    Figure 1: 3-row × 3-column grid
    Row  = split  (train / val / test)
    Col  = [Calibration | DCA | ROC]
    """
    splits_list = list(split_results.keys())
    n_splits = len(splits_list)
    fig, axes = plt.subplots(n_splits, 3, figsize=(18, 5 * n_splits))
    # fig.suptitle("Figure 1 — Calibration Curves, DCA & ROC  (Tri-Split)",
    #              fontsize=13, fontweight="bold")

    for row, split in enumerate(splits_list):
        res    = split_results[split]
        color  = SPLITS[split]["color"]
        label  = SPLITS[split]["label"]
        y_true = res["y_true"]
        y_prob = res["mean_pred"]

        # — Calibration —
        ax_cal = axes[row, 0]
        brier, auc_val = _add_calibration_panel(ax_cal, y_true, y_prob, color, label)
        ax_cal.set_title(f"{label} — Calibration Curve\nAUC={auc_val:.3f}  Brier={brier:.3f}",
                         fontsize=10, fontweight="bold")

        # — DCA —
        ax_dca = axes[row, 1]
        _add_dca_panel(ax_dca, y_true, y_prob, color)
        ax_dca.set_title(f"{label} — Decision Curve Analysis", fontsize=10, fontweight="bold")

        # — ROC with CI —
        ax_roc = axes[row, 2]
        fpr, tpr = res["metrics"]["fpr"], res["metrics"]["tpr"]
        auc_ci   = (res["metrics"]["AUC_CI_lo"], res["metrics"]["AUC_CI_hi"])
        ax_roc.plot(fpr, tpr, color=color, lw=2.5,
                    label=f"{label}  AUC={auc_val:.3f}\n"
                          f"95%CI [{auc_ci[0]:.3f},{auc_ci[1]:.3f}]")
        ax_roc.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
        ax_roc.fill_between(fpr, tpr - 0.03, tpr + 0.03, alpha=0.1, color=color)
        ax_roc.set(xlabel="False Positive Rate", ylabel="True Positive Rate",
                   xlim=[0, 1], ylim=[0, 1.02])
        ax_roc.legend(loc="lower right", fontsize=9)
        ax_roc.grid(alpha=0.3)
        ax_roc.set_title(f"{label} — ROC Curve (with 95% CI)", fontsize=10, fontweight="bold")

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.close()
    print(f"  ✓ Figure 1 saved → {output_path}")


# ═══════════════════════════════════════════════════════════════════════════
# § 7  Figure 2 — Stratified performance (age / WifeResident)
#        Layout: rows = covariate (age / WifeResident),
#                cols = [Train ROC | Val ROC | Test ROC | Bar chart]
# ═══════════════════════════════════════════════════════════════════════════

def plot_stratified_performance(split_results: dict,
                                 strat_results: dict,
                                 output_path: str) -> None:
    """
    Figure 2: 2 rows × 4 columns
    Row  = covariate  (WifeAge | WifeResident)
    Col  = [Train ROC | Val ROC | Test ROC | Grouped bar chart]
    """
    cov_list   = list(STRAT_AXES.keys())
    split_list = list(split_results.keys())
    metric_list = ["AUC", "Sensitivity", "Specificity", "F1"]

    fig, axes = plt.subplots(len(cov_list), 4,
                             figsize=(22, 6 * len(cov_list)))
    # fig.suptitle("Figure 2 — Stratified Performance by Age Group & Region  (Tri-Split)",
    #              fontsize=13, fontweight="bold")

    for row, cov in enumerate(cov_list):
        cov_cfg = STRAT_AXES[cov]
        groups  = list(cov_cfg.keys())

        # ── Columns 0-2: ROC per split ─────────────────────────────────
        for col, split in enumerate(split_list):
            ax   = axes[row, col]
            res  = split_results[split]
            meta = res["meta"]
            ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.4)

            for grp in groups:
                mask = meta[cov].values == grp
                if mask.sum() < 10:
                    continue
                y_sub = res["y_true"][mask]
                p_sub = res["mean_pred"][mask]
                if len(np.unique(y_sub)) < 2:
                    continue
                fpr_g, tpr_g, _ = roc_curve(y_sub, p_sub)
                auc_g = roc_auc_score(y_sub, p_sub)
                clr   = cov_cfg[grp]["color"]
                lbl   = cov_cfg[grp]["label"]
                ax.plot(fpr_g, tpr_g, color=clr, lw=2,
                        label=f"{lbl}  AUC={auc_g:.3f}  n={mask.sum()}")
                ax.fill_between(fpr_g, tpr_g - 0.03, tpr_g + 0.03,
                                alpha=0.08, color=clr)

            ax.set(xlabel="FPR", ylabel="TPR", xlim=[0, 1], ylim=[0, 1.02])
            ax.legend(loc="lower right", fontsize=8)
            ax.grid(alpha=0.3)
            title_cov = "Age Group" if cov == "WifeAge" else "WifeResident"
            ax.set_title(f"{title_cov}  |  {SPLITS[split]['label']} ROC",
                         fontsize=10, fontweight="bold")

        # ── Column 3: grouped bar chart across all splits ───────────────
        ax_bar = axes[row, 3]
        n_grps = len(groups)
        x_pos  = np.arange(len(metric_list))
        bar_w  = 0.72 / (len(split_list) * n_grps)
        offset = 0

        for split in split_list:
            res  = split_results[split]
            meta = res["meta"]
            for grp in groups:
                mask = meta[cov].values == grp
                if mask.sum() < 5:
                    offset += 1
                    continue
                y_sub = res["y_true"][mask]
                p_sub = res["mean_pred"][mask]
                if len(np.unique(y_sub)) < 2:
                    offset += 1
                    continue
                mets  = compute_metrics(y_sub, p_sub, n_boot=100)
                vals  = [mets[m] for m in metric_list]
                clr   = cov_cfg[grp]["color"]
                alpha = {"train": 1.0, "val": 0.7, "test": 0.45}[split]
                hatch = {"train": "", "val": "/", "test": "x"}[split]
                bars  = ax_bar.bar(x_pos + offset * bar_w, vals,
                                   bar_w, color=clr, alpha=alpha,
                                   hatch=hatch, edgecolor="white")
                for b, v in zip(bars, vals):
                    ax_bar.text(b.get_x() + b.get_width() / 2,
                                b.get_height() + 0.01,
                                f"{v:.2f}", ha="center", va="bottom", fontsize=6)
                offset += 1

        ax_bar.set_xticks(x_pos + bar_w * (len(split_list) * n_grps - 1) / 2)
        ax_bar.set_xticklabels(metric_list, fontsize=9)
        ax_bar.set(ylim=[0, 1.18], ylabel="Metric Value")
        title_cov = "Age Group" if cov == "WifeAge" else "WifeResident"
        ax_bar.set_title(f"{title_cov}  |  Metric Comparison (All Splits)",
                         fontsize=10, fontweight="bold")
        ax_bar.grid(alpha=0.3, axis="y")

        # Legend patches
        legend_patches = []
        for split in split_list:
            for grp in groups:
                clr = cov_cfg[grp]["color"]
                alpha_v = {"train": 1.0, "val": 0.7, "test": 0.45}[split]
                lbl = f"{SPLITS[split]['label']}-{cov_cfg[grp]['label']}"
                legend_patches.append(
                    mpatches.Patch(color=clr, alpha=alpha_v, label=lbl))
        ax_bar.legend(handles=legend_patches, fontsize=6, ncol=2, loc="upper right")

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.close()
    print(f"  ✓ Figure 2 saved → {output_path}")


# ═══════════════════════════════════════════════════════════════════════════
# § 8  Figure 3 — Selective prediction + Error-detection ROC
#        Layout: rows = splits, cols = [Selective | ErrorROC | UncertaintyBox]
# ═══════════════════════════════════════════════════════════════════════════

def plot_selective_and_error(split_results: dict,
                              output_path: str) -> None:
    """
    Figure 3: 3 rows × 3 columns
    Row  = split
    Col  = [Selective-prediction | Error-detection ROC | Uncertainty boxplot]
    """
    splits_list = list(split_results.keys())
    n_splits    = len(splits_list)
    fig, axes   = plt.subplots(n_splits, 3, figsize=(18, 5.5 * n_splits))
    # fig.suptitle(
    #     "Figure 3 — Selective Prediction Curves & Error-Detection ROC  (Tri-Split)",
    #     fontsize=13, fontweight="bold",
    # )

    for row, split in enumerate(splits_list):
        res     = split_results[split]
        color   = SPLITS[split]["color"]
        label   = SPLITS[split]["label"]
        y_true  = res["y_true"]
        y_pred  = res["y_pred"]
        mean_p  = res["mean_pred"]
        TU      = res["unc"]["TU"]
        EU      = res["unc"]["EU"]
        AU      = res["unc"]["AU"]
        accuracy = res["metrics"]["Accuracy"]
        errors  = (y_pred != y_true).astype(int)
        correct = y_pred == y_true

        # ── Col 0: Selective prediction ───────────────────────────────
        ax_sel = axes[row, 0]
        cov, acc = selective_prediction_curve(y_true, y_pred, TU)
        ax_sel.plot(cov, acc, color=color, lw=2.5, marker="o", ms=3,
                    label="Selective accuracy")
        ax_sel.axhline(accuracy, ls="--", lw=2, color="grey",
                       label=f"Baseline {accuracy*100:.2f}%")
        ax_sel.fill_between(cov, accuracy, acc,
                            where=(acc >= accuracy), alpha=0.2, color=color,
                            label="Improvement zone")
        ax_sel.set(xlabel="Coverage", ylabel="Accuracy",
                   xlim=[0, 1], ylim=[max(0, accuracy), 1.02])
        ax_sel.legend(fontsize=8)
        ax_sel.grid(alpha=0.3)
        ax_sel.set_title(f"{label} — Selective Prediction", fontsize=10, fontweight="bold")

        # ── Col 1: Error-detection ROC ─────────────────────────────────
        ax_err = axes[row, 1]
        unc_m   = build_uncertainty_measures(TU, EU, AU, res["unc"]["std_pred"], mean_p)
        roc_res = error_detection_roc(errors, unc_m)
        res["error_roc"] = roc_res                # cache for reporting

        for name in ERROR_ROC_ORDER:
            if name not in roc_res:
                continue
            r = roc_res[name]
            ax_err.plot(r["fpr"], r["tpr"], lw=1.8,
                        label=f"{name}\n AUC={r['auc']:.3f}")
        ax_err.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Random")
        ax_err.set(xlabel="FPR", ylabel="TPR",
                   xlim=[-0.01, 1.01], ylim=[-0.01, 1.01])
        ax_err.legend(loc="lower right", fontsize=7)
        ax_err.grid(alpha=0.3)
        ax_err.set_title(f"{label} — Error-Detection ROC", fontsize=10, fontweight="bold")

        # ── Col 2: Uncertainty boxplot (correct vs incorrect) ──────────
        ax_box = axes[row, 2]
        C_WRONG, C_RIGHT = "#FF6B6B", "#4ECDC4"
        positions = np.arange(3) * 3
        for offset, mask, clr in [(-0.5, ~correct, C_WRONG),
                                   (+0.5,  correct, C_RIGHT)]:
            if mask.sum() < 2:
                continue
            bp = ax_box.boxplot(
                [TU[mask], EU[mask], AU[mask]],
                positions=positions + offset, widths=0.8,
                patch_artist=True, medianprops=dict(linewidth=2),
            )
            for box in bp["boxes"]:
                box.set(facecolor=clr, alpha=0.65)

        ax_box.set_xticks(positions)
        ax_box.set_xticklabels(["TU\nPredictive", "EU\nEpistemic", "AU\nAleatoric"],
                                fontsize=9)
        ax_box.set_ylabel("Uncertainty Value")
        ax_box.legend(
            handles=[mpatches.Patch(color=C_WRONG, alpha=0.65, label="Incorrect"),
                     mpatches.Patch(color=C_RIGHT, alpha=0.65, label="Correct")],
            loc="upper right", fontsize=8,
        )
        ax_box.grid(alpha=0.3, axis="y")
        ax_box.set_title(f"{label} — Uncertainty: Correct vs. Incorrect",
                         fontsize=10, fontweight="bold")

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.close()
    print(f"  ✓ Figure 3 saved → {output_path}")


# ═══════════════════════════════════════════════════════════════════════════
# § 9  Figure 4 — Four-group heatmap summary
#        Age × Split (4 combos) and Region × Split (4 combos)
#        Layout: rows = covariate (2), cols = models-as-subsets (1 here,
#                expandable); inner = metric heatmap
# ═══════════════════════════════════════════════════════════════════════════

def plot_four_group_heatmap(split_results: dict,
                             output_path: str) -> None:
    """
    Figure 4: 2 rows × 3 columns heatmap grid
    Row = covariate (WifeAge | WifeResident)
    Col = split (train | val | test)

    Each cell = heatmap of [group × metric].
    Groups per covariate: 2  → 2 rows in each heatmap.
    Metrics: AUC, Brier, Accuracy, Sensitivity, Specificity, F1, PPV  (7 cols).
    """
    metric_cols  = ["AUC", "AUPR", "Brier", "Accuracy", "Sensitivity", "Specificity", "F1", "PPV"]
    cov_list     = list(STRAT_AXES.keys())
    split_list   = list(split_results.keys())

    fig, axes = plt.subplots(len(cov_list), len(split_list),
                             figsize=(7 * len(split_list), 5 * len(cov_list)))
    # fig.suptitle("Figure 4 — Four-Group Stratified Metric Heatmap  (Tri-Split)",
    #              fontsize=13, fontweight="bold")

    for row, cov in enumerate(cov_list):
        cov_cfg = STRAT_AXES[cov]
        groups  = list(cov_cfg.keys())

        for col, split in enumerate(split_list):
            ax   = axes[row, col]
            res  = split_results[split]
            meta = res["meta"]

            table_data, row_labels = [], []
            for grp in groups:
                mask = meta[cov].values == grp
                y_sub = res["y_true"][mask]
                p_sub = res["mean_pred"][mask]
                if mask.sum() < 5 or len(np.unique(y_sub)) < 2:
                    continue
                mets = compute_metrics(y_sub, p_sub, n_boot=100)
                table_data.append([mets[m] for m in metric_cols])
                row_labels.append(cov_cfg[grp]["label"])

            arr = np.array(table_data)
            im  = ax.imshow(arr, aspect="auto", cmap="RdYlGn",
                            vmin=0, vmax=1, interpolation="nearest")

            ax.set_xticks(range(len(metric_cols)))
            ax.set_xticklabels(metric_cols, rotation=40, ha="right", fontsize=9)
            ax.set_yticks(range(len(row_labels)))
            ax.set_yticklabels(row_labels, fontsize=10, fontweight="bold")

            for r_idx in range(arr.shape[0]):
                for c_idx in range(arr.shape[1]):
                    v = arr[r_idx, c_idx]
                    txt_c = "white" if (v < 0.3 or v > 0.8) else "black"
                    ax.text(c_idx, r_idx, f"{v:.3f}",
                            ha="center", va="center",
                            fontsize=9, color=txt_c, fontweight="bold")

            plt.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
            cov_title = "Age Group" if cov == "WifeAge" else "WifeResident"
            ax.set_title(f"{cov_title}  |  {SPLITS[split]['label']}",
                         fontsize=11, fontweight="bold")

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.close()
    print(f"  ✓ Figure 4 saved → {output_path}")


# ═══════════════════════════════════════════════════════════════════════════
# § 10  Figure 5 — Comprehensive 3×3 per-split dashboard
#         (preserved structure from base module, applied to each split)
# ═══════════════════════════════════════════════════════════════════════════

def _plot_roc(ax, y_true, mean_pred, color, label):
    fpr, tpr, _ = roc_curve(y_true, mean_pred)
    auc = roc_auc_score(y_true, mean_pred)
    ax.plot(fpr, tpr, lw=2.5, color=color, label=f"AUC = {auc:.3f}")
    ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax.set(title=f"ROC Curve — {label}", xlabel="FPR", ylabel="TPR")
    ax.legend(); ax.grid(alpha=0.3)


def _plot_tu_dist(ax, TU, correct_mask, label):
    for mask, lbl in [(correct_mask, "Correct"), (~correct_mask, "Incorrect")]:
        ax.hist(TU[mask], bins=30, alpha=0.6, label=lbl, density=True)
    ax.set(title=f"TU Distribution — {label}",
           xlabel="TU (Predictive Entropy)", ylabel="Density")
    ax.legend(); ax.grid(alpha=0.3)


def _plot_prob_dist(ax, mean_pred, y_true, label):
    bins = np.linspace(0, 1, 30)
    for cls, lbl in [(0, "Negative"), (1, "Positive")]:
        ax.hist(mean_pred[y_true == cls], bins=bins, alpha=0.6,
                label=lbl, density=True)
    ax.axvline(0.5, ls="--", lw=2)
    ax.set(title=f"Predicted Probability Distribution — {label}",
           xlabel="Mean Predicted Probability", ylabel="Density")
    ax.legend(); ax.grid(alpha=0.3)


def _plot_agreement(ax, mean_pred, predictions, TU, label):
    agreement = (predictions > 0.5).mean(axis=0)
    sc = ax.scatter(mean_pred, agreement, c=TU, cmap="viridis",
                    alpha=0.7, s=30, edgecolors="black", linewidths=0.2)
    plt.colorbar(sc, ax=ax, label="TU")
    ax.set(title=f"Mean Prediction vs. Agreement — {label}",
           xlabel="Mean Predicted Probability", ylabel="Vote Agreement Ratio")
    ax.grid(alpha=0.3)


def _plot_corr_heatmap(ax, TU, AU, EU, std_pred, mean_pred, errors):
    df_corr = pd.DataFrame({
        "TU": TU, "AU": AU, "EU": EU, "Std": std_pred,
        "Boundary": 0.5 - np.abs(mean_pred - 0.5),
        "Error": errors, "MeanPred": mean_pred,
    }).corr()
    sns.heatmap(df_corr, annot=True, fmt=".2f", cmap="coolwarm",
                center=0, square=True, cbar_kws={"shrink": 0.8}, ax=ax)
    ax.set_title("Uncertainty Correlation Matrix")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha="right")
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)


def plot_split_dashboard(split: str, res: dict,
                          output_path: str) -> None:
    """
    Figure 5a/b/c: One 3×3 dashboard per split.
    Panels:  ROC | TU distribution | Selective prediction
             Confusion matrix | Probability distribution | Agreement scatter
             Error ROC | Uncertainty boxplot | Correlation heatmap
    """
    label   = SPLITS[split]["label"]
    color   = SPLITS[split]["color"]
    y_true  = res["y_true"]
    y_pred  = res["y_pred"]
    mean_p  = res["mean_pred"]
    preds   = res["predictions"]
    TU      = res["unc"]["TU"]
    AU      = res["unc"]["AU"]
    EU      = res["unc"]["EU"]
    std_p   = res["unc"]["std_pred"]
    cm      = res["metrics"]["confusion_matrix"]
    accuracy = res["metrics"]["Accuracy"]
    correct = y_pred == y_true
    errors  = (y_pred != y_true).astype(int)
    unc_m   = build_uncertainty_measures(TU, EU, AU, std_p, mean_p)
    roc_res = error_detection_roc(errors, unc_m)
    cov, acc = selective_prediction_curve(y_true, y_pred, TU)

    fig, axes = plt.subplots(3, 3, figsize=(16, 15))
    fig.suptitle(f"Figure 5 — Comprehensive Dashboard  [{label}]",
                 fontsize=13, fontweight="bold")
    ax = axes.flatten()

    # 0 ROC
    _plot_roc(ax[0], y_true, mean_p, color, label)

    # 1 TU distribution
    _plot_tu_dist(ax[1], TU, correct, label)

    # 2 Selective prediction
    ax[2].plot(cov, acc, lw=2.5, marker="o", ms=3, color=color,
               label="Selective accuracy")
    ax[2].axhline(accuracy, ls="--", lw=2, color="grey",
                  label=f"Baseline {accuracy*100:.2f}%")
    ax[2].fill_between(cov, accuracy, acc,
                       where=(acc >= accuracy), alpha=0.2, color=color)
    ax[2].set(title=f"Selective Prediction — {label}",
              xlabel="Coverage", ylabel="Accuracy", xlim=[0, 1])
    ax[2].legend(); ax[2].grid(alpha=0.3)

    # 3 Confusion matrix
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax[3],
                annot_kws={"size": 14}, cbar_kws={"label": "Count"})
    ax[3].set(title=f"Confusion Matrix — {label}",
              xlabel="Predicted", ylabel="True")

    # 4 Probability distribution
    _plot_prob_dist(ax[4], mean_p, y_true, label)

    # 5 Agreement scatter
    _plot_agreement(ax[5], mean_p, preds, TU, label)

    # 6 Error-detection ROC
    for name in ERROR_ROC_ORDER:
        if name not in roc_res:
            continue
        r = roc_res[name]
        ax[6].plot(r["fpr"], r["tpr"], lw=1.8,
                   label=f"{name} ({r['auc']:.3f})")
    ax[6].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.4, label="Random")
    ax[6].set(title=f"Error-Detection ROC — {label}",
              xlabel="FPR", ylabel="TPR", xlim=[-0.01, 1.01])
    ax[6].legend(loc="lower right", fontsize=7); ax[6].grid(alpha=0.3)

    # 7 Uncertainty boxplot
    C_W, C_R = "#FF6B6B", "#4ECDC4"
    pos = np.arange(3) * 3
    for off, mask, clr in [(-0.5, ~correct, C_W), (+0.5, correct, C_R)]:
        if mask.sum() < 2:
            continue
        bp = ax[7].boxplot([TU[mask], EU[mask], AU[mask]],
                           positions=pos + off, widths=0.8,
                           patch_artist=True,
                           medianprops=dict(linewidth=2))
        for box in bp["boxes"]:
            box.set(facecolor=clr, alpha=0.65)
    ax[7].set_xticks(pos)
    ax[7].set_xticklabels(["TU", "EU", "AU"], fontsize=10)
    ax[7].set(title=f"Uncertainty Boxplot — {label}", ylabel="Value")
    ax[7].legend(handles=[mpatches.Patch(color=C_W, alpha=0.65, label="Incorrect"),
                           mpatches.Patch(color=C_R, alpha=0.65, label="Correct")],
                 loc="upper right")
    ax[7].grid(alpha=0.3, axis="y")

    # 8 Correlation heatmap
    _plot_corr_heatmap(ax[8], TU, AU, EU, std_p, mean_p, errors)

    plt.tight_layout()
    plt.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.close()
    print(f"  ✓ Figure 5 ({label}) saved → {output_path}")


# ═══════════════════════════════════════════════════════════════════════════
# § 11  Summary table export
# ═══════════════════════════════════════════════════════════════════════════

def export_summary_table(split_results: dict,
                          strat_results: dict,
                          output_path: str) -> pd.DataFrame:
    """
    Export one CSV covering:
      - Overall metrics for each split
      - Stratified metrics for every split × covariate × group
    """
    rows = []
    metric_cols = ["AUC", "AUC_CI_lo", "AUC_CI_hi", "AUPR",
                   "Brier", "CalibSlope", "CalibIntercept",
                   "Accuracy", "Sensitivity", "Specificity", "F1", "PPV"]

    # Overall
    for split, res in split_results.items():
        m = res["metrics"]
        row = {"Split": SPLITS[split]["label"], "Subset": "Overall",
               "n": len(res["y_true"]),
               "Prevalence": round(res["y_true"].mean(), 4)}
        row.update({k: m[k] for k in metric_cols})
        rows.append(row)

    # Stratified
    for split, by_cov in strat_results.items():
        for cov, by_grp in by_cov.items():
            for grp, info in by_grp.items():
                cov_label = ("Age" if cov == "WifeAge" else "WifeResident")
                grp_label = STRAT_AXES[cov][grp]["label"]
                row = {
                    "Split":  SPLITS[split]["label"],
                    "Subset": f"{cov_label}-{grp_label}",
                    "n":      info["n"],
                    "Prevalence": info["prevalence"],
                }
                row.update({k: info[k] for k in metric_cols})
                rows.append(row)

    df = pd.DataFrame(rows)
    col_order = ["Split", "Subset", "n", "Prevalence"] + metric_cols
    df = df[col_order]
    df.to_csv(output_path, index=False)
    print(f"  ✓ Summary table saved → {output_path}")
    return df


# ═══════════════════════════════════════════════════════════════════════════
# § 12  Main pipeline
# ═══════════════════════════════════════════════════════════════════════════

def run_evaluation(data: dict,
                   output_dir: str = OUTPUT_DIR) -> dict:
    """
    Full tri-split evaluation pipeline.

    Parameters
    ----------
    data : dict with keys 'train', 'val', 'test'.  Each value must contain:
             y           — (n,) int array of true labels
             predictions — (n_models, n) float array of per-model probabilities
             meta        — DataFrame[WifeAge, WifeResident]
    output_dir : directory for saved figures and CSV

    Returns
    -------
    dict of per-split processed results (useful for custom downstream analysis)
    """
    import os
    os.makedirs(output_dir, exist_ok=True)

    print("\n" + "═" * 64)
    print("  Bootstrap Ensemble — Tri-Split Comprehensive Evaluation")
    print("═" * 64)

    # ── Step 1: Uncertainty decomposition & metrics per split ──────────
    split_results = {}
    for split, d in data.items():
        y_true = np.asarray(d["y"]).reshape(-1)
        preds  = np.asarray(d["predictions"])        # (n_models, n_samples)
        meta   = d["meta"]

        unc       = compute_uncertainties(preds)
        mean_pred = unc["mean_pred"]
        y_pred    = unc["ensemble_pred_class"]
        metrics   = compute_metrics(y_true, mean_pred, n_boot=400)

        split_results[split] = {
            "y_true":      y_true,
            "y_pred":      y_pred,
            "mean_pred":   mean_pred,
            "predictions": preds,
            "unc":         unc,
            "metrics":     metrics,
            "meta":        meta,
        }

    # ── Step 2: Stratified metrics ─────────────────────────────────────
    strat_results = {}
    for split, res in split_results.items():
        strat_results[split] = {}
        meta = res["meta"]
        for cov, grp_cfg in STRAT_AXES.items():
            strat_results[split][cov] = {}
            for grp in grp_cfg:
                mask = meta[cov].values == grp
                y_sub = res["y_true"][mask]
                p_sub = res["mean_pred"][mask]
                if mask.sum() < 5 or len(np.unique(y_sub)) < 2:
                    continue
                m = compute_metrics(y_sub, p_sub, n_boot=200)
                m["n"]          = int(mask.sum())
                m["prevalence"] = round(y_sub.mean(), 4)
                strat_results[split][cov][grp] = m

    # ── Step 3: Console reports ────────────────────────────────────────
    report_split_metrics(split_results)
    report_stratified_metrics(strat_results)

    for split, res in split_results.items():
        label    = SPLITS[split]["label"]
        y_true   = res["y_true"]
        y_pred   = res["y_pred"]
        mean_p   = res["mean_pred"]
        TU       = res["unc"]["TU"]
        AU       = res["unc"]["AU"]
        EU       = res["unc"]["EU"]
        std_p    = res["unc"]["std_pred"]
        accuracy = res["metrics"]["Accuracy"]
        errors   = (y_pred != y_true).astype(int)

        cov, acc = selective_prediction_curve(y_true, y_pred, TU)
        report_selective_prediction(label, cov, acc, accuracy)

        unc_m   = build_uncertainty_measures(TU, EU, AU, std_p, mean_p)
        roc_res = error_detection_roc(errors, unc_m)
        res["error_roc"] = roc_res
        report_error_roc(label, roc_res)

        report_prob_intervals(label, mean_p, y_true, y_pred, TU, AU, EU, errors)
        report_extreme_samples(label, TU, mean_p, y_true, y_pred)

    # ── Step 4: Figures ────────────────────────────────────────────────
    print("\n" + "─" * 64)
    print("  Generating figures …")
    print("─" * 64)

    plot_calibration_dca(
        split_results,
        f"{output_dir}/fig1_calibration_dca.png",
    )
    plot_stratified_performance(
        split_results, strat_results,
        f"{output_dir}/fig2_stratified_performance.png",
    )
    plot_selective_and_error(
        split_results,
        f"{output_dir}/fig3_selective_error_roc.png",
    )
    plot_four_group_heatmap(
        split_results,
        f"{output_dir}/fig4_four_group_heatmap.png",
    )
    for split, res in split_results.items():
        plot_split_dashboard(
            split, res,
            f"{output_dir}/fig5_{split}_dashboard.png",
        )

    # ── Step 5: Summary CSV ───────────────────────────────────────────
    summary_df = export_summary_table(
        split_results, strat_results,
        f"{output_dir}/summary_metrics.csv",
    )

    print("\n" + "═" * 64)
    print("  ✅  Evaluation complete.  Outputs:")
    for fname in [
        "fig1_calibration_dca.png",
        "fig2_stratified_performance.png",
        "fig3_selective_error_roc.png",
        "fig4_four_group_heatmap.png",
        "fig5_train_dashboard.png",
        "fig5_val_dashboard.png",
        "fig5_test_dashboard.png",
        "summary_metrics.csv",
    ]:
        print(f"     {output_dir}/{fname}")
    print("═" * 64)

    return split_results


# ═══════════════════════════════════════════════════════════════════════════
# § 13  Entry point
# ═══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    print("Generating synthetic tri-split data …")
    # data = generate_tri_split(n_train=1000, n_val=400, n_test=400, n_models=10)
    data = load_data(
      X_train, y_train, predictions_train,
      X_val,   y_val,   predictions_val,
      X_test,  y_test,  predictions_test,
    )
    results = run_evaluation(data, output_dir=OUTPUT_DIR)

    # ── Quick peek at summary table ────────────────────────────────────
    print("\nSummary table preview:")
    df = pd.read_csv(f"{OUTPUT_DIR}/summary_metrics.csv")
    cols_show = ["Split", "Subset", "n", "AUC", "AUC_CI_lo", "AUC_CI_hi",
                 "Accuracy", "Sensitivity", "Specificity", "F1"]
    print(df[cols_show].to_string(index=False))

## 13. Feature Importance Analysis

Extract and aggregate feature importance from 300 bootstrap XGBoost models:
- **Gain** — normalized gain importance (mean ± SD across 300 models)
- **Weight** — split count
- **SHAP** — mean |SHAP value| (50-model subsample × 3000-sample subsample)
- **Aggregated** — normalized mean of all methods


In [ ]:
"""
Ensemble Feature Importance Analysis (Bootstrap XGBoost)
=========================================================
Extract and aggregate feature importance from 300 bootstrap XGBoost models.

Methods:
  1. Gain-based importance (mean ± SD across 300 models)
  2. Split-count importance (mean ± SD)
  3. SHAP values (mean |SHAP| on a training subsample)
  4. Permutation importance (optional, slower)

Outputs:
  - importance_barplot.tiff           Top features bar chart with error bars
  - importance_heatmap.tiff           Per-model importance heatmap
  - importance_boxplot.tiff           Boxplot of gain across 300 models
  - importance_shap_summary.tiff      SHAP summary beeswarm plot
  - importance_shap_bar.tiff          SHAP bar plot
  - importance_dot_ranking.tiff       Dot-ranking plot (gain + SHAP side by side)
  - feature_importance_results.xlsx   Full results table

Requires: models, X_train, y_train, features (from earlier cells)
"""

from __future__ import annotations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import seaborn as sns
from tqdm import tqdm
import warnings, gc

warnings.filterwarnings("ignore")

# ─────────────────── Configuration ───────────────────
FI_DPI = 300
FI_TOP_N = 30            # Top N features for plots
SHAP_SUBSAMPLE = 3000    # Subsample size for SHAP (speed vs accuracy)
FI_RANDOM_STATE = 42

# Variable labels (short, for plots)
_VAR_LABELS = {
    "WifeAge": "Wife Age", "HusbandAge": "Husband Age", "MenarcheAge": "Menarche Age",
    "WifeHeight": "Wife Height", "WifeBMI": "Wife BMI",
    "WifeHR": "Wife HR", "WifeSBP": "Wife SBP", "WifeDBP": "Wife DBP",
    "HusbHeight": "Husb Height", "HusbBMI": "Husb BMI",
    "HusbHR": "Husb HR", "HusbSBP": "Husb SBP", "HusbDBP": "Husb DBP",
    "Hemoglobin": "Hemoglobin", "RBC": "RBC", "Platelet": "Platelet", "WBC": "WBC",
    "NeutrophilPct": "Neutrophil%", "LymphocytePct": "Lymphocyte%",
    "Glucose": "Glucose", "WifeALT": "Wife ALT", "WifeCreat": "Wife Creat",
    "TSH": "TSH", "HusbALT": "Husb ALT", "HusbCreat": "Husb Creat",
    "LeftTestVol": "L.TestVol", "RightTestVol": "R.TestVol",
    "WifeEdu": "Wife Edu", "HusbandEdu": "Husb Edu",
    "WifeOcc": "Wife Occ", "HusbandOcc": "Husb Occ",
    "PrevPregnancy": "Prev Preg", "Consanguinity": "Consanguinity",
    "GynDisease": "Gyn Disease", "GynUltrasound": "Gyn US",
    "WifeSmoke": "Wife Smoke", "HusbSmoke": "Husb Smoke",
    "WifeAlcohol": "Wife Alcohol", "HusbAlcohol": "Husb Alcohol",
    "WifePassSmoke": "Wife PassSmoke", "HusbPassSmoke": "Husb PassSmoke",
}

# Clinical groups (for color coding)
_VAR_GROUPS = {
    "Demographics": ["WifeAge","HusbandAge","WifeEdu","HusbandEdu","WifeOcc","HusbandOcc",
                      "WifeEthnic","HusbandEthnic","WifeResident","HusbandResident"],
    "Wife Physical": ["WifeHeight","WifeBMI","WifeHR","WifeSBP","WifeDBP",
                       "WifeMental","WifeIntel","WifeFace","WifePosture","WifeFaceSpec","WifeSkinHair",
                       "WifeThyroid","WifeLung","WifeRhythm","WifeMurmur","WifeLiverSpleen","WifeExtremity"],
    "Husband Physical": ["HusbHeight","HusbBMI","HusbHR","HusbSBP","HusbDBP",
                          "HusbMental","HusbIntel","HusbFace","HusbPosture","HusbFaceSpec","HusbSkinHair",
                          "HusbThyroid","HusbLung","HusbRhythm","HusbMurmur","HusbLiverSpleen","HusbExtremity"],
    "Reproduction": ["MenarcheAge","CycleRegular","MenstrualVol","Dysmenorrhea",
                      "PrevPregnancy","Contraception","WifeGynExam","GynUltrasound",
                      "WifePubHair","Breast","Vulva","GynDisease"],
    "Andrology": ["HusbPubHair","AdamsApple","Penis","Foreskin","Testis","Epididymis",
                   "VasDeferens","Varicocele","LeftTestVol","RightTestVol","HusbAndroExam","HusbAndrologyDis"],
    "Lab Tests": ["Hemoglobin","RBC","Platelet","WBC","NeutrophilPct","LymphocytePct",
                   "Glucose","WifeALT","WifeCreat","HusbALT","HusbCreat","TSH"],
    "Blood Type": ["WifeABO","HusbABO","WifeRh","HusbRh"],
    "Infection": ["RubellaIgG","CMVIgG","CMVIgM","ToxoIgG","ToxoIgM",
                   "SyphilisScr","HusbSyphilis","WifeHepB","HusbHepB","WifeUrine","HusbUrine"],
    "Medical Hx": ["WifeDiseaseHx","WifeBirthDefect","WifeMedication","WifeVaccine",
                    "WifeFamConsang","WifeFamDisease","Consanguinity",
                    "HusbDiseaseHis","HusbBirthDefect","HusbMedication","HusbVaccinated",
                    "HusbFamConsang","HusbFamDisease"],
    "Lifestyle": ["WifeMeatEgg","WifeVegAverse","WifeRawMeat","WifeSmoke","WifePassSmoke",
                   "WifeAlcohol","WifeDrugUse","WifeHalitosis","WifeGumBleed",
                   "HusbMeatEgg","HusbVegAverse","HusbRawMeat","HusbSmoke","HusbPassSmoke",
                   "HusbAlcohol","HusbDrugUse"],
    "Psychosocial": ["WifeEnvExpose","WifeStress","WifeRelatStress","WifeFinance","WifeReady",
                      "HusbEnvExpose","HusbStress","HusbRelatStress","HusbFinance","HusbReady"],
}

def _lbl(v): return _VAR_LABELS.get(v, v)
def _grp(v):
    for g, vs in _VAR_GROUPS.items():
        if v in vs: return g
    return "Other"

# Group color palette
_GROUP_PALETTE = {
    "Demographics": "#4e79a7", "Wife Physical": "#f28e2b", "Husband Physical": "#e15759",
    "Reproduction": "#76b7b2", "Andrology": "#59a14f", "Lab Tests": "#edc948",
    "Blood Type": "#b07aa1", "Infection": "#ff9da7", "Medical Hx": "#9c755f",
    "Lifestyle": "#bab0ac", "Psychosocial": "#86bcb6", "Other": "#cccccc",
}

print("Feature Importance Analysis module loaded.\n")


# ═══════════════════════════════════════════════════════
# STEP 1: Extract Gain & Weight from 300 bootstrap models
# ═══════════════════════════════════════════════════════

def extract_gain_importance(models, feature_names):
    """Extract gain and weight importance from all bootstrap models."""
    n = len(models)
    gain_matrix = np.zeros((n, len(feature_names)))
    weight_matrix = np.zeros((n, len(feature_names)))
    fname_to_idx = {f: i for i, f in enumerate(feature_names)}

    for i, model in enumerate(tqdm(models, desc="Extracting importance")):
        booster = model.get_booster()

        for imp_type, matrix in [("gain", gain_matrix), ("weight", weight_matrix)]:
            scores = booster.get_score(importance_type=imp_type)
            for key, val in scores.items():
                # Handle both "feature_name" and "fN" format
                if key in fname_to_idx:
                    matrix[i, fname_to_idx[key]] = val
                else:
                    try:
                        idx = int(key.replace("f", ""))
                        if idx < len(feature_names):
                            matrix[i, idx] = val
                    except ValueError:
                        pass

    # Normalize each model's gain to sum to 1
    row_sums = gain_matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    gain_norm = gain_matrix / row_sums

    return gain_matrix, gain_norm, weight_matrix


print("Extracting Gain & Weight importance from 300 models...")
gain_raw, gain_norm, weight_raw = extract_gain_importance(models, features)

# Summary DataFrame
df_gain = pd.DataFrame({
    "Feature": features,
    "Label": [_lbl(f) for f in features],
    "Group": [_grp(f) for f in features],
    "Gain_Mean": gain_norm.mean(axis=0),
    "Gain_SD": gain_norm.std(axis=0),
    "Gain_Median": np.median(gain_norm, axis=0),
    "Gain_Q25": np.percentile(gain_norm, 25, axis=0),
    "Gain_Q75": np.percentile(gain_norm, 75, axis=0),
    "Weight_Mean": weight_raw.mean(axis=0),
    "Weight_SD": weight_raw.std(axis=0),
    "N_Models_Used": (gain_raw > 0).sum(axis=0),  # how many models used this feature
}).sort_values("Gain_Mean", ascending=False).reset_index(drop=True)

print(f"\nTop 15 features by mean normalized gain:")
print(df_gain[["Feature", "Label", "Gain_Mean", "Gain_SD", "N_Models_Used"]].head(15).to_string(index=False))


# ═══════════════════════════════════════════════════════
# STEP 2: SHAP Values
# ═══════════════════════════════════════════════════════

print(f"\nComputing SHAP values (subsample={SHAP_SUBSAMPLE})...")
try:
    import shap

    # Subsample for speed
    rng = np.random.RandomState(FI_RANDOM_STATE)
    n_shap = min(SHAP_SUBSAMPLE, len(X_train))
    idx_shap = rng.choice(len(X_train), n_shap, replace=False)
    X_shap = X_train[idx_shap]
    X_shap_df = pd.DataFrame(X_shap, columns=features)

    # Average SHAP across a subset of models (speed)
    n_shap_models = min(50, len(models))
    shap_model_idx = rng.choice(len(models), n_shap_models, replace=False)

    shap_values_all = []
    for mi in tqdm(shap_model_idx, desc="SHAP across models"):
        explainer = shap.TreeExplainer(models[mi])
        sv = explainer.shap_values(X_shap)
        shap_values_all.append(sv)

    # Average SHAP values across models
    shap_values_mean = np.mean(shap_values_all, axis=0)  # (n_shap, n_features)
    shap_abs_mean = np.abs(shap_values_mean).mean(axis=0)  # (n_features,)

    # Also compute per-model mean|SHAP| for uncertainty
    shap_per_model = np.array([np.abs(sv).mean(axis=0) for sv in shap_values_all])  # (n_models, n_features)

    df_gain["SHAP_Mean"] = 0.0
    df_gain["SHAP_SD"] = 0.0
    for i, f in enumerate(features):
        mask = df_gain["Feature"] == f
        if mask.any():
            df_gain.loc[mask, "SHAP_Mean"] = shap_abs_mean[i]
            df_gain.loc[mask, "SHAP_SD"] = shap_per_model[:, i].std()

    shap_available = True
    print(f"  SHAP computed on {n_shap} samples x {n_shap_models} models")

except ImportError:
    print("  SHAP not installed, skipping. Install with: pip install shap")
    shap_available = False
except Exception as e:
    print(f"  SHAP failed: {e}")
    shap_available = False


# ═══════════════════════════════════════════════════════
# STEP 3: Aggregated Importance (normalized rank fusion)
# ═══════════════════════════════════════════════════════
shap_available = True
# Normalize each metric to [0, 1]
for col in ["Gain_Mean", "Weight_Mean"]:
    mx = df_gain[col].max()
    if mx > 0:
        df_gain[f"{col}_Norm"] = df_gain[col] / mx
    else:
        df_gain[f"{col}_Norm"] = 0.0

if shap_available:
    mx = df_gain["SHAP_Mean"].max()
    df_gain["SHAP_Mean_Norm"] = df_gain["SHAP_Mean"] / mx if mx > 0 else 0.0
    df_gain["Aggregated"] = (df_gain["Gain_Mean_Norm"] + df_gain["Weight_Mean_Norm"] + df_gain["SHAP_Mean_Norm"]) / 3
else:
    df_gain["Aggregated"] = (df_gain["Gain_Mean_Norm"] + df_gain["Weight_Mean_Norm"]) / 2

df_gain = df_gain.sort_values("Aggregated", ascending=False).reset_index(drop=True)
df_gain.insert(0, "Rank", range(1, len(df_gain) + 1))

print(f"\nFinal aggregated ranking (top 15):")
agg_cols = ["Rank", "Feature", "Label", "Aggregated", "Gain_Mean", "SHAP_Mean"] if shap_available else ["Rank", "Feature", "Label", "Aggregated", "Gain_Mean"]
print(df_gain[agg_cols].head(15).to_string(index=False))


### 13a. Bar Chart — Top Feature Importance with Error Bars

In [ ]:

# ═══════════════════════════════════════════════════════
# PLOT 1: Aggregated Importance Bar Chart (with error bars)
# ═══════════════════════════════════════════════════════

top = df_gain.head(FI_TOP_N).copy()
top = top.iloc[::-1]  # reverse for horizontal bar

fig, ax = plt.subplots(figsize=(10, max(6, FI_TOP_N * 0.38)))

labels = [_lbl(v) for v in top["Feature"]]
groups = [_grp(v) for v in top["Feature"]]
colors = [_GROUP_PALETTE.get(g, "#cccccc") for g in groups]

bars = ax.barh(range(len(top)), top["Gain_Mean"].values,
               xerr=top["Gain_SD"].values, capsize=3,
               color=colors, edgecolor="white", height=0.72,
               error_kw={"linewidth": 0.8, "alpha": 0.7})

ax.set_yticks(range(len(top)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel("Mean Normalized Gain (± SD across 300 models)", fontsize=11)
ax.set_title(f"Top {FI_TOP_N} Feature Importance — Bootstrap Ensemble (n=300 XGBoost)",
             fontsize=13, fontweight="bold", pad=12)

# Legend by group
legend_groups = list(dict.fromkeys(groups))
legend_elements = [Patch(facecolor=_GROUP_PALETTE.get(g, "#ccc"), label=g) for g in legend_groups]
ax.legend(handles=legend_elements, loc="lower right", fontsize=8, framealpha=0.9,
          title="Clinical Group", title_fontsize=9)

ax.grid(axis="x", alpha=0.25, linewidth=0.5)
ax.set_xlim(0, top["Gain_Mean"].max() * 1.25)
sns.despine(left=True, bottom=False)
plt.tight_layout()
plt.savefig("importance_barplot.tiff", dpi=FI_DPI, bbox_inches="tight")
plt.show()
print("Saved: importance_barplot.tiff")


### 13b. Boxplot — Gain Distribution Across 300 Models

In [ ]:

# ═══════════════════════════════════════════════════════
# PLOT 2: Boxplot — Gain distribution across 300 models
# ═══════════════════════════════════════════════════════

top_feats = df_gain.head(FI_TOP_N)["Feature"].tolist()
top_idx = [features.index(f) for f in top_feats]

box_data = gain_norm[:, top_idx]  # (300, top_n)

fig, ax = plt.subplots(figsize=(10, max(6, FI_TOP_N * 0.38)))

bp = ax.boxplot(box_data, vert=False, patch_artist=True,
                whis=[5, 95],
                boxprops=dict(linewidth=0.8),
                whiskerprops=dict(linewidth=0.6),
                medianprops=dict(color="black", linewidth=1.2),
                flierprops=dict(markersize=2, alpha=0.3))

labels_bp = [_lbl(f) for f in top_feats]
groups_bp = [_grp(f) for f in top_feats]
colors_bp = [_GROUP_PALETTE.get(g, "#ccc") for g in groups_bp]

for patch, color in zip(bp["boxes"], colors_bp):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_yticklabels(labels_bp, fontsize=9)
ax.set_xlabel("Normalized Gain Importance", fontsize=11)
ax.set_title(f"Feature Importance Distribution (300 Bootstrap Models)",
             fontsize=13, fontweight="bold", pad=12)
ax.grid(axis="x", alpha=0.25)
ax.invert_yaxis()
sns.despine(left=True)
plt.tight_layout()
plt.savefig("importance_boxplot.tiff", dpi=FI_DPI, bbox_inches="tight")
plt.show()
print("Saved: importance_boxplot.tiff")


### 13c. SHAP Summary — Beeswarm + Bar

In [ ]:

# ═══════════════════════════════════════════════════════
# PLOT 3: SHAP Summary Beeswarm + Bar (if available)
# ═══════════════════════════════════════════════════════

if shap_available:
    # --- Beeswarm ---
    top_feat_idx = [features.index(f) for f in df_gain.head(FI_TOP_N)["Feature"]]
    shap_top = shap_values_mean[:, top_feat_idx]
    X_shap_top = X_shap_df.iloc[:, top_feat_idx]

    fig, axes = plt.subplots(1, 2, figsize=(18, max(7, FI_TOP_N * 0.35)),
                              gridspec_kw={"width_ratios": [1.3, 1]})

    # Left: Beeswarm
    plt.sca(axes[0])
    shap.summary_plot(shap_top, X_shap_top, plot_type="dot",
                      max_display=FI_TOP_N, show=False, plot_size=None)
    axes[0].set_title("SHAP Beeswarm Plot", fontsize=13, fontweight="bold", pad=10)
    axes[0].set_xlabel("SHAP Value (impact on prediction)", fontsize=10)

    # Right: Bar
    plt.sca(axes[1])
    shap.summary_plot(shap_top, X_shap_top, plot_type="bar",
                      max_display=FI_TOP_N, show=False, plot_size=None)
    axes[1].set_title("Mean |SHAP Value|", fontsize=13, fontweight="bold", pad=10)
    axes[1].set_xlabel("Mean |SHAP Value|", fontsize=10)

    plt.tight_layout()
    plt.savefig("importance_shap_summary.tiff", dpi=FI_DPI, bbox_inches="tight")
    plt.show()
    print("Saved: importance_shap_summary.tiff")
else:
    print("SHAP not available — skipping beeswarm plot.")


### 13d. Dot-Ranking — Gain vs SHAP Comparison

In [ ]:

# ═══════════════════════════════════════════════════════
# PLOT 4: Dot-Ranking — Gain vs SHAP comparison
# ═══════════════════════════════════════════════════════

top = df_gain.head(FI_TOP_N).copy()
labels_dot = [_lbl(f) for f in top["Feature"]]
groups_dot = [_grp(f) for f in top["Feature"]]
colors_dot = [_GROUP_PALETTE.get(g, "#ccc") for g in groups_dot]

fig, axes = plt.subplots(1, 2 if shap_available else 1,
                          figsize=(14 if shap_available else 8, max(6, FI_TOP_N * 0.36)),
                          sharey=True)
if not isinstance(axes, np.ndarray):
    axes = [axes]

# Left: Gain
ax = axes[0]
ax.scatter(top["Gain_Mean"], range(len(top) - 1, -1, -1),
           c=colors_dot[::-1], s=80, edgecolors="white", linewidth=0.8, zorder=3)
ax.hlines(range(len(top) - 1, -1, -1), 0, top["Gain_Mean"].values[::-1],
          colors=[c for c in colors_dot[::-1]], alpha=0.4, linewidth=2)
ax.set_yticks(range(len(top) - 1, -1, -1))
ax.set_yticklabels(labels_dot, fontsize=9)
ax.set_xlabel("Mean Normalized Gain", fontsize=11)
ax.set_title("Gain Importance", fontsize=13, fontweight="bold")
ax.grid(axis="x", alpha=0.2)

# Right: SHAP
if shap_available:
    ax2 = axes[1]
    ax2.scatter(top["SHAP_Mean"], range(len(top) - 1, -1, -1),
                c=colors_dot[::-1], s=80, edgecolors="white", linewidth=0.8, zorder=3)
    ax2.hlines(range(len(top) - 1, -1, -1), 0, top["SHAP_Mean"].values[::-1],
               colors=[c for c in colors_dot[::-1]], alpha=0.4, linewidth=2)
    ax2.set_xlabel("Mean |SHAP Value|", fontsize=11)
    ax2.set_title("SHAP Importance", fontsize=13, fontweight="bold")
    ax2.grid(axis="x", alpha=0.2)

# Legend
legend_groups = list(dict.fromkeys(groups_dot))
legend_elements = [Patch(facecolor=_GROUP_PALETTE.get(g, "#ccc"), label=g) for g in legend_groups]
axes[-1].legend(handles=legend_elements, loc="lower right", fontsize=7.5,
                framealpha=0.9, title="Group", title_fontsize=8)

sns.despine(left=True)
plt.tight_layout()
plt.savefig("importance_dot_ranking.tiff", dpi=FI_DPI, bbox_inches="tight")
plt.show()
print("Saved: importance_dot_ranking.tiff")


### 13e. Per-Model Importance Heatmap

In [ ]:

# ═══════════════════════════════════════════════════════
# PLOT 5: Per-Model Importance Heatmap (top features)
# ═══════════════════════════════════════════════════════

top_feats = df_gain.head(min(FI_TOP_N, 25))["Feature"].tolist()
top_idx = [features.index(f) for f in top_feats]
hm_data = gain_norm[:, top_idx].T  # (n_feats, 300)

# Subsample models for readability
n_show_models = min(100, hm_data.shape[1])
model_idx = np.linspace(0, hm_data.shape[1] - 1, n_show_models, dtype=int)
hm_sub = hm_data[:, model_idx]

fig, ax = plt.subplots(figsize=(14, max(5, len(top_feats) * 0.35)))
sns.heatmap(hm_sub, ax=ax, cmap="YlOrRd", vmin=0,
            yticklabels=[_lbl(f) for f in top_feats],
            xticklabels=False,
            cbar_kws={"label": "Normalized Gain", "shrink": 0.6})
ax.set_xlabel(f"Bootstrap Model Index (showing {n_show_models} of {len(models)})", fontsize=10)
ax.set_title("Feature Importance Stability Across Bootstrap Models",
             fontsize=13, fontweight="bold", pad=12)
plt.yticks(fontsize=9, rotation=0)
plt.tight_layout()
plt.savefig("importance_heatmap.tiff", dpi=FI_DPI, bbox_inches="tight")
plt.show()
print("Saved: importance_heatmap.tiff")


### 13f. Export Feature Importance to Excel

In [ ]:

# ═══════════════════════════════════════════════════════
# Save Results to Excel
# ═══════════════════════════════════════════════════════

# Sheet 1: Full importance table
output_cols = ["Rank", "Feature", "Label", "Group",
               "Gain_Mean", "Gain_SD", "Gain_Median", "Gain_Q25", "Gain_Q75",
               "Weight_Mean", "Weight_SD", "N_Models_Used"]
if shap_available:
    output_cols += ["SHAP_Mean", "SHAP_SD"]
output_cols += ["Aggregated"]

with pd.ExcelWriter("feature_importance_results.xlsx", engine="openpyxl") as writer:
    # Sheet 1: Full table
    df_gain[output_cols].to_excel(writer, sheet_name="Feature Importance", index=False)

    # Sheet 2: Group summary
    grp_summary = df_gain.groupby("Group").agg(
        N_Features=("Feature", "count"),
        Sum_Gain=("Gain_Mean", "sum"),
        Mean_Gain=("Gain_Mean", "mean"),
        Top_Feature=("Gain_Mean", lambda x: df_gain.loc[x.idxmax(), "Feature"]),
        Top_Gain=("Gain_Mean", "max"),
    ).sort_values("Sum_Gain", ascending=False)
    if shap_available:
        grp_shap = df_gain.groupby("Group")["SHAP_Mean"].agg(["sum", "mean"])
        grp_shap.columns = ["Sum_SHAP", "Mean_SHAP"]
        grp_summary = grp_summary.join(grp_shap)
    grp_summary.to_excel(writer, sheet_name="Group Summary")

    # Sheet 3: Raw gain matrix (300 models x features) for top features
    top30 = df_gain.head(30)["Feature"].tolist()
    top30_idx = [features.index(f) for f in top30]
    raw_df = pd.DataFrame(gain_norm[:, top30_idx], columns=top30)
    raw_df.index.name = "Model"
    raw_df.to_excel(writer, sheet_name="Gain Matrix (Top 30)")

    # Sheet 4: Model consistency (CV of importance)
    consistency = pd.DataFrame({
        "Feature": features,
        "Label": [_lbl(f) for f in features],
        "Gain_Mean": gain_norm.mean(axis=0),
        "Gain_SD": gain_norm.std(axis=0),
        "CV": gain_norm.std(axis=0) / (gain_norm.mean(axis=0) + 1e-10),
        "N_Models_Used": (gain_raw > 0).sum(axis=0),
        "Usage_Rate": (gain_raw > 0).sum(axis=0) / len(models) * 100,
    }).sort_values("Gain_Mean", ascending=False)
    consistency.to_excel(writer, sheet_name="Model Consistency", index=False)

print("\nSaved: feature_importance_results.xlsx")
print(f"  Sheets: Feature Importance / Group Summary / Gain Matrix (Top 30) / Model Consistency")

# ── Summary ──
print(f"\n{'='*70}")
print("FEATURE IMPORTANCE ANALYSIS COMPLETE")
print(f"{'='*70}")
print(f"  Models: {len(models)} bootstrap XGBoost")
print(f"  Features: {len(features)}")
print(f"  Methods: Gain, Weight" + (", SHAP" if shap_available else ""))
print(f"\n  Top 5:")
for _, row in df_gain.head(5).iterrows():
    print(f"    {row['Rank']:2.0f}. {row['Label']:20s} Gain={row['Gain_Mean']:.4f}  "
          + (f"SHAP={row['SHAP_Mean']:.4f}" if shap_available else ""))
print(f"\n  Output files:")
print(f"    importance_barplot.tiff")
print(f"    importance_boxplot.tiff")
print(f"    importance_heatmap.tiff")
if shap_available:
    print(f"    importance_shap_summary.tiff")
print(f"    importance_dot_ranking.tiff")
print(f"    feature_importance_results.xlsx")


In [ ]:
"""
Optimized Feature Importance Boxplot + SHAP Side-by-Side
========================================================
Fixes:
  1. Sorts features consistently by chosen metric (gain/SHAP/aggregated)
  2. Adds SHAP bar plot alongside boxplot for direct comparison
  3. Error bars on SHAP show ±1 SD across 50 bootstrap models
  4. Color-coded by clinical group with shared legend
  5. Publication-ready TIFF output at 300 DPI
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.patches import Patch
import seaborn as sns

# ═══════════════════════════════════════════════════════
# CONFIG — adjust these to match your pipeline
# ═══════════════════════════════════════════════════════
FI_DPI = 300
FI_TOP_N = 20          # Number of features to display
SORT_BY = "Aggregated" # Options: "Gain_Mean", "SHAP_Mean", "Aggregated", "Weight_Mean"

# ═══════════════════════════════════════════════════════
# PLOT: Side-by-side Boxplot (Gain) + SHAP Bar
# ═══════════════════════════════════════════════════════

# --- Sort by chosen metric ---
df_plot = df_gain.sort_values(SORT_BY, ascending=False).head(FI_TOP_N).copy()
# Reverse for bottom-to-top plotting (highest at top)
df_plot = df_plot.iloc[::-1].reset_index(drop=True)

top_feats = df_plot["Feature"].tolist()
top_idx = [features.index(f) for f in top_feats]

# Extract boxplot data in the SAME sorted order
box_data = gain_norm[:, top_idx]  # (300, FI_TOP_N)

labels_plot = df_plot["Label"].tolist()
groups_plot = df_plot["Group"].tolist()
colors_plot = [_GROUP_PALETTE.get(g, "#ccc") for g in groups_plot]

# --- Create figure with two subplots ---
fig, (ax_box, ax_shap) = plt.subplots(
    1, 2,
    figsize=(16, max(7, FI_TOP_N * 0.40)),
    gridspec_kw={"width_ratios": [1.1, 1], "wspace": 0.05},
    sharey=True,
)

# ── Left panel: Gain boxplot ──
bp = ax_box.boxplot(
    box_data,
    vert=False,
    patch_artist=True,
    whis=[5, 95],
    boxprops=dict(linewidth=0.8),
    whiskerprops=dict(linewidth=0.6, color="#888"),
    capprops=dict(linewidth=0.6, color="#888"),
    medianprops=dict(color="black", linewidth=1.2),
    flierprops=dict(markersize=2, alpha=0.3),
)

for patch, color in zip(bp["boxes"], colors_plot):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax_box.set_yticklabels(labels_plot, fontsize=9)
ax_box.set_xlabel("Normalized Gain Importance", fontsize=11)
ax_box.set_title("Gain Distribution (300 Bootstrap Models)", fontsize=12, fontweight="bold", pad=10)
ax_box.grid(axis="x", alpha=0.2, linewidth=0.5)
ax_box.invert_yaxis()
ax_box.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))

# ── Right panel: SHAP bar with error bars ──
if shap_available:
    shap_means = df_plot["SHAP_Mean"].values
    shap_sds = df_plot["SHAP_SD"].values
    y_pos = np.arange(len(df_plot))

    bars = ax_shap.barh(
        y_pos,
        shap_means,
        xerr=shap_sds,
        height=0.65,
        color=colors_plot,
        alpha=0.75,
        edgecolor=[c for c in colors_plot],
        linewidth=0.8,
        error_kw=dict(elinewidth=0.8, capsize=2, capthick=0.6, color="#555"),
    )

    ax_shap.set_xlabel("Mean |SHAP Value|", fontsize=11)
    ax_shap.set_title(
        f"SHAP Importance ({min(50, len(models))} Models × {min(SHAP_SUBSAMPLE, len(X_train))} Samples)",
        fontsize=12, fontweight="bold", pad=10,
    )
    ax_shap.grid(axis="x", alpha=0.2, linewidth=0.5)
    ax_shap.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    ax_shap.tick_params(axis="y", left=False)
else:
    # Fallback: show weight (split count) if SHAP unavailable
    weight_means = df_plot["Weight_Mean"].values
    weight_sds = df_plot["Weight_SD"].values
    y_pos = np.arange(len(df_plot))

    ax_shap.barh(
        y_pos,
        weight_means,
        xerr=weight_sds,
        height=0.65,
        color=colors_plot,
        alpha=0.75,
        edgecolor=colors_plot,
        linewidth=0.8,
        error_kw=dict(elinewidth=0.8, capsize=2, capthick=0.6, color="#555"),
    )
    ax_shap.set_xlabel("Mean Split Count", fontsize=11)
    ax_shap.set_title("Split Count (300 Bootstrap Models)", fontsize=12, fontweight="bold", pad=10)
    ax_shap.grid(axis="x", alpha=0.2, linewidth=0.5)
    ax_shap.tick_params(axis="y", left=False)

# ── Shared legend (clinical groups) ──
unique_groups = list(dict.fromkeys(groups_plot))  # preserve order
legend_patches = [
    Patch(facecolor=_GROUP_PALETTE.get(g, "#ccc"), alpha=0.75, edgecolor="#666", linewidth=0.5, label=g)
    for g in unique_groups
]
fig.legend(
    handles=legend_patches,
    loc="lower center",
    ncol=min(6, len(unique_groups)),
    fontsize=9,
    frameon=False,
    bbox_to_anchor=(0.5, -0.02),
)

# ── Sort indicator annotation ──
fig.text(
    0.5, 0.98,
    f"Sorted by: {SORT_BY.replace('_', ' ')}  |  Top {FI_TOP_N} features",
    ha="center", va="top", fontsize=10, style="italic", color="#666",
)

sns.despine(left=True)
plt.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.savefig("importance_boxplot_shap.tiff", dpi=FI_DPI, bbox_inches="tight")
plt.show()
print("Saved: importance_boxplot_shap.tiff")


# ═══════════════════════════════════════════════════════
# BONUS: Dot-ranking plot (Gain rank vs SHAP rank)
# ═══════════════════════════════════════════════════════

if shap_available:
    df_rank = df_gain.head(FI_TOP_N).copy()
    df_rank["Gain_Rank"] = df_rank["Gain_Mean"].rank(ascending=False).astype(int)
    df_rank["SHAP_Rank"] = df_rank["SHAP_Mean"].rank(ascending=False).astype(int)

    fig2, ax2 = plt.subplots(figsize=(8, max(6, FI_TOP_N * 0.35)))

    for _, row in df_rank.iterrows():
        color = _GROUP_PALETTE.get(row["Group"], "#ccc")
        ax2.plot(
            [0, 1],
            [row["Gain_Rank"], row["SHAP_Rank"]],
            color=color, alpha=0.6, linewidth=1.5,
        )
        ax2.scatter(0, row["Gain_Rank"], color=color, s=50, zorder=5, edgecolors="white", linewidth=0.5)
        ax2.scatter(1, row["SHAP_Rank"], color=color, s=50, zorder=5, edgecolors="white", linewidth=0.5)
        ax2.text(-0.05, row["Gain_Rank"], row["Label"], ha="right", va="center", fontsize=8, color="#444")
        ax2.text(1.05, row["SHAP_Rank"], row["Label"], ha="left", va="center", fontsize=8, color="#444")

    ax2.set_xlim(-0.4, 1.4)
    ax2.set_ylim(FI_TOP_N + 0.5, 0.5)
    ax2.set_xticks([0, 1])
    ax2.set_xticklabels(["Gain Rank", "SHAP Rank"], fontsize=11, fontweight="bold")
    ax2.set_ylabel("Rank", fontsize=11)
    ax2.set_title("Feature Ranking Comparison: Gain vs SHAP", fontsize=13, fontweight="bold", pad=12)
    ax2.grid(axis="y", alpha=0.15)
    sns.despine()
    plt.tight_layout()
    plt.savefig("importance_dot_ranking.tiff", dpi=FI_DPI, bbox_inches="tight")
    plt.show()
    print("Saved: importance_dot_ranking.tiff")

In [41]:
"""
Priority Revision Analyses for SA Prediction Manuscript
========================================================
Addresses 5 critical reviewer concerns for IF≥8 journal submission:

  Priority 1: Prevalence shift investigation (3.26% → 14.38%)
  Priority 2: Logistic recalibration + Venn-ABERS
  Priority 3: Full calibration metrics (CITL, slope, ICI, E/O, calibration plot)
  Priority 5: AUPRC with bootstrap 95% CI
  Priority 6: Decision curve analysis with bootstrap 95% CI

HOW TO USE:
-----------
This script assumes your notebook has already loaded:
  - X_train, X_val, X_test  (pd.DataFrame or np.ndarray with feature columns)
  - y_train, y_val, y_test  (1-d arrays, int 0/1)
  - predictions_train, predictions_val, predictions_test
      (each shape [300, n_samples] — per-model predicted probabilities)
  - features  (list of feature column names)
  - dfs       (dict with keys "train", "val", "test" — raw DataFrames)

Copy the functions you need into your notebook, or run:
  %run priority_analyses.py
then call the top-level runners at the bottom.

Output:
  - Console tables (copy into manuscript supplementary)
  - Publication-quality TIFF figures at 300 DPI
  - CSV exports of key results
"""

from __future__ import annotations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_recall_curve, roc_curve, f1_score,
    accuracy_score, recall_score, precision_score,
    log_loss,
)
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
import warnings
warnings.filterwarnings("ignore")

DPI = 300
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


# ═══════════════════════════════════════════════════════════════════════════
# PRIORITY 1: Prevalence Shift Investigation
# ═══════════════════════════════════════════════════════════════════════════

def investigate_prevalence_shift(dfs: dict, features: list, outcome: str = "SA_outcome"):
    """
    Generate a comprehensive comparison table between training/val/test cohorts.

    Outputs:
      1. Standardized Mean Differences (SMD) for all features
      2. Prevalence comparison with 95% CI
      3. Key demographic distributions
      4. CSV export for manuscript Table S_shift

    Parameters
    ----------
    dfs : dict with keys "train", "val", "test", each a pd.DataFrame
    features : list of feature column names
    outcome : str, name of outcome column
    """
    print("\n" + "=" * 80)
    print("PRIORITY 1: Prevalence Shift Investigation")
    print("=" * 80)

    splits = {"Training": dfs["train"], "Internal Val": dfs["val"], "Temporal Val": dfs["test"]}

    # ── 1a. Prevalence with 95% Wilson CI ──
    print("\n── 1a. Outcome Prevalence ──")
    print(f"{'Split':<18} {'N':>10} {'Events':>8} {'Prevalence':>12} {'95% CI':>18}")
    print("-" * 70)
    for name, df in splits.items():
        y = df[outcome].values.astype(int)
        n = len(y)
        p = y.mean()
        # Wilson score interval
        z = 1.96
        denom = 1 + z**2 / n
        centre = (p + z**2 / (2 * n)) / denom
        margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
        lo, hi = max(0, centre - margin), min(1, centre + margin)
        print(f"{name:<18} {n:>10,} {y.sum():>8,} {p*100:>11.2f}% [{lo*100:.2f}%, {hi*100:.2f}%]")

    # ── 1b. Standardized Mean Differences ──
    print("\n── 1b. Standardized Mean Differences (Training vs Temporal Val) ──")

    results = []
    df_train = dfs["train"]
    df_test = dfs["test"]

    for feat in features:
        try:
            x1 = pd.to_numeric(df_train[feat], errors="coerce").dropna()
            x2 = pd.to_numeric(df_test[feat], errors="coerce").dropna()
            if len(x1) < 10 or len(x2) < 10:
                continue

            # SMD = (mean1 - mean2) / pooled_sd
            mean1, mean2 = x1.mean(), x2.mean()
            sd1, sd2 = x1.std(), x2.std()
            pooled_sd = np.sqrt((sd1**2 + sd2**2) / 2)
            smd = (mean1 - mean2) / pooled_sd if pooled_sd > 0 else 0

            results.append({
                "Feature": feat,
                "Train Mean": mean1,
                "Train SD": sd1,
                "TempVal Mean": mean2,
                "TempVal SD": sd2,
                "SMD": smd,
                "Abs SMD": abs(smd),
            })
        except Exception:
            continue

    df_smd = pd.DataFrame(results).sort_values("Abs SMD", ascending=False)

    # Flag features with |SMD| > 0.1 (standard imbalance threshold)
    imbalanced = df_smd[df_smd["Abs SMD"] > 0.1]
    print(f"\nFeatures with |SMD| > 0.1 (imbalanced): {len(imbalanced)} / {len(df_smd)}")
    print(f"\n{'Feature':<30} {'Train Mean':>12} {'TempVal Mean':>14} {'SMD':>8}")
    print("-" * 68)
    for _, row in imbalanced.head(20).iterrows():
        flag = " ***" if abs(row["SMD"]) > 0.25 else " **" if abs(row["SMD"]) > 0.2 else ""
        print(f"{row['Feature']:<30} {row['Train Mean']:>12.3f} {row['TempVal Mean']:>14.3f} {row['SMD']:>8.3f}{flag}")

    # ── 1c. Save to CSV ──
    df_smd.to_csv("prevalence_shift_SMD.csv", index=False)
    print(f"\n→ Saved: prevalence_shift_SMD.csv")

    # ── 1d. Love plot (SMD visualization) ──
    fig, ax = plt.subplots(figsize=(8, max(6, len(df_smd) * 0.25)))
    top_n = min(36, len(df_smd))
    plot_df = df_smd.head(top_n).sort_values("Abs SMD", ascending=True)

    colors = ["#E53935" if abs(s) > 0.25 else "#FF9800" if abs(s) > 0.1 else "#4CAF50"
              for s in plot_df["SMD"]]

    ax.barh(range(top_n), plot_df["Abs SMD"].values, color=colors, height=0.7, edgecolor="white")
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(plot_df["Feature"].values, fontsize=8)
    ax.axvline(0.1, color="gray", linestyle="--", linewidth=1, label="|SMD| = 0.1")
    ax.axvline(0.25, color="red", linestyle="--", linewidth=1, label="|SMD| = 0.25")
    ax.set_xlabel("Absolute Standardized Mean Difference")
    ax.set_title("Covariate Balance: Training vs Temporal Validation Cohort", fontweight="bold")
    ax.legend(loc="lower right", fontsize=8)
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    fig.savefig("Fig_S_prevalence_shift_love_plot.tiff", dpi=DPI, bbox_inches="tight")
    plt.close()
    print("→ Saved: Fig_S_prevalence_shift_love_plot.tiff")

    return df_smd


# ═══════════════════════════════════════════════════════════════════════════
# PRIORITY 3: Full Calibration Metrics
# ═══════════════════════════════════════════════════════════════════════════

def calibration_in_the_large(y_true, y_prob):
    """CITL: logit(mean_pred) - logit(prevalence)."""
    prev = y_true.mean()
    mean_pred = y_prob.mean()
    eps = 1e-10
    citl = np.log(mean_pred / (1 - mean_pred + eps) + eps) - np.log(prev / (1 - prev + eps) + eps)
    return citl


def calibration_slope_intercept(y_true, y_prob):
    """Fit logistic regression of y on logit(p) to get slope and intercept."""
    eps = 1e-10
    lp = np.log(y_prob / (1 - y_prob + eps) + eps)
    lp = np.clip(lp, -15, 15)
    from sklearn.linear_model import LogisticRegression
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000)
    lr.fit(lp.reshape(-1, 1), y_true)
    slope = lr.coef_[0][0]
    intercept = lr.intercept_[0]
    return slope, intercept


def eo_ratio(y_true, y_prob):
    """Expected-to-Observed ratio."""
    expected = y_prob.sum()
    observed = y_true.sum()
    return expected / max(observed, 1)


def integrated_calibration_index(y_true, y_prob, n_bins=50):
    """
    ICI: mean absolute difference between loess-smoothed calibration curve
    and the diagonal. Approximated with binned approach.
    """
    order = np.argsort(y_prob)
    y_sorted = y_true[order]
    p_sorted = y_prob[order]

    bins = np.array_split(np.arange(len(y_sorted)), n_bins)
    diffs = []
    for b in bins:
        if len(b) == 0:
            continue
        obs = y_sorted[b].mean()
        pred = p_sorted[b].mean()
        diffs.append(abs(obs - pred) * len(b))
    return sum(diffs) / len(y_true)


def compute_full_calibration(y_true, y_prob, n_bootstrap=200, label=""):
    """
    Compute all calibration metrics with bootstrap 95% CI.

    Returns dict with:
      CITL, Calibration Slope, Calibration Intercept, Brier Score,
      E/O Ratio, ICI — each with (point, lower, upper).
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    def _metrics(yt, yp):
        slope, intercept = calibration_slope_intercept(yt, yp)
        return {
            "CITL": calibration_in_the_large(yt, yp),
            "Calibration Slope": slope,
            "Calibration Intercept": intercept,
            "Brier Score": brier_score_loss(yt, yp),
            "E/O Ratio": eo_ratio(yt, yp),
            "ICI": integrated_calibration_index(yt, yp),
        }

    # Point estimates
    point = _metrics(y_true, y_prob)

    # Bootstrap CI
    boot_results = {k: [] for k in point}
    rng = np.random.RandomState(RANDOM_STATE)
    for _ in range(n_bootstrap):
        idx = rng.choice(len(y_true), len(y_true), replace=True)
        try:
            m = _metrics(y_true[idx], y_prob[idx])
            for k in boot_results:
                boot_results[k].append(m[k])
        except Exception:
            continue

    results = {}
    for k in point:
        arr = np.array(boot_results[k])
        lo, hi = np.percentile(arr, [2.5, 97.5])
        results[k] = {"point": point[k], "lower": lo, "upper": hi}

    # Print table
    if label:
        print(f"\n── Calibration Metrics: {label} ──")
    print(f"{'Metric':<28} {'Estimate':>10} {'95% CI':>22}")
    print("-" * 62)
    for k, v in results.items():
        print(f"{k:<28} {v['point']:>10.4f}  [{v['lower']:.4f}, {v['upper']:.4f}]")

    return results


def plot_calibration_comprehensive(datasets: dict, output_path="Fig_calibration_comprehensive.tiff"):
    """
    Generate publication-quality calibration plots for all datasets.

    Parameters
    ----------
    datasets : dict like {"Training": (y, p), "Internal Val": (y, p), "Temporal Val": (y, p)}
    """
    from sklearn.calibration import calibration_curve

    n = len(datasets)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]

    colors = ["#2196F3", "#4CAF50", "#F44336"]

    for i, (name, (y, p)) in enumerate(datasets.items()):
        ax = axes[i]
        # Calibration curve
        prob_true, prob_pred = calibration_curve(y, p, n_bins=10, strategy="quantile")
        ax.plot(prob_pred, prob_true, "o-", color=colors[i], lw=2, markersize=6, label="Model")
        ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Perfect")

        # Histogram of predictions
        ax2 = ax.twinx()
        ax2.hist(p, bins=50, alpha=0.15, color=colors[i])
        ax2.set_ylabel("Frequency", fontsize=8, alpha=0.5)
        ax2.tick_params(axis="y", labelsize=7, colors="gray")

        # Metrics annotation
        brier = brier_score_loss(y, p)
        slope, intercept = calibration_slope_intercept(y, p)
        ici = integrated_calibration_index(y, p)
        eo = eo_ratio(y, p)

        textstr = (f"Brier = {brier:.4f}\n"
                   f"Slope = {slope:.3f}\n"
                   f"Intercept = {intercept:.3f}\n"
                   f"ICI = {ici:.4f}\n"
                   f"E/O = {eo:.3f}")
        ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=8,
                verticalalignment="top", bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

        ax.set_xlabel("Predicted Probability", fontsize=10)
        ax.set_ylabel("Observed Proportion", fontsize=10)
        ax.set_title(name, fontsize=12, fontweight="bold")
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)
        ax.set_aspect("equal")
        ax.legend(loc="lower right", fontsize=8)
        ax.grid(alpha=0.2)

    plt.tight_layout()
    fig.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.close()
    print(f"→ Saved: {output_path}")


# ═══════════════════════════════════════════════════════════════════════════
# PRIORITY 2: Logistic Recalibration
# ═══════════════════════════════════════════════════════════════════════════

def logistic_recalibration(y_cal, p_cal, y_test, p_test, label=""):
    """
    Apply logistic recalibration: fit logistic(y ~ logit(p)) on calibration
    data, transform test predictions.

    Parameters
    ----------
    y_cal, p_cal : calibration set (use internal validation)
    y_test, p_test : test set to recalibrate

    Returns
    -------
    p_recal : recalibrated probabilities for test set
    """
    eps = 1e-10
    lp_cal = np.log(np.clip(p_cal, eps, 1 - eps) / (1 - np.clip(p_cal, eps, 1 - eps)))
    lp_test = np.log(np.clip(p_test, eps, 1 - eps) / (1 - np.clip(p_test, eps, 1 - eps)))

    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000)
    lr.fit(lp_cal.reshape(-1, 1), y_cal)

    p_recal = lr.predict_proba(lp_test.reshape(-1, 1))[:, 1]

    if label:
        print(f"\n── Logistic Recalibration: {label} ──")
        print(f"  Recalibration intercept: {lr.intercept_[0]:.4f}")
        print(f"  Recalibration slope:     {lr.coef_[0][0]:.4f}")

    return p_recal


def isotonic_recalibration(y_cal, p_cal, p_test):
    """Apply isotonic regression recalibration."""
    ir = IsotonicRegression(out_of_bounds="clip")
    ir.fit(p_cal, y_cal)
    return ir.predict(p_test)


def run_recalibration_comparison(y_val, p_val, y_test, p_test):
    """
    Compare raw vs logistic vs isotonic recalibration on temporal validation.

    Uses internal validation as calibration set.
    """
    print("\n" + "=" * 80)
    print("PRIORITY 2: Recalibration Comparison")
    print("=" * 80)

    # Raw
    print("\n── Raw (uncalibrated) ──")
    raw_metrics = compute_full_calibration(y_test, p_test, label="Temporal Val — Raw")

    # Logistic recalibration
    p_logistic = logistic_recalibration(y_val, p_val, y_test, p_test,
                                         label="Val→Test logistic recal")
    print("\n── After logistic recalibration ──")
    log_metrics = compute_full_calibration(y_test, p_logistic,
                                            label="Temporal Val — Logistic Recal")

    # Isotonic recalibration
    p_isotonic = isotonic_recalibration(y_val, p_val, p_test)
    print("\n── After isotonic recalibration ──")
    iso_metrics = compute_full_calibration(y_test, p_isotonic,
                                            label="Temporal Val — Isotonic Recal")

    # Check discrimination is preserved
    print("\n── Discrimination check (should be unchanged) ──")
    for name, p in [("Raw", p_test), ("Logistic", p_logistic), ("Isotonic", p_isotonic)]:
        auroc = roc_auc_score(y_test, p)
        auprc = average_precision_score(y_test, p)
        print(f"  {name:<12} AUROC={auroc:.4f}  AUPRC={auprc:.4f}")

    # Plot comparison
    from sklearn.calibration import calibration_curve
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    for ax, (name, p, color) in zip(axes, [
        ("Raw", p_test, "#F44336"),
        ("Logistic Recal", p_logistic, "#4CAF50"),
        ("Isotonic Recal", p_isotonic, "#2196F3"),
    ]):
        prob_true, prob_pred = calibration_curve(y_test, p, n_bins=10, strategy="quantile")
        ax.plot(prob_pred, prob_true, "o-", color=color, lw=2, markersize=6)
        ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)

        brier = brier_score_loss(y_test, p)
        ici = integrated_calibration_index(y_test, p)
        ax.set_title(f"{name}\nBrier={brier:.4f}, ICI={ici:.4f}", fontsize=10, fontweight="bold")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Observed")
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)
        ax.set_aspect("equal")
        ax.grid(alpha=0.2)

    plt.tight_layout()
    fig.savefig("Fig_recalibration_comparison.tiff", dpi=DPI, bbox_inches="tight")
    plt.close()
    print("\n→ Saved: Fig_recalibration_comparison.tiff")

    return p_logistic, p_isotonic, raw_metrics, log_metrics, iso_metrics


# ═══════════════════════════════════════════════════════════════════════════
# PRIORITY 5: AUPRC with Bootstrap 95% CI
# ═══════════════════════════════════════════════════════════════════════════

def compute_auprc_with_ci(y_true, y_prob, n_bootstrap=200, label=""):
    """
    Compute AUPRC with bootstrap percentile 95% CI.

    For low-prevalence settings, also reports:
      - Baseline AUPRC (= prevalence)
      - Lift over baseline
      - F1 at optimal PR threshold
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    # Point estimate
    auprc = average_precision_score(y_true, y_prob)
    prev = y_true.mean()
    auroc = roc_auc_score(y_true, y_prob)

    # Optimal F1 from PR curve
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-10)
    best_idx = np.argmax(f1_scores)
    best_f1 = f1_scores[best_idx]
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

    # Bootstrap
    boot_auprc = []
    boot_auroc = []
    rng = np.random.RandomState(RANDOM_STATE)
    for _ in range(n_bootstrap):
        idx = rng.choice(len(y_true), len(y_true), replace=True)
        if y_true[idx].sum() < 2 or y_true[idx].sum() == len(idx):
            continue
        boot_auprc.append(average_precision_score(y_true[idx], y_prob[idx]))
        boot_auroc.append(roc_auc_score(y_true[idx], y_prob[idx]))

    auprc_lo, auprc_hi = np.percentile(boot_auprc, [2.5, 97.5])
    auroc_lo, auroc_hi = np.percentile(boot_auroc, [2.5, 97.5])

    if label:
        print(f"\n── AUPRC Analysis: {label} ──")
    print(f"  Prevalence (baseline AUPRC): {prev:.4f} ({prev*100:.2f}%)")
    print(f"  AUROC:  {auroc:.4f}  [{auroc_lo:.4f}, {auroc_hi:.4f}]")
    print(f"  AUPRC:  {auprc:.4f}  [{auprc_lo:.4f}, {auprc_hi:.4f}]")
    print(f"  Lift over baseline:          {auprc / prev:.2f}x")
    print(f"  Best F1 = {best_f1:.4f} at threshold = {best_threshold:.4f}")

    return {
        "AUPRC": auprc, "AUPRC_lo": auprc_lo, "AUPRC_hi": auprc_hi,
        "AUROC": auroc, "AUROC_lo": auroc_lo, "AUROC_hi": auroc_hi,
        "prevalence": prev, "lift": auprc / prev,
        "best_f1": best_f1, "best_threshold": best_threshold,
    }


def plot_pr_curves(datasets: dict, output_path="Fig_PR_curves.tiff"):
    """
    Plot precision-recall curves for all datasets with baseline annotation.

    Parameters
    ----------
    datasets : dict like {"Training": (y, p), "Internal Val": (y, p), ...}
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ["#2196F3", "#4CAF50", "#F44336"]

    for i, (name, (y, p)) in enumerate(datasets.items()):
        precision, recall, _ = precision_recall_curve(y, p)
        auprc = average_precision_score(y, p)
        prev = y.mean()
        ax.plot(recall, precision, color=colors[i], lw=2,
                label=f"{name} (AUPRC={auprc:.3f}, prev={prev*100:.1f}%)")
        # Baseline
        ax.axhline(prev, color=colors[i], linestyle=":", alpha=0.4)

    ax.set_xlabel("Recall (Sensitivity)", fontsize=12)
    ax.set_ylabel("Precision (PPV)", fontsize=12)
    ax.set_title("Precision-Recall Curves", fontsize=14, fontweight="bold")
    ax.legend(loc="upper right", fontsize=9)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    fig.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.close()
    print(f"→ Saved: {output_path}")


# ═══════════════════════════════════════════════════════════════════════════
# PRIORITY 6: Decision Curve Analysis with Bootstrap 95% CI
# ═══════════════════════════════════════════════════════════════════════════

def compute_dca_with_ci(y_true, y_prob, thresholds=None, n_bootstrap=200):
    """
    DCA with bootstrap 95% CI for net benefit.

    Parameters
    ----------
    y_true : array-like, true labels
    y_prob : array-like, predicted probabilities
    thresholds : array-like, threshold probabilities (default: 0.01 to 0.50)
    n_bootstrap : int

    Returns
    -------
    dict with thresholds, nb_model (point, lo, hi), nb_all, nb_none
    """
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.50, 100)

    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    n = len(y_true)

    def _nb(yt, yp, t):
        pred = (yp >= t).astype(int)
        tp = ((pred == 1) & (yt == 1)).sum()
        fp = ((pred == 1) & (yt == 0)).sum()
        return tp / n - fp / n * (t / (1 - t))

    def _nb_all(yt, t):
        prev = yt.mean()
        return prev - (1 - prev) * t / (1 - t)

    # Point estimates
    nb_model = np.array([_nb(y_true, y_prob, t) for t in thresholds])
    nb_all = np.array([_nb_all(y_true, t) for t in thresholds])
    nb_none = np.zeros_like(thresholds)

    # Bootstrap CI for model net benefit
    boot_nb = np.zeros((n_bootstrap, len(thresholds)))
    rng = np.random.RandomState(RANDOM_STATE)
    for b in range(n_bootstrap):
        idx = rng.choice(n, n, replace=True)
        for j, t in enumerate(thresholds):
            boot_nb[b, j] = _nb(y_true[idx], y_prob[idx], t)

    nb_lo = np.percentile(boot_nb, 2.5, axis=0)
    nb_hi = np.percentile(boot_nb, 97.5, axis=0)

    return {
        "thresholds": thresholds,
        "nb_model": nb_model, "nb_model_lo": nb_lo, "nb_model_hi": nb_hi,
        "nb_all": nb_all, "nb_none": nb_none,
    }


def compute_dca_uq_strategies(y_true, y_prob, uncertainty, thresholds=None):
    """
    Compare 3 DCA strategies:
      1. Standard model (risk only)
      2. Model + UQ selective prediction (abstain when uncertain)
      3. Model + UQ joint stratification

    Parameters
    ----------
    y_true, y_prob : arrays
    uncertainty : array-like, total uncertainty per sample
    thresholds : threshold range
    """
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.50, 100)

    n = len(y_true)
    u_median = np.median(uncertainty)

    # Strategy 1: Standard model
    dca_standard = compute_dca_with_ci(y_true, y_prob, thresholds, n_bootstrap=200)

    # Strategy 2: Selective prediction — only predict for certain samples
    certain_mask = uncertainty <= u_median
    if certain_mask.sum() > 10:
        dca_selective = compute_dca_with_ci(
            y_true[certain_mask], y_prob[certain_mask], thresholds, n_bootstrap=200
        )
        # Scale net benefit by coverage
        coverage = certain_mask.mean()
        dca_selective["nb_model"] *= coverage
        dca_selective["nb_model_lo"] *= coverage
        dca_selective["nb_model_hi"] *= coverage
    else:
        dca_selective = None

    # Strategy 3: UQ-aware — use model for certain, treat-all for uncertain high-risk
    nb_uq = np.zeros(len(thresholds))
    for j, t in enumerate(thresholds):
        # Certain samples: use model prediction
        certain_pred = (y_prob[certain_mask] >= t).astype(int)
        tp_c = ((certain_pred == 1) & (y_true[certain_mask] == 1)).sum()
        fp_c = ((certain_pred == 1) & (y_true[certain_mask] == 0)).sum()

        # Uncertain samples: treat all (conservative approach)
        uncertain_mask = ~certain_mask
        tp_u = y_true[uncertain_mask].sum()
        fp_u = (1 - y_true[uncertain_mask]).sum()

        nb_uq[j] = (tp_c + tp_u) / n - (fp_c + fp_u) / n * (t / (1 - t))

    return dca_standard, dca_selective, nb_uq


def plot_dca(datasets: dict, output_path="Fig_DCA.tiff"):
    """
    Plot DCA for all datasets with bootstrap 95% CI.
    Clinically relevant threshold range: 5-25%.

    Parameters
    ----------
    datasets : dict like {"Temporal Val": (y, p)}
    """
    n = len(datasets)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]

    thresholds = np.linspace(0.01, 0.50, 100)

    for i, (name, (y, p)) in enumerate(datasets.items()):
        ax = axes[i]
        dca = compute_dca_with_ci(y, p, thresholds)

        # Model with CI
        ax.plot(thresholds, dca["nb_model"], color="#E53935", lw=2, label="Model")
        ax.fill_between(thresholds, dca["nb_model_lo"], dca["nb_model_hi"],
                        color="#E53935", alpha=0.15)
        # Treat all
        ax.plot(thresholds, dca["nb_all"], "b--", lw=1.5, label="Treat all")
        # Treat none
        ax.plot(thresholds, dca["nb_none"], "k-.", lw=1, label="Treat none")

        # Shade clinically relevant range
        ax.axvspan(0.05, 0.25, alpha=0.05, color="green", label="Clinical range (5–25%)")

        prev = y.mean()
        ax.set_xlim(0, 0.50)
        ax.set_ylim(-0.05, prev * 1.3)
        ax.set_xlabel("Threshold Probability", fontsize=11)
        ax.set_ylabel("Net Benefit", fontsize=11)
        ax.set_title(f"Decision Curve Analysis — {name}", fontsize=12, fontweight="bold")
        ax.legend(loc="upper right", fontsize=8)
        ax.grid(alpha=0.2)

    plt.tight_layout()
    fig.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.close()
    print(f"→ Saved: {output_path}")


def plot_dca_3strategy(y_true, y_prob, uncertainty,
                        output_path="Fig_DCA_3strategy.tiff"):
    """
    Novel 3-strategy DCA comparison for the manuscript.
    Shows the incremental value of UQ over standard risk prediction.
    """
    thresholds = np.linspace(0.01, 0.50, 100)
    dca_std, dca_sel, nb_uq = compute_dca_uq_strategies(
        y_true, y_prob, uncertainty, thresholds
    )

    fig, ax = plt.subplots(figsize=(8, 5.5))

    # Standard model
    ax.plot(thresholds, dca_std["nb_model"], color="#E53935", lw=2.5,
            label="Strategy 1: Risk prediction only")
    ax.fill_between(thresholds, dca_std["nb_model_lo"], dca_std["nb_model_hi"],
                    color="#E53935", alpha=0.1)

    # Selective prediction
    if dca_sel is not None:
        coverage = (~(uncertainty >= np.median(uncertainty))).mean()
        ax.plot(thresholds, dca_sel["nb_model"], color="#4CAF50", lw=2,
                label=f"Strategy 2: Selective prediction ({coverage*100:.0f}% coverage)")

    # UQ-aware
    ax.plot(thresholds, nb_uq, color="#9C27B0", lw=2, linestyle="--",
            label="Strategy 3: UQ-aware stratification")

    # Treat all / none
    ax.plot(thresholds, dca_std["nb_all"], "b:", lw=1.2, label="Treat all")
    ax.plot(thresholds, dca_std["nb_none"], "k-.", lw=1, label="Treat none")

    # Clinical range
    ax.axvspan(0.05, 0.25, alpha=0.05, color="green")
    ax.annotate("Clinical\nrange", xy=(0.15, ax.get_ylim()[1] * 0.9),
                fontsize=8, ha="center", color="green", alpha=0.7)

    prev = y_true.mean()
    ax.set_xlim(0, 0.50)
    ax.set_ylim(-0.05, prev * 1.3)
    ax.set_xlabel("Threshold Probability", fontsize=12)
    ax.set_ylabel("Net Benefit", fontsize=12)
    ax.set_title("Decision Curve Analysis: Standard vs UQ-Aware Strategies\n(Temporal Validation Cohort)",
                 fontsize=12, fontweight="bold")
    ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
    ax.grid(alpha=0.2)

    plt.tight_layout()
    fig.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.close()
    print(f"→ Saved: {output_path}")


# ═══════════════════════════════════════════════════════════════════════════
# HELPER: Compute ensemble mean and uncertainty from predictions matrix
# ═══════════════════════════════════════════════════════════════════════════

def ensemble_mean_and_tu(predictions, eps=1e-10):
    """
    From (n_models, n_samples) predictions matrix, compute:
      - p_mean: ensemble mean probability
      - TU: total uncertainty (predictive entropy of mean)

    Returns p_mean, tu
    """
    p_mean = predictions.mean(axis=0)
    p_mean = np.clip(p_mean, eps, 1 - eps)
    tu = -p_mean * np.log(p_mean) - (1 - p_mean) * np.log(1 - p_mean)
    return p_mean, tu


# ═══════════════════════════════════════════════════════════════════════════
# MASTER RUNNER
# ═══════════════════════════════════════════════════════════════════════════

def run_all_priority_analyses(
    dfs, features, outcome,
    y_train, y_val, y_test,
    predictions_train, predictions_val, predictions_test,
):
    """
    Execute all 5 priority analyses in sequence.

    Parameters
    ----------
    dfs : dict with "train", "val", "test" DataFrames
    features : list of feature names
    outcome : str, outcome column name
    y_train, y_val, y_test : 1-d int arrays
    predictions_train, predictions_val, predictions_test :
        (300, n) float arrays — per-model class-1 probabilities
    """
    # Compute ensemble means
    p_train, tu_train = ensemble_mean_and_tu(predictions_train)
    p_val, tu_val = ensemble_mean_and_tu(predictions_val)
    p_test, tu_test = ensemble_mean_and_tu(predictions_test)

    print("\n" + "█" * 80)
    print("  RUNNING ALL 5 PRIORITY REVISION ANALYSES")
    print("█" * 80)

    # ── PRIORITY 1 ──
    df_smd = investigate_prevalence_shift(dfs, features, outcome)

    # ── PRIORITY 3 ──
    print("\n" + "=" * 80)
    print("PRIORITY 3: Full Calibration Metrics")
    print("=" * 80)
    cal_train = compute_full_calibration(y_train, p_train, label="Training Set")
    cal_val = compute_full_calibration(y_val, p_val, label="Internal Validation")
    cal_test = compute_full_calibration(y_test, p_test, label="Temporal Validation")

    plot_calibration_comprehensive(
        {"Training": (y_train, p_train),
         "Internal Validation": (y_val, p_val),
         "Temporal Validation": (y_test, p_test)},
        "Fig_calibration_comprehensive.tiff"
    )

    # ── PRIORITY 2 ──
    p_logistic, p_isotonic, raw_m, log_m, iso_m = run_recalibration_comparison(
        y_val, p_val, y_test, p_test
    )

    # ── PRIORITY 5 ──
    print("\n" + "=" * 80)
    print("PRIORITY 5: AUPRC Analysis")
    print("=" * 80)
    auprc_train = compute_auprc_with_ci(y_train, p_train, label="Training Set")
    auprc_val = compute_auprc_with_ci(y_val, p_val, label="Internal Validation")
    auprc_test = compute_auprc_with_ci(y_test, p_test, label="Temporal Validation")

    plot_pr_curves(
        {"Training": (y_train, p_train),
         "Internal Validation": (y_val, p_val),
         "Temporal Validation": (y_test, p_test)},
        "Fig_PR_curves.tiff"
    )

    # ── PRIORITY 6 ──
    print("\n" + "=" * 80)
    print("PRIORITY 6: Decision Curve Analysis")
    print("=" * 80)

    # Standard DCA
    plot_dca(
        {"Training": (y_train, p_train),
         "Internal Validation": (y_val, p_val),
         "Temporal Validation": (y_test, p_test)},
        "Fig_DCA_all_splits.tiff"
    )

    # 3-strategy DCA (temporal validation only)
    plot_dca_3strategy(
        y_test, p_test, tu_test,
        "Fig_DCA_3strategy.tiff"
    )

    # ── Summary table export ──
    print("\n" + "=" * 80)
    print("SUMMARY: Export to CSV")
    print("=" * 80)

    summary = []
    for name, cal, auprc_res in [
        ("Training", cal_train, auprc_train),
        ("Internal Validation", cal_val, auprc_val),
        ("Temporal Validation", cal_test, auprc_test),
    ]:
        row = {"Dataset": name}
        for k, v in cal.items():
            row[k] = f"{v['point']:.4f} [{v['lower']:.4f}, {v['upper']:.4f}]"
        row["AUROC"] = f"{auprc_res['AUROC']:.4f} [{auprc_res['AUROC_lo']:.4f}, {auprc_res['AUROC_hi']:.4f}]"
        row["AUPRC"] = f"{auprc_res['AUPRC']:.4f} [{auprc_res['AUPRC_lo']:.4f}, {auprc_res['AUPRC_hi']:.4f}]"
        row["Prevalence"] = f"{auprc_res['prevalence']*100:.2f}%"
        summary.append(row)

    # Add recalibrated results
    row_recal = {"Dataset": "Temporal Val (Logistic Recal)"}
    for k, v in log_m.items():
        row_recal[k] = f"{v['point']:.4f} [{v['lower']:.4f}, {v['upper']:.4f}]"
    row_recal["AUROC"] = f"{roc_auc_score(y_test, p_logistic):.4f}"
    row_recal["AUPRC"] = f"{average_precision_score(y_test, p_logistic):.4f}"
    summary.append(row_recal)

    df_summary = pd.DataFrame(summary)
    df_summary.to_csv("priority_analyses_summary.csv", index=False)
    print("→ Saved: priority_analyses_summary.csv")

    print("\n" + "█" * 80)
    print("  ALL PRIORITY ANALYSES COMPLETE")
    print("  Output files:")
    print("    prevalence_shift_SMD.csv")
    print("    Fig_S_prevalence_shift_love_plot.tiff")
    print("    Fig_calibration_comprehensive.tiff")
    print("    Fig_recalibration_comparison.tiff")
    print("    Fig_PR_curves.tiff")
    print("    Fig_DCA_all_splits.tiff")
    print("    Fig_DCA_3strategy.tiff")
    print("    priority_analyses_summary.csv")
    print("█" * 80)


# ═══════════════════════════════════════════════════════════════════════════
# USAGE EXAMPLE (paste into your notebook after loading data)
# ═══════════════════════════════════════════════════════════════════════════

USAGE = """
# After running cells 1-11 in your notebook:

%run priority_analyses.py

run_all_priority_analyses(
    dfs=dfs,
    features=features,
    outcome="SpontAbortion",      # ← adjust to your actual outcome column name
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    predictions_train=predictions_train,
    predictions_val=predictions_val,
    predictions_test=predictions_test,
)
"""

run_all_priority_analyses(
    dfs=dfs,
    features=features,
    outcome="SpontAbortion",      # ← adjust to your actual outcome column name
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    predictions_train=predictions_train,
    predictions_val=predictions_val,
    predictions_test=predictions_test,
)




████████████████████████████████████████████████████████████████████████████████
  RUNNING ALL 5 PRIORITY REVISION ANALYSES
████████████████████████████████████████████████████████████████████████████████

PRIORITY 1: Prevalence Shift Investigation

── 1a. Outcome Prevalence ──
Split                       N   Events   Prevalence             95% CI
----------------------------------------------------------------------
Training              360,363   11,753        3.26% [3.20%, 3.32%]
Internal Val           40,041    1,306        3.26% [3.09%, 3.44%]
Temporal Val            1,822      262       14.38% [12.84%, 16.07%]

── 1b. Standardized Mean Differences (Training vs Temporal Val) ──

Features with |SMD| > 0.1 (imbalanced): 22 / 36

Feature                          Train Mean   TempVal Mean      SMD
--------------------------------------------------------------------
WifeCreat                            60.245         51.381    0.722 ***
LymphocytePct                        31.656     

Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


→ Saved: Fig_calibration_comprehensive.tiff

PRIORITY 2: Recalibration Comparison

── Raw (uncalibrated) ──

── Calibration Metrics: Temporal Val — Raw ──
Metric                         Estimate                 95% CI
--------------------------------------------------------------
CITL                             1.1282  [1.0015, 1.2457]
Calibration Slope                1.1625  [0.9845, 1.3925]
Calibration Intercept           -1.2378  [-1.3676, -1.0969]
Brier Score                      0.1510  [0.1440, 0.1572]
E/O Ratio                        2.3760  [2.1332, 2.6290]
ICI                              0.1979  [0.1842, 0.2122]

── Logistic Recalibration: Val→Test logistic recal ──
  Recalibration intercept: -3.0784
  Recalibration slope:     0.7906

── After logistic recalibration ──

── Calibration Metrics: Temporal Val — Logistic Recal ──
Metric                         Estimate                 95% CI
--------------------------------------------------------------
CITL                     

In [45]:
"""
Priority Revision Analyses for SA Prediction Manuscript
========================================================
Addresses 5 critical reviewer concerns for IF≥8 journal submission:

  Priority 1: Prevalence shift investigation (3.26% → 14.38%)
  Priority 2: Logistic recalibration + Venn-ABERS
  Priority 3: Full calibration metrics (CITL, slope, ICI, E/O, calibration plot)
  Priority 5: AUPRC with bootstrap 95% CI
  Priority 6: Decision curve analysis with bootstrap 95% CI

HOW TO USE:
-----------
This script assumes your notebook has already loaded:
  - X_train, X_val, X_test  (pd.DataFrame or np.ndarray with feature columns)
  - y_train, y_val, y_test  (1-d arrays, int 0/1)
  - predictions_train, predictions_val, predictions_test
      (each shape [300, n_samples] — per-model predicted probabilities)
  - features  (list of feature column names)
  - dfs       (dict with keys "train", "val", "test" — raw DataFrames)

Copy the functions you need into your notebook, or run:
  %run priority_analyses.py
then call the top-level runners at the bottom.

Output:
  - Console tables (copy into manuscript supplementary)
  - Publication-quality TIFF figures at 300 DPI
  - CSV exports of key results
"""

from __future__ import annotations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    precision_recall_curve, roc_curve, f1_score,
    accuracy_score, recall_score, precision_score,
    log_loss,
)
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
import warnings
warnings.filterwarnings("ignore")

DPI = 300
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


# ═══════════════════════════════════════════════════════════════════════════
# PRIORITY 1: Prevalence Shift Investigation
# ═══════════════════════════════════════════════════════════════════════════

def investigate_prevalence_shift(dfs: dict, features: list, outcome: str = "SA_outcome"):
    """
    Generate a comprehensive comparison table between training/val/test cohorts.

    Outputs:
      1. Standardized Mean Differences (SMD) for all features
      2. Prevalence comparison with 95% CI
      3. Key demographic distributions
      4. CSV export for manuscript Table S_shift

    Parameters
    ----------
    dfs : dict with keys "train", "val", "test", each a pd.DataFrame
    features : list of feature column names
    outcome : str, name of outcome column
    """
    print("\n" + "=" * 80)
    print("PRIORITY 1: Prevalence Shift Investigation")
    print("=" * 80)

    splits = {"Training": dfs["train"], "Internal Val": dfs["val"], "Temporal Val": dfs["test"]}

    # ── 1a. Prevalence with 95% Wilson CI ──
    print("\n── 1a. Outcome Prevalence ──")
    print(f"{'Split':<18} {'N':>10} {'Events':>8} {'Prevalence':>12} {'95% CI':>18}")
    print("-" * 70)
    for name, df in splits.items():
        y = df[outcome].values.astype(int)
        n = len(y)
        p = y.mean()
        # Wilson score interval
        z = 1.96
        denom = 1 + z**2 / n
        centre = (p + z**2 / (2 * n)) / denom
        margin = z * np.sqrt((p * (1 - p) + z**2 / (4 * n)) / n) / denom
        lo, hi = max(0, centre - margin), min(1, centre + margin)
        print(f"{name:<18} {n:>10,} {y.sum():>8,} {p*100:>11.2f}% [{lo*100:.2f}%, {hi*100:.2f}%]")

    # ── 1b. Standardized Mean Differences ──
    print("\n── 1b. Standardized Mean Differences (Training vs Temporal Val) ──")

    results = []
    df_train = dfs["train"]
    df_test = dfs["test"]

    for feat in features:
        try:
            x1 = pd.to_numeric(df_train[feat], errors="coerce").dropna()
            x2 = pd.to_numeric(df_test[feat], errors="coerce").dropna()
            if len(x1) < 10 or len(x2) < 10:
                continue

            # SMD = (mean1 - mean2) / pooled_sd
            mean1, mean2 = x1.mean(), x2.mean()
            sd1, sd2 = x1.std(), x2.std()
            pooled_sd = np.sqrt((sd1**2 + sd2**2) / 2)
            smd = (mean1 - mean2) / pooled_sd if pooled_sd > 0 else 0

            results.append({
                "Feature": feat,
                "Train Mean": mean1,
                "Train SD": sd1,
                "TempVal Mean": mean2,
                "TempVal SD": sd2,
                "SMD": smd,
                "Abs SMD": abs(smd),
            })
        except Exception:
            continue

    df_smd = pd.DataFrame(results).sort_values("Abs SMD", ascending=False)

    # Flag features with |SMD| > 0.1 (standard imbalance threshold)
    imbalanced = df_smd[df_smd["Abs SMD"] > 0.1]
    print(f"\nFeatures with |SMD| > 0.1 (imbalanced): {len(imbalanced)} / {len(df_smd)}")
    print(f"\n{'Feature':<30} {'Train Mean':>12} {'TempVal Mean':>14} {'SMD':>8}")
    print("-" * 68)
    for _, row in imbalanced.head(20).iterrows():
        flag = " ***" if abs(row["SMD"]) > 0.25 else " **" if abs(row["SMD"]) > 0.2 else ""
        print(f"{row['Feature']:<30} {row['Train Mean']:>12.3f} {row['TempVal Mean']:>14.3f} {row['SMD']:>8.3f}{flag}")

    # ── 1c. Save to CSV ──
    df_smd.to_csv("prevalence_shift_SMD.csv", index=False)
    print(f"\n→ Saved: prevalence_shift_SMD.csv")

    # ── 1d. Love plot (SMD visualization) ──
    fig, ax = plt.subplots(figsize=(8, max(6, len(df_smd) * 0.25)))
    top_n = min(36, len(df_smd))
    plot_df = df_smd.head(top_n).sort_values("Abs SMD", ascending=True)

    colors = ["#E53935" if abs(s) > 0.25 else "#FF9800" if abs(s) > 0.1 else "#4CAF50"
              for s in plot_df["SMD"]]

    ax.barh(range(top_n), plot_df["Abs SMD"].values, color=colors, height=0.7, edgecolor="white")
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(plot_df["Feature"].values, fontsize=8)
    ax.axvline(0.1, color="gray", linestyle="--", linewidth=1, label="|SMD| = 0.1")
    ax.axvline(0.25, color="red", linestyle="--", linewidth=1, label="|SMD| = 0.25")
    ax.set_xlabel("Absolute Standardized Mean Difference")
    ax.set_title("Covariate Balance: Training vs Temporal Validation Cohort", fontweight="bold")
    ax.legend(loc="lower right", fontsize=8)
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    fig.savefig("Fig_S_prevalence_shift_love_plot.tiff", dpi=DPI, bbox_inches="tight")
    plt.close()
    print("→ Saved: Fig_S_prevalence_shift_love_plot.tiff")

    return df_smd


# ═══════════════════════════════════════════════════════════════════════════
# PRIORITY 3: Full Calibration Metrics
# ═══════════════════════════════════════════════════════════════════════════

def calibration_in_the_large(y_true, y_prob):
    """CITL: logit(mean_pred) - logit(prevalence)."""
    prev = y_true.mean()
    mean_pred = y_prob.mean()
    eps = 1e-10
    citl = np.log(mean_pred / (1 - mean_pred + eps) + eps) - np.log(prev / (1 - prev + eps) + eps)
    return citl


def calibration_slope_intercept(y_true, y_prob):
    """Fit logistic regression of y on logit(p) to get slope and intercept."""
    eps = 1e-10
    lp = np.log(y_prob / (1 - y_prob + eps) + eps)
    lp = np.clip(lp, -15, 15)
    from sklearn.linear_model import LogisticRegression
    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000)
    lr.fit(lp.reshape(-1, 1), y_true)
    slope = lr.coef_[0][0]
    intercept = lr.intercept_[0]
    return slope, intercept


def eo_ratio(y_true, y_prob):
    """Expected-to-Observed ratio."""
    expected = y_prob.sum()
    observed = y_true.sum()
    return expected / max(observed, 1)


def integrated_calibration_index(y_true, y_prob, n_bins=50):
    """
    ICI: mean absolute difference between loess-smoothed calibration curve
    and the diagonal. Approximated with binned approach.
    """
    order = np.argsort(y_prob)
    y_sorted = y_true[order]
    p_sorted = y_prob[order]

    bins = np.array_split(np.arange(len(y_sorted)), n_bins)
    diffs = []
    for b in bins:
        if len(b) == 0:
            continue
        obs = y_sorted[b].mean()
        pred = p_sorted[b].mean()
        diffs.append(abs(obs - pred) * len(b))
    return sum(diffs) / len(y_true)


def compute_full_calibration(y_true, y_prob, n_bootstrap=200, label=""):
    """
    Compute all calibration metrics with bootstrap 95% CI.

    Returns dict with:
      CITL, Calibration Slope, Calibration Intercept, Brier Score,
      E/O Ratio, ICI — each with (point, lower, upper).
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    def _metrics(yt, yp):
        slope, intercept = calibration_slope_intercept(yt, yp)
        return {
            "CITL": calibration_in_the_large(yt, yp),
            "Calibration Slope": slope,
            "Calibration Intercept": intercept,
            "Brier Score": brier_score_loss(yt, yp),
            "E/O Ratio": eo_ratio(yt, yp),
            "ICI": integrated_calibration_index(yt, yp),
        }

    # Point estimates
    point = _metrics(y_true, y_prob)

    # Bootstrap CI
    boot_results = {k: [] for k in point}
    rng = np.random.RandomState(RANDOM_STATE)
    for _ in range(n_bootstrap):
        idx = rng.choice(len(y_true), len(y_true), replace=True)
        try:
            m = _metrics(y_true[idx], y_prob[idx])
            for k in boot_results:
                boot_results[k].append(m[k])
        except Exception:
            continue

    results = {}
    for k in point:
        arr = np.array(boot_results[k])
        lo, hi = np.percentile(arr, [2.5, 97.5])
        results[k] = {"point": point[k], "lower": lo, "upper": hi}

    # Print table
    if label:
        print(f"\n── Calibration Metrics: {label} ──")
    print(f"{'Metric':<28} {'Estimate':>10} {'95% CI':>22}")
    print("-" * 62)
    for k, v in results.items():
        print(f"{k:<28} {v['point']:>10.4f}  [{v['lower']:.4f}, {v['upper']:.4f}]")

    return results


def plot_calibration_comprehensive(datasets: dict, output_path="Fig_calibration_comprehensive.tiff"):
    """
    Generate publication-quality calibration plots for all datasets.

    Parameters
    ----------
    datasets : dict like {"Training": (y, p), "Internal Val": (y, p), "Temporal Val": (y, p)}
    """
    from sklearn.calibration import calibration_curve

    n = len(datasets)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]

    colors = ["#2196F3", "#4CAF50", "#F44336"]

    for i, (name, (y, p)) in enumerate(datasets.items()):
        ax = axes[i]
        # Calibration curve
        prob_true, prob_pred = calibration_curve(y, p, n_bins=10, strategy="quantile")
        ax.plot(prob_pred, prob_true, "o-", color=colors[i], lw=2, markersize=6, label="Model")
        ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="Perfect")

        # Histogram of predictions
        ax2 = ax.twinx()
        ax2.hist(p, bins=50, alpha=0.15, color=colors[i])
        ax2.set_ylabel("Frequency", fontsize=8, alpha=0.5)
        ax2.tick_params(axis="y", labelsize=7, colors="gray")

        # Metrics annotation
        brier = brier_score_loss(y, p)
        slope, intercept = calibration_slope_intercept(y, p)
        ici = integrated_calibration_index(y, p)
        eo = eo_ratio(y, p)

        textstr = (f"Brier = {brier:.4f}\n"
                   f"Slope = {slope:.3f}\n"
                   f"Intercept = {intercept:.3f}\n"
                   f"ICI = {ici:.4f}\n"
                   f"E/O = {eo:.3f}")
        ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=8,
                verticalalignment="top", bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

        ax.set_xlabel("Predicted Probability", fontsize=10)
        ax.set_ylabel("Observed Proportion", fontsize=10)
        ax.set_title(name, fontsize=12, fontweight="bold")
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)
        ax.set_aspect("equal")
        ax.legend(loc="lower right", fontsize=8)
        ax.grid(alpha=0.2)

    plt.tight_layout()
    fig.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.close()
    print(f"→ Saved: {output_path}")


# ═══════════════════════════════════════════════════════════════════════════
# PRIORITY 2: Logistic Recalibration
# ═══════════════════════════════════════════════════════════════════════════

def logistic_recalibration(y_cal, p_cal, y_test, p_test, label=""):
    """
    Apply logistic recalibration: fit logistic(y ~ logit(p)) on calibration
    data, transform test predictions.

    Parameters
    ----------
    y_cal, p_cal : calibration set (use internal validation)
    y_test, p_test : test set to recalibrate

    Returns
    -------
    p_recal : recalibrated probabilities for test set
    """
    eps = 1e-10
    lp_cal = np.log(np.clip(p_cal, eps, 1 - eps) / (1 - np.clip(p_cal, eps, 1 - eps)))
    lp_test = np.log(np.clip(p_test, eps, 1 - eps) / (1 - np.clip(p_test, eps, 1 - eps)))

    lr = LogisticRegression(penalty=None, solver="lbfgs", max_iter=1000)
    lr.fit(lp_cal.reshape(-1, 1), y_cal)

    p_recal = lr.predict_proba(lp_test.reshape(-1, 1))[:, 1]

    if label:
        print(f"\n── Logistic Recalibration: {label} ──")
        print(f"  Recalibration intercept: {lr.intercept_[0]:.4f}")
        print(f"  Recalibration slope:     {lr.coef_[0][0]:.4f}")

    return p_recal


def isotonic_recalibration(y_cal, p_cal, p_test):
    """Apply isotonic regression recalibration."""
    ir = IsotonicRegression(out_of_bounds="clip")
    ir.fit(p_cal, y_cal)
    return ir.predict(p_test)


def run_recalibration_comparison(y_val, p_val, y_test, p_test):
    """
    Compare raw vs logistic vs isotonic recalibration on temporal validation.

    Uses internal validation as calibration set.
    """
    print("\n" + "=" * 80)
    print("PRIORITY 2: Recalibration Comparison")
    print("=" * 80)

    # Raw
    print("\n── Raw (uncalibrated) ──")
    raw_metrics = compute_full_calibration(y_test, p_test, label="Temporal Val — Raw")

    # Logistic recalibration
    p_logistic = logistic_recalibration(y_val, p_val, y_test, p_test,
                                         label="Val→Test logistic recal")
    print("\n── After logistic recalibration ──")
    log_metrics = compute_full_calibration(y_test, p_logistic,
                                            label="Temporal Val — Logistic Recal")

    # Isotonic recalibration
    p_isotonic = isotonic_recalibration(y_val, p_val, p_test)
    print("\n── After isotonic recalibration ──")
    iso_metrics = compute_full_calibration(y_test, p_isotonic,
                                            label="Temporal Val — Isotonic Recal")

    # Check discrimination is preserved
    print("\n── Discrimination check (should be unchanged) ──")
    for name, p in [("Raw", p_test), ("Logistic", p_logistic), ("Isotonic", p_isotonic)]:
        auroc = roc_auc_score(y_test, p)
        auprc = average_precision_score(y_test, p)
        print(f"  {name:<12} AUROC={auroc:.4f}  AUPRC={auprc:.4f}")

    # Plot comparison
    from sklearn.calibration import calibration_curve
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    for ax, (name, p, color) in zip(axes, [
        ("Raw", p_test, "#F44336"),
        ("Logistic Recal", p_logistic, "#4CAF50"),
        ("Isotonic Recal", p_isotonic, "#2196F3"),
    ]):
        prob_true, prob_pred = calibration_curve(y_test, p, n_bins=10, strategy="quantile")
        ax.plot(prob_pred, prob_true, "o-", color=color, lw=2, markersize=6)
        ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)

        brier = brier_score_loss(y_test, p)
        ici = integrated_calibration_index(y_test, p)
        ax.set_title(f"{name}\nBrier={brier:.4f}, ICI={ici:.4f}", fontsize=10, fontweight="bold")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Observed")
        ax.set_xlim(-0.02, 1.02)
        ax.set_ylim(-0.02, 1.02)
        ax.set_aspect("equal")
        ax.grid(alpha=0.2)

    plt.tight_layout()
    fig.savefig("Fig_recalibration_comparison.tiff", dpi=DPI, bbox_inches="tight")
    plt.close()
    print("\n→ Saved: Fig_recalibration_comparison.tiff")

    return p_logistic, p_isotonic, raw_metrics, log_metrics, iso_metrics


# ═══════════════════════════════════════════════════════════════════════════
# PRIORITY 5: AUPRC with Bootstrap 95% CI
# ═══════════════════════════════════════════════════════════════════════════

def compute_auprc_with_ci(y_true, y_prob, n_bootstrap=200, label=""):
    """
    Compute AUPRC with bootstrap percentile 95% CI.

    For low-prevalence settings, also reports:
      - Baseline AUPRC (= prevalence)
      - Lift over baseline
      - F1 at optimal PR threshold
    """
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    # Point estimate
    auprc = average_precision_score(y_true, y_prob)
    prev = y_true.mean()
    auroc = roc_auc_score(y_true, y_prob)

    # Optimal F1 from PR curve
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-10)
    best_idx = np.argmax(f1_scores)
    best_f1 = f1_scores[best_idx]
    best_threshold = thresholds[best_idx] if best_idx < len(thresholds) else 0.5

    # Bootstrap
    boot_auprc = []
    boot_auroc = []
    rng = np.random.RandomState(RANDOM_STATE)
    for _ in range(n_bootstrap):
        idx = rng.choice(len(y_true), len(y_true), replace=True)
        if y_true[idx].sum() < 2 or y_true[idx].sum() == len(idx):
            continue
        boot_auprc.append(average_precision_score(y_true[idx], y_prob[idx]))
        boot_auroc.append(roc_auc_score(y_true[idx], y_prob[idx]))

    auprc_lo, auprc_hi = np.percentile(boot_auprc, [2.5, 97.5])
    auroc_lo, auroc_hi = np.percentile(boot_auroc, [2.5, 97.5])

    if label:
        print(f"\n── AUPRC Analysis: {label} ──")
    print(f"  Prevalence (baseline AUPRC): {prev:.4f} ({prev*100:.2f}%)")
    print(f"  AUROC:  {auroc:.4f}  [{auroc_lo:.4f}, {auroc_hi:.4f}]")
    print(f"  AUPRC:  {auprc:.4f}  [{auprc_lo:.4f}, {auprc_hi:.4f}]")
    print(f"  Lift over baseline:          {auprc / prev:.2f}x")
    print(f"  Best F1 = {best_f1:.4f} at threshold = {best_threshold:.4f}")

    return {
        "AUPRC": auprc, "AUPRC_lo": auprc_lo, "AUPRC_hi": auprc_hi,
        "AUROC": auroc, "AUROC_lo": auroc_lo, "AUROC_hi": auroc_hi,
        "prevalence": prev, "lift": auprc / prev,
        "best_f1": best_f1, "best_threshold": best_threshold,
    }


def plot_pr_curves(datasets: dict, output_path="Fig_PR_curves.tiff"):
    """
    Plot precision-recall curves for all datasets with baseline annotation.

    Parameters
    ----------
    datasets : dict like {"Training": (y, p), "Internal Val": (y, p), ...}
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ["#2196F3", "#4CAF50", "#F44336"]

    for i, (name, (y, p)) in enumerate(datasets.items()):
        precision, recall, _ = precision_recall_curve(y, p)
        auprc = average_precision_score(y, p)
        prev = y.mean()
        ax.plot(recall, precision, color=colors[i], lw=2,
                label=f"{name} (AUPRC={auprc:.3f}, prev={prev*100:.1f}%)")
        # Baseline
        ax.axhline(prev, color=colors[i], linestyle=":", alpha=0.4)

    ax.set_xlabel("Recall (Sensitivity)", fontsize=12)
    ax.set_ylabel("Precision (PPV)", fontsize=12)
    ax.set_title("Precision-Recall Curves", fontsize=14, fontweight="bold")
    ax.legend(loc="upper right", fontsize=9)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    fig.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.close()
    print(f"→ Saved: {output_path}")


# ═══════════════════════════════════════════════════════════════════════════
# PRIORITY 6: Decision Curve Analysis with Bootstrap 95% CI
# ═══════════════════════════════════════════════════════════════════════════

def compute_dca_with_ci(y_true, y_prob, thresholds=None, n_bootstrap=200):
    """
    DCA with bootstrap 95% CI for net benefit.

    Parameters
    ----------
    y_true : array-like, true labels
    y_prob : array-like, predicted probabilities
    thresholds : array-like, threshold probabilities (default: 0.01 to 0.50)
    n_bootstrap : int

    Returns
    -------
    dict with thresholds, nb_model (point, lo, hi), nb_all, nb_none
    """
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.50, 100)

    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    n = len(y_true)

    def _nb(yt, yp, t):
        pred = (yp >= t).astype(int)
        tp = ((pred == 1) & (yt == 1)).sum()
        fp = ((pred == 1) & (yt == 0)).sum()
        return tp / n - fp / n * (t / (1 - t))

    def _nb_all(yt, t):
        prev = yt.mean()
        return prev - (1 - prev) * t / (1 - t)

    # Point estimates
    nb_model = np.array([_nb(y_true, y_prob, t) for t in thresholds])
    nb_all = np.array([_nb_all(y_true, t) for t in thresholds])
    nb_none = np.zeros_like(thresholds)

    # Bootstrap CI for model net benefit
    boot_nb = np.zeros((n_bootstrap, len(thresholds)))
    rng = np.random.RandomState(RANDOM_STATE)
    for b in range(n_bootstrap):
        idx = rng.choice(n, n, replace=True)
        for j, t in enumerate(thresholds):
            boot_nb[b, j] = _nb(y_true[idx], y_prob[idx], t)

    nb_lo = np.percentile(boot_nb, 2.5, axis=0)
    nb_hi = np.percentile(boot_nb, 97.5, axis=0)

    return {
        "thresholds": thresholds,
        "nb_model": nb_model, "nb_model_lo": nb_lo, "nb_model_hi": nb_hi,
        "nb_all": nb_all, "nb_none": nb_none,
    }


def compute_dca_uq_strategies(y_true, y_prob, uncertainty, thresholds=None,
                               coverage_levels=None):
    """
    Compare DCA strategies at multiple uncertainty quantiles:
      1. Standard model (risk only, 100% coverage)
      2. Selective prediction at each coverage level — retain only the most
         certain samples (sorted by ascending TU) and abstain on the rest
      3. UQ-aware stratification — use model for certain samples, treat-all
         for uncertain samples (conservative clinical protocol)

    Net benefit for selective prediction is computed on the FULL population:
      NB_selective(t) = TP_retained/N_total − FP_retained/N_total × t/(1−t)
    Abstained samples contribute 0 to both TP and FP (they are deferred to
    clinician judgment, equivalent to "treat none" for that subset).

    Parameters
    ----------
    y_true, y_prob : arrays of shape (n,)
    uncertainty : array of shape (n,), total uncertainty per sample
    thresholds : array of threshold probabilities (default: 0.01–0.50)
    coverage_levels : list of floats in (0, 1], fractions of samples to
                      retain sorted by ascending uncertainty.
                      Default: [0.90, 0.80, 0.70, 0.60, 0.50]

    Returns
    -------
    dca_standard : dict — full-population DCA with bootstrap 95% CI
    selective_results : list of dicts, one per coverage level, each with
        keys "coverage", "mask", "nb_model" (array over thresholds)
    uq_aware_results : list of dicts, one per coverage level, each with
        keys "coverage", "nb_uq" (array over thresholds)
    """
    if thresholds is None:
        thresholds = np.linspace(0.01, 0.50, 100)
    if coverage_levels is None:
        coverage_levels = [0.90, 0.80, 0.70, 0.60, 0.50, 0.40, 0.30, 0.20, 0.10]

    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    uncertainty = np.asarray(uncertainty).astype(float)
    n = len(y_true)

    # Sort indices by ascending uncertainty (most certain first)
    order = np.argsort(uncertainty)

    # Strategy 1: Standard model — full population
    dca_standard = compute_dca_with_ci(y_true, y_prob, thresholds, n_bootstrap=200)

    # Strategy 2: Selective prediction at each coverage quantile
    selective_results = []
    for cov in coverage_levels:
        k = int(round(n * cov))             # number of retained samples
        retained_idx = order[:k]            # the k most-certain samples
        mask = np.zeros(n, dtype=bool)
        mask[retained_idx] = True

        actual_cov = mask.sum() / n
        y_ret = y_true[mask]
        p_ret = y_prob[mask]

        # Net benefit on the FULL population denominator
        nb_sel = np.zeros(len(thresholds))
        for j, t in enumerate(thresholds):
            pred = (p_ret >= t).astype(int)
            tp = ((pred == 1) & (y_ret == 1)).sum()
            fp = ((pred == 1) & (y_ret == 0)).sum()
            nb_sel[j] = tp / n - fp / n * (t / (1 - t))

        selective_results.append({
            "coverage": actual_cov,
            "mask": mask,
            "nb_model": nb_sel,
            "n_retained": mask.sum(),
            "prev_retained": y_ret.mean() if len(y_ret) > 0 else 0,
        })

    # Strategy 3: UQ-aware — certain → model, uncertain → treat all
    uq_aware_results = []
    for cov, sel in zip(coverage_levels, selective_results):
        certain_mask = sel["mask"]
        uncertain_mask = ~certain_mask
        nb_uq = np.zeros(len(thresholds))
        for j, t in enumerate(thresholds):
            # Certain: use model threshold
            pred_c = (y_prob[certain_mask] >= t).astype(int)
            tp_c = ((pred_c == 1) & (y_true[certain_mask] == 1)).sum()
            fp_c = ((pred_c == 1) & (y_true[certain_mask] == 0)).sum()

            # Uncertain: treat all (conservative)
            tp_u = y_true[uncertain_mask].sum()
            fp_u = (1 - y_true[uncertain_mask]).sum()

            nb_uq[j] = (tp_c + tp_u) / n - (fp_c + fp_u) / n * (t / (1 - t))

        uq_aware_results.append({
            "coverage": sel["coverage"],
            "nb_uq": nb_uq,
        })

    return dca_standard, selective_results, uq_aware_results


def plot_dca(datasets: dict, output_path="Fig_DCA.tiff"):
    """
    Plot DCA for all datasets with bootstrap 95% CI.
    Clinically relevant threshold range: 5-25%.

    Parameters
    ----------
    datasets : dict like {"Temporal Val": (y, p)}
    """
    n = len(datasets)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]

    thresholds = np.linspace(0.01, 0.50, 100)

    for i, (name, (y, p)) in enumerate(datasets.items()):
        ax = axes[i]
        dca = compute_dca_with_ci(y, p, thresholds)

        # Model with CI
        ax.plot(thresholds, dca["nb_model"], color="#E53935", lw=2, label="Model")
        ax.fill_between(thresholds, dca["nb_model_lo"], dca["nb_model_hi"],
                        color="#E53935", alpha=0.15)
        # Treat all
        ax.plot(thresholds, dca["nb_all"], "b--", lw=1.5, label="Treat all")
        # Treat none
        ax.plot(thresholds, dca["nb_none"], "k-.", lw=1, label="Treat none")

        # Shade clinically relevant range
        # ax.axvspan(0.05, 0.25, alpha=0.05, color="green", label="Clinical range (5–25%)")

        prev = y.mean()
        ax.set_xlim(0, 0.50)
        ax.set_ylim(-0.05, prev * 1.3)
        ax.set_xlabel("Threshold Probability", fontsize=11)
        ax.set_ylabel("Net Benefit", fontsize=11)
        ax.set_title(f"Decision Curve Analysis — {name}", fontsize=12, fontweight="bold")
        ax.legend(loc="upper right", fontsize=8)
        ax.grid(alpha=0.2)

    plt.tight_layout()
    fig.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.close()
    print(f"→ Saved: {output_path}")


def plot_dca_3strategy(y_true, y_prob, uncertainty,
                        coverage_levels=None,
                        output_path="Fig_DCA_3strategy.tiff"):
    """
    Novel multi-quantile DCA comparison for the manuscript.

    Produces a two-panel figure:
      Left:  Selective prediction — one curve per coverage quantile
             showing the accuracy-coverage trade-off across 9 levels
      Right: UQ-aware stratification — model for certain, treat-all for
             uncertain, one curve per coverage split

    Both panels include "Treat all" and "Treat none" baselines and the
    standard full-model curve (100%) for reference.  The clinically
    relevant threshold range (5–25%) is shaded.

    Parameters
    ----------
    y_true, y_prob : 1-d arrays
    uncertainty : 1-d array, total uncertainty per sample
    coverage_levels : list of floats, default
        [0.90, 0.80, 0.70, 0.60, 0.50, 0.40, 0.30, 0.20, 0.10]
    output_path : str, filename for the saved figure
    """
    if coverage_levels is None:
        coverage_levels = [0.90, 0.80, 0.70, 0.60, 0.50, 0.40, 0.30, 0.20, 0.10]

    thresholds = np.linspace(0.01, 0.50, 100)
    dca_std, sel_list, uq_list = compute_dca_uq_strategies(
        y_true, y_prob, uncertainty, thresholds, coverage_levels
    )

    prev = np.asarray(y_true).mean()
    n_levels = len(coverage_levels)

    # ── colour ramp: cool→warm as coverage decreases (more selective) ──
    from matplotlib.colors import Normalize
    from matplotlib.cm import ScalarMappable
    cmap = plt.cm.RdYlGn          # green=high coverage, red=low coverage
    norm = Normalize(vmin=min(coverage_levels) - 0.05,
                     vmax=max(coverage_levels) + 0.05)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7), sharey=True)

    # ────────── Panel A: Selective Prediction ──────────
    ax = ax1

    # Full model baseline (100%)
    ax.plot(thresholds, dca_std["nb_model"], color="black", lw=2.8,
            label="Full model (100%)", zorder=10)
    ax.fill_between(thresholds, dca_std["nb_model_lo"], dca_std["nb_model_hi"],
                    color="black", alpha=0.06)

    # One curve per coverage level
    for sel in sel_list:
        c = cmap(norm(sel["coverage"]))
        pct = sel["coverage"] * 100
        ax.plot(thresholds, sel["nb_model"], color=c, lw=1.6,
                label=f"{pct:.0f}% (n={sel['n_retained']}, "
                      f"prev={sel['prev_retained']*100:.1f}%)")

    ax.plot(thresholds, dca_std["nb_all"],  color="#1565C0", linestyle=":",
            lw=1.3, label="Treat all")
    ax.plot(thresholds, dca_std["nb_none"], color="gray", linestyle="-.",
            lw=1.0, label="Treat none")
    # ax.axvspan(0.05, 0.25, alpha=0.04, color="green")

    ax.set_xlim(0, 0.50)
    ax.set_ylim(-0.08, prev * 1.5)
    ax.set_xlabel("Threshold Probability", fontsize=12)
    ax.set_ylabel("Net Benefit", fontsize=12)
    ax.set_title("A  Selective Prediction by Uncertainty Quantile",
                 fontsize=12, fontweight="bold", loc="left")
    ax.legend(loc="upper right", fontsize=6.5, framealpha=0.92,
              ncol=1, title="Coverage", title_fontsize=7)
    ax.grid(alpha=0.2)

    # ────────── Panel B: UQ-Aware Stratification ──────────
    ax = ax2

    ax.plot(thresholds, dca_std["nb_model"], color="black", lw=2.8,
            label="Full model (100%)", zorder=10)
    ax.fill_between(thresholds, dca_std["nb_model_lo"], dca_std["nb_model_hi"],
                    color="black", alpha=0.06)

    for uq, sel in zip(uq_list, sel_list):
        c = cmap(norm(uq["coverage"]))
        pct = uq["coverage"] * 100
        ax.plot(thresholds, uq["nb_uq"], color=c, lw=1.6, linestyle="--",
                label=f"Certain {pct:.0f}% → model, rest → treat all")

    ax.plot(thresholds, dca_std["nb_all"],  color="#1565C0", linestyle=":",
            lw=1.3, label="Treat all")
    ax.plot(thresholds, dca_std["nb_none"], color="gray", linestyle="-.",
            lw=1.0, label="Treat none")
    # ax.axvspan(0.05, 0.25, alpha=0.04, color="green")

    ax.set_xlim(0, 0.50)
    ax.set_xlabel("Threshold Probability", fontsize=12)
    ax.set_title("B  UQ-Aware Stratification by Uncertainty Quantile",
                 fontsize=12, fontweight="bold", loc="left")
    ax.legend(loc="upper right", fontsize=6.5, framealpha=0.92,
              ncol=1, title="Strategy", title_fontsize=7)
    ax.grid(alpha=0.2)

    # # ── Shared colorbar ──
    # sm = ScalarMappable(cmap=cmap, norm=norm)
    # sm.set_array([])
    # cbar = fig.colorbar(sm, ax=[ax1, ax2], shrink=0.6, pad=0.02, aspect=30)
    # cbar.set_label("Coverage (fraction retained)", fontsize=10)
    # cbar.set_ticks(coverage_levels)
    # cbar.set_ticklabels([f"{c*100:.0f}%" for c in coverage_levels])

    fig.suptitle("Decision Curve Analysis — Temporal Validation Cohort",
                 fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout(rect=[0, 0, 0.92, 0.98])
    fig.savefig(output_path, dpi=DPI, bbox_inches="tight")
    plt.close()

    # ── Console summary table ──
    idx10 = np.argmin(np.abs(thresholds - 0.10))
    idx15 = np.argmin(np.abs(thresholds - 0.15))
    idx20 = np.argmin(np.abs(thresholds - 0.20))

    print(f"\n── DCA Selective Prediction Summary (Temporal Validation) ──")
    print(f"{'Coverage':>10} {'N retained':>12} {'Prev':>8} "
          f"{'NB@t=0.10':>10} {'NB@t=0.15':>10} {'NB@t=0.20':>10}")
    print("-" * 65)
    print(f"{'100%':>10} {len(y_true):>12,} {prev*100:>7.1f}% "
          f"{dca_std['nb_model'][idx10]:>10.4f} "
          f"{dca_std['nb_model'][idx15]:>10.4f} "
          f"{dca_std['nb_model'][idx20]:>10.4f}")
    for sel in sel_list:
        pct = f"{sel['coverage']*100:.0f}%"
        print(f"{pct:>10} {sel['n_retained']:>12,} {sel['prev_retained']*100:>7.1f}% "
              f"{sel['nb_model'][idx10]:>10.4f} "
              f"{sel['nb_model'][idx15]:>10.4f} "
              f"{sel['nb_model'][idx20]:>10.4f}")

    print(f"\n── DCA UQ-Aware Stratification Summary ──")
    print(f"{'Certain%':>10} "
          f"{'NB@t=0.10':>10} {'NB@t=0.15':>10} {'NB@t=0.20':>10}")
    print("-" * 45)
    for uq in uq_list:
        pct = f"{uq['coverage']*100:.0f}%"
        print(f"{pct:>10} "
              f"{uq['nb_uq'][idx10]:>10.4f} "
              f"{uq['nb_uq'][idx15]:>10.4f} "
              f"{uq['nb_uq'][idx20]:>10.4f}")

    print(f"\n→ Saved: {output_path}")


# ═══════════════════════════════════════════════════════════════════════════
# HELPER: Compute ensemble mean and uncertainty from predictions matrix
# ═══════════════════════════════════════════════════════════════════════════

def ensemble_mean_and_tu(predictions, eps=1e-10):
    """
    From (n_models, n_samples) predictions matrix, compute:
      - p_mean: ensemble mean probability
      - TU: total uncertainty (predictive entropy of mean)

    Returns p_mean, tu
    """
    p_mean = predictions.mean(axis=0)
    p_mean = np.clip(p_mean, eps, 1 - eps)
    tu = -p_mean * np.log(p_mean) - (1 - p_mean) * np.log(1 - p_mean)
    return p_mean, tu


# ═══════════════════════════════════════════════════════════════════════════
# MASTER RUNNER
# ═══════════════════════════════════════════════════════════════════════════

def run_all_priority_analyses(
    dfs, features, outcome,
    y_train, y_val, y_test,
    predictions_train, predictions_val, predictions_test,
):
    """
    Execute all 5 priority analyses in sequence.

    Parameters
    ----------
    dfs : dict with "train", "val", "test" DataFrames
    features : list of feature names
    outcome : str, outcome column name
    y_train, y_val, y_test : 1-d int arrays
    predictions_train, predictions_val, predictions_test :
        (300, n) float arrays — per-model class-1 probabilities
    """
    # Compute ensemble means
    p_train, tu_train = ensemble_mean_and_tu(predictions_train)
    p_val, tu_val = ensemble_mean_and_tu(predictions_val)
    p_test, tu_test = ensemble_mean_and_tu(predictions_test)

    print("\n" + "█" * 80)
    print("  RUNNING ALL 5 PRIORITY REVISION ANALYSES")
    print("█" * 80)

    # ── PRIORITY 1 ──
    df_smd = investigate_prevalence_shift(dfs, features, outcome)

    # ── PRIORITY 3 ──
    print("\n" + "=" * 80)
    print("PRIORITY 3: Full Calibration Metrics")
    print("=" * 80)
    cal_train = compute_full_calibration(y_train, p_train, label="Training Set")
    cal_val = compute_full_calibration(y_val, p_val, label="Internal Validation")
    cal_test = compute_full_calibration(y_test, p_test, label="Temporal Validation")

    plot_calibration_comprehensive(
        {"Training": (y_train, p_train),
         "Internal Validation": (y_val, p_val),
         "Temporal Validation": (y_test, p_test)},
        "Fig_calibration_comprehensive.tiff"
    )

    # ── PRIORITY 2 ──
    p_logistic, p_isotonic, raw_m, log_m, iso_m = run_recalibration_comparison(
        y_val, p_val, y_test, p_test
    )

    # ── PRIORITY 5 ──
    print("\n" + "=" * 80)
    print("PRIORITY 5: AUPRC Analysis")
    print("=" * 80)
    auprc_train = compute_auprc_with_ci(y_train, p_train, label="Training Set")
    auprc_val = compute_auprc_with_ci(y_val, p_val, label="Internal Validation")
    auprc_test = compute_auprc_with_ci(y_test, p_test, label="Temporal Validation")

    plot_pr_curves(
        {"Training": (y_train, p_train),
         "Internal Validation": (y_val, p_val),
         "Temporal Validation": (y_test, p_test)},
        "Fig_PR_curves.tiff"
    )

    # ── PRIORITY 6 ──
    print("\n" + "=" * 80)
    print("PRIORITY 6: Decision Curve Analysis")
    print("=" * 80)

    # Standard DCA
    plot_dca(
        {"Training": (y_train, p_train),
         "Internal Validation": (y_val, p_val),
         "Temporal Validation": (y_test, p_test)},
        "Fig_DCA_all_splits.tiff"
    )

    # 3-strategy DCA (temporal validation only)
    plot_dca_3strategy(
        y_test, p_test, tu_test,
        output_path="Fig_DCA_3strategy.tiff"
    )

    # ── Summary table export ──
    print("\n" + "=" * 80)
    print("SUMMARY: Export to CSV")
    print("=" * 80)

    summary = []
    for name, cal, auprc_res in [
        ("Training", cal_train, auprc_train),
        ("Internal Validation", cal_val, auprc_val),
        ("Temporal Validation", cal_test, auprc_test),
    ]:
        row = {"Dataset": name}
        for k, v in cal.items():
            row[k] = f"{v['point']:.4f} [{v['lower']:.4f}, {v['upper']:.4f}]"
        row["AUROC"] = f"{auprc_res['AUROC']:.4f} [{auprc_res['AUROC_lo']:.4f}, {auprc_res['AUROC_hi']:.4f}]"
        row["AUPRC"] = f"{auprc_res['AUPRC']:.4f} [{auprc_res['AUPRC_lo']:.4f}, {auprc_res['AUPRC_hi']:.4f}]"
        row["Prevalence"] = f"{auprc_res['prevalence']*100:.2f}%"
        summary.append(row)

    # Add recalibrated results
    row_recal = {"Dataset": "Temporal Val (Logistic Recal)"}
    for k, v in log_m.items():
        row_recal[k] = f"{v['point']:.4f} [{v['lower']:.4f}, {v['upper']:.4f}]"
    row_recal["AUROC"] = f"{roc_auc_score(y_test, p_logistic):.4f}"
    row_recal["AUPRC"] = f"{average_precision_score(y_test, p_logistic):.4f}"
    summary.append(row_recal)

    df_summary = pd.DataFrame(summary)
    df_summary.to_csv("priority_analyses_summary.csv", index=False)
    print("→ Saved: priority_analyses_summary.csv")

    print("\n" + "█" * 80)
    print("  ALL PRIORITY ANALYSES COMPLETE")
    print("  Output files:")
    print("    prevalence_shift_SMD.csv")
    print("    Fig_S_prevalence_shift_love_plot.tiff")
    print("    Fig_calibration_comprehensive.tiff")
    print("    Fig_recalibration_comparison.tiff")
    print("    Fig_PR_curves.tiff")
    print("    Fig_DCA_all_splits.tiff")
    print("    Fig_DCA_3strategy.tiff")
    print("    priority_analyses_summary.csv")
    print("█" * 80)


# ═══════════════════════════════════════════════════════════════════════════
# USAGE EXAMPLE (paste into your notebook after loading data)
# ═══════════════════════════════════════════════════════════════════════════

run_all_priority_analyses(
    dfs=dfs,
    features=features,
    outcome="SpontAbortion",      # ← adjust to your actual outcome column name
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    predictions_train=predictions_train,
    predictions_val=predictions_val,
    predictions_test=predictions_test,
)


████████████████████████████████████████████████████████████████████████████████
  RUNNING ALL 5 PRIORITY REVISION ANALYSES
████████████████████████████████████████████████████████████████████████████████

PRIORITY 1: Prevalence Shift Investigation

── 1a. Outcome Prevalence ──
Split                       N   Events   Prevalence             95% CI
----------------------------------------------------------------------
Training              360,363   11,753        3.26% [3.20%, 3.32%]
Internal Val           40,041    1,306        3.26% [3.09%, 3.44%]
Temporal Val            1,822      262       14.38% [12.84%, 16.07%]

── 1b. Standardized Mean Differences (Training vs Temporal Val) ──

Features with |SMD| > 0.1 (imbalanced): 22 / 36

Feature                          Train Mean   TempVal Mean      SMD
--------------------------------------------------------------------
WifeCreat                            60.245         51.381    0.722 ***
LymphocytePct                        31.656     

Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.
Ignoring fixed y limits to fulfill fixed data aspect with adjustable data limits.


→ Saved: Fig_calibration_comprehensive.tiff

PRIORITY 2: Recalibration Comparison

── Raw (uncalibrated) ──

── Calibration Metrics: Temporal Val — Raw ──
Metric                         Estimate                 95% CI
--------------------------------------------------------------
CITL                             1.1282  [1.0078, 1.2462]
Calibration Slope                1.1625  [0.9655, 1.3679]
Calibration Intercept           -1.2378  [-1.3876, -1.0965]
Brier Score                      0.1510  [0.1456, 0.1577]
E/O Ratio                        2.3760  [2.1376, 2.6244]
ICI                              0.1979  [0.1826, 0.2140]

── Logistic Recalibration: Val→Test logistic recal ──
  Recalibration intercept: -3.0784
  Recalibration slope:     0.7906

── After logistic recalibration ──

── Calibration Metrics: Temporal Val — Logistic Recal ──
Metric                         Estimate                 95% CI
--------------------------------------------------------------
CITL                     

---
## Pipeline Complete

All outputs saved to current directory.